In [ ]:

from pathlib import Path
import os
REPRO_MODE = os.environ.get('DATEPALM_REPRO_MODE', 'VERIFY_ARCHIVE')
assert REPRO_MODE in ('VERIFY_ARCHIVE','REBUILD_TABLES'), 'Historical training/testing is not automatically replayed from released artifacts.'
CWD=Path.cwd().resolve()
REPO=CWD.parent if CWD.name=='notebooks' and (CWD.parent/'README.md').is_file() else CWD
ARCHIVE_ROOT = Path("external_data")
SCRATCH_ROOT=Path(os.environ.get('DATEPALM_SCRATCH_DIR',str(REPO/'scratch_recomputed'))).expanduser().resolve()
if ARCHIVE_ROOT==SCRATCH_ROOT or ARCHIVE_ROOT in SCRATCH_ROOT.parents:
    raise RuntimeError('Scratch output must not be inside the immutable released artifact directory')
assert not (REPRO_MODE=='VERIFY_ARCHIVE' and 'RUN_PHASE' in globals())
print('Reproduction mode:',REPRO_MODE)
print('Immutable artifacts:',ARCHIVE_ROOT)
print('Scratch output:',SCRATCH_ROOT)


In [ ]:
from __future__ import annotations
import os, sys, time, json, hashlib, marshal, types, traceback, platform, warnings
from pathlib import Path
from types import SimpleNamespace
import numpy as np
import pandas as pd
import torch
import cv2
from tqdm.auto import tqdm
from IPython.display import display


OUT=ARCHIVE_ROOT
RAW=Path(os.environ.get('DATEPALM_RAW_DATA_DIR', str(REPO/'raw_data'))).expanduser().resolve()
SEED_SPLIT=40721
MAX_EPOCHS=60
BATCH_SIZE=24
BOOTSTRAPS=2000
PRETRAINED=True
SOURCE_CITATION='https://doi.org/10.17632/d986hyr2kb.1'
LICENSE_NOTE='Original source data must be credited; dataset publisher shows CC BY 4.0.'



print('Official source-photo location (needed only for new inference):',RAW)
print('Extracted frozen artifact root:',OUT)


In [ ]:
"""DatePalm V4 shared non-mechanism utilities. No previous-study dependencies."""
from __future__ import annotations
import hashlib, json, os, random, sys, time
from pathlib import Path
import numpy as np
import cv2
from PIL import Image, ImageOps

SHAPE=(256,256)
FAMILIES_TRAIN=['TR_S1','TR_S2','TR_S3','TR_S4']
FAMILIES_TEST=['HO_T1','HO_T2','HO_T3','HO_T4']
SEVERITIES=['low','medium','high']
SEEDS=[1049,2053,3067,4093,5051]
LEVEL_AREA={'low':(.003,.010),'medium':(.010,.030),'high':(.030,.070)}
LEVEL_POWER={'low':0.35,'medium':0.56,'high':0.78}

def digest(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(2**20),b''):h.update(b)
    return h.hexdigest()

def idhash(*parts):
    return hashlib.sha256('|'.join(map(str,parts)).encode()).hexdigest()

def make_rng(*parts):
    return np.random.default_rng(int(idhash(*parts)[:16],16))

def write_json(path,object):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_suffix(path.suffix+'.tmp');tmp.write_text(json.dumps(object,indent=2,sort_keys=False,default=str))
    os.replace(tmp,path)

def read_json(path):return json.loads(Path(path).read_text())

def image_read(path):
    with Image.open(path) as im:
        im=ImageOps.exif_transpose(im).convert('RGB'); im.thumbnail(SHAPE, Image.Resampling.BICUBIC)
        canvas=Image.new('RGB',SHAPE,(128,128,128))
        ox=(SHAPE[0]-im.width)//2;oy=(SHAPE[1]-im.height)//2
        canvas.paste(im,(ox,oy))
    return np.array(canvas,dtype=np.uint8)

def save_png(path,im):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    cv2.imwrite(str(path),cv2.cvtColor(im,cv2.COLOR_RGB2BGR))

def save_mask(path,mask):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    cv2.imwrite(str(path),(mask>0).astype(np.uint8)*255)

def foreground(im):
    """Fixed automatic GrabCut mask; doesn't use ground-truth anomaly masks."""
    h,w=im.shape[:2]; bgr=cv2.cvtColor(im,cv2.COLOR_RGB2BGR)
    seeds=np.zeros((h,w),np.uint8)
    pad=8;seeds[:pad,:]=0;seeds[-pad:,:]=0;seeds[:,:pad]=0;seeds[:,-pad:]=0

    rect=(pad,pad,w-2*pad,h-2*pad)
    try:
        bg=np.zeros((1,65),np.float64);fg=np.zeros((1,65),np.float64)
        cv2.grabCut(bgr,seeds,rect,bg,fg,3,cv2.GC_INIT_WITH_RECT)
        mask=np.where((seeds==2)|(seeds==0),0,1).astype('uint8')
    except cv2.error:
        mask=np.zeros((h,w),np.uint8)
    mask=cv2.morphologyEx(mask,cv2.MORPH_CLOSE,np.ones((7,7),np.uint8))
    n,l,stats,_=cv2.connectedComponentsWithStats(mask,8)
    if n<2:return mask.astype(bool),'empty'
    main=1+np.argmax(stats[1:,cv2.CC_STAT_AREA]); mask=(l==main).astype('uint8')
    area=float(mask.mean())
    if not .045<=area<=.90:return mask.astype(bool),'area'
    if np.sum(mask[0,:])+np.sum(mask[-1,:])+np.sum(mask[:,0])+np.sum(mask[:,-1])>50:
        return mask.astype(bool),'border'
    ys,xs=np.where(mask>0)
    if len(xs)<256:return mask.astype(bool),'too_small'
    return mask.astype(bool),'ok'

def orientation(fg):
    yy,xx=np.where(fg)
    if len(xx)<30:return np.array([0.,1.],np.float32)
    z=np.stack([xx-xx.mean(),yy-yy.mean()],1).astype('float32')
    cov=z.T@z/len(z); vals,vec=np.linalg.eigh(cov)
    v=vec[:,np.argmax(vals)];return v/np.linalg.norm(v)

def choose_point(fg,rng):
    inner=cv2.erode(fg.astype('uint8'),np.ones((9,9),np.uint8),iterations=1)>0
    coords=np.argwhere(inner)
    if not len(coords): coords=np.argwhere(fg)
    if not len(coords):raise ValueError('Empty fruit mask')
    y,x=coords[rng.integers(len(coords))];return float(x),float(y)

def gaussian_alpha(h,w,point,sx,sy,angle=0):
    yy,xx=np.mgrid[:h,:w].astype(np.float32);dx=xx-point[0];dy=yy-point[1]
    c=np.cos(angle);s=np.sin(angle); xxr=c*dx+s*dy; yyr=-s*dx+c*dy
    return np.exp(-.5*((xxr/max(sx,1))**2+(yyr/max(sy,1))**2))

def finish(original,modified,fg,alpha=None,threshold=3):

    modified=np.clip(modified,0,255).astype('uint8');modified[~fg]=original[~fg]
    diff=np.max(np.abs(modified.astype('int16')-original.astype('int16')),axis=2)
    mask=fg&(diff>=threshold)
    if not np.any(mask):raise ValueError('No visible image alteration')
    return modified,mask.astype(np.uint8)

def benign(im,kind,source_id):
    rng=make_rng('benign',kind,source_id); imf=im.astype(np.float32)
    if kind==0:
        a=float(rng.uniform(.94,1.06)); color=rng.uniform(.98,1.02,3)
        z=np.clip(imf*a*color[None,None],0,255).astype(np.uint8)
    elif kind==1:
        quality=int(rng.integers(84,96));bgr=cv2.cvtColor(im,cv2.COLOR_RGB2BGR)
        ok,buf=cv2.imencode('.jpg',bgr,[cv2.IMWRITE_JPEG_QUALITY,quality]);assert ok
        z=cv2.cvtColor(cv2.imdecode(buf,cv2.IMREAD_COLOR),cv2.COLOR_BGR2RGB)
    elif kind==2:
        z=cv2.GaussianBlur(im,(3,3),float(rng.uniform(.2,.7)))
    else:raise ValueError(kind)
    return z

def severity_target(fg,sev,rng):
    low,high=LEVEL_AREA[sev];return float(rng.uniform(low,high)*fg.sum())

def change_stats(orig,changed,mask,fg):
    return {'support_fraction':float(mask.sum()/max(fg.sum(),1)),
            'off_fruit_pixels':int((mask.astype(bool)&(~fg)).sum()),
            'mean_abs_change':float(np.abs(orig.astype('float32')-changed.astype('float32'))[mask>0].mean()),
            'nonempty':bool(mask.sum()>0)}


def load_source(row, raw_root):
    image=image_read(Path(raw_root)/row['relative_path'])
    cached=OUT/'data'/'foreground_cache'/f"{row['source_id']}.png"
    if cached.exists():
        mask=cv2.imread(str(cached),cv2.IMREAD_GRAYSCALE)
        if mask is None or mask.shape!=image.shape[:2]:
            raise RuntimeError('Corrupt cached source foreground: '+str(cached))
        return image,mask>127,'ok'
    mask,status=foreground(image)
    if status=='ok':save_mask(cached,mask)
    return image,mask,status


In [ ]:
"""Training-only synthetic mechanisms TR_S1..4. Never import heldout_generators."""
import numpy as np, cv2


def _train_generate_once(im,fg,method,sev,rng,mode='fruit'):
    if method not in ['TR_S1','TR_S2','TR_S3','TR_S4']:raise ValueError(method)
    h,w=fg.shape;point=choose_point(fg,rng); target=severity_target(fg,sev,rng)
    radius=np.clip(np.sqrt(target/np.pi),3,50)
    if method=='TR_S1':radius*=.58
    if method=='TR_S2':radius*=.75
    v=orientation(fg) if mode=='fruit' else np.array([np.cos(rng.uniform(0,2*np.pi)),np.sin(rng.uniform(0,2*np.pi))])
    angle=np.arctan2(v[1],v[0]);strength=LEVEL_POWER[sev]
    y,x=np.mgrid[:h,:w].astype(np.float32)
    if method=='TR_S1':
        a=gaussian_alpha(h,w,point,radius,radius*(.65 if mode=='fruit' else 1),angle)
        a=a*(1+.18*np.sin(x/5+y/9))*fg
        z=im.astype('float32')*(1-strength*a[...,None]*.70)
    elif method=='TR_S2':
        a=gaussian_alpha(h,w,point,radius,radius*.9,angle)*fg
        a=np.clip((a-.16)/.84,0,1)
        lab=cv2.cvtColor(im,cv2.COLOR_RGB2LAB).astype('float32')
        lab[:,:,1]+=a*strength*48; lab[:,:,2]-=a*strength*31
        z=cv2.cvtColor(np.clip(lab,0,255).astype('uint8'),cv2.COLOR_LAB2RGB).astype('float32')
    elif method=='TR_S3':
        v=np.array([np.cos(angle),np.sin(angle)])
        perp=np.array([-v[1],v[0]])
        length=max(18.,radius*5.5)
        t=np.linspace(-1,1,85)
        path=np.array(point)[None]+t[:,None]*length/2*v[None]+(np.sin(t*np.pi)*radius*.25)[:,None]*perp[None]
        ink=np.zeros((h,w),np.uint8)
        pts=np.round(path).astype(np.int32)
        cv2.polylines(ink,[pts],False,255,int(max(1,round(radius*.13))),cv2.LINE_AA)
        a=cv2.GaussianBlur(ink.astype('float32')/255,(0,0),1.3)*fg
        z=im.astype('float32')*(1-strength*.78*a[...,None])
    else:

        ink=np.zeros((h,w),np.uint8)
        for _ in range(max(8,int(target/8))):
            ox,oy=rng.normal(0,radius*.65,2);pt=np.array(point)+[ox,oy]
            length=float(rng.uniform(2,10));v=np.array([np.cos(angle),np.sin(angle)])
            p1=np.round(pt-length/2*v).astype(int);p2=np.round(pt+length/2*v).astype(int)
            cv2.line(ink,tuple(p1),tuple(p2),255,1,cv2.LINE_AA)
        a=cv2.GaussianBlur(ink.astype('float32')/255,(0,0),.8)*fg
        z=im.astype('float32')*(1-strength*.67*a[...,None])
    return finish(im,z,fg)


"""Held-out mechanisms: separate renderers, not used by training, development or calibration."""
import numpy as np, cv2


def _heldout_generate_once(im,fg,method,sev,rng):
    if method not in ['HO_T1','HO_T2','HO_T3','HO_T4']:raise ValueError(method)
    h,w=fg.shape;point=choose_point(fg,rng);size=severity_target(fg,sev,rng)
    r=np.clip(np.sqrt(size/np.pi),4,37); yy,xx=np.mgrid[:h,:w].astype(np.float32)
    strength=LEVEL_POWER[sev];dx=xx-point[0];dy=yy-point[1]
    if method=='HO_T1':

        distance=np.sqrt(dx*dx+dy*dy)/r
        rim=np.exp(-((distance-.72)/.20)**2)
        center=np.clip(1-distance,0,1)**1.5
        shadow=-.60*center+.32*rim*(dx/max(r,1))
        z=im.astype('float32')+strength*65*shadow[...,None]
    elif method=='HO_T2':

        canvas=np.zeros((h,w),np.uint8)
        v=orientation(fg);angle=np.arctan2(v[1],v[0]);nbranch=int(np.clip(round(r/2),3,12))
        trunk=np.array(point);end=trunk+v*r*2.8
        cv2.line(canvas,tuple(np.round(trunk).astype(int)),tuple(np.round(end).astype(int)),255,1,cv2.LINE_AA)
        for k in range(nbranch):
            u=float(rng.uniform(.14,.9));base=trunk+(end-trunk)*u
            phi=angle+(-1 if k%2 else 1)*float(rng.uniform(.35,1.12))
            tip=base+np.array([np.cos(phi),np.sin(phi)])*float(rng.uniform(r*.4,r*1.5))
            cv2.line(canvas,tuple(np.round(base).astype(int)),tuple(np.round(tip).astype(int)),255,1,cv2.LINE_AA)
        a=cv2.GaussianBlur(canvas.astype('float32')/255,(0,0),1.4)
        z=im.astype('float32')*(1-strength*.78*a[...,None])
    elif method=='HO_T3':

        canvas=np.zeros((h,w),np.uint8)
        n=int(np.clip(size/12,5,350))
        for _ in range(n):
            px=point[0]+rng.normal(0,r*.8);py=point[1]+rng.normal(0,r*.8)
            cv2.circle(canvas,(int(px),int(py)),int(rng.choice([1,2,3])),255,-1,cv2.LINE_AA)
        a=cv2.GaussianBlur(canvas.astype('float32')/255,(0,0),.8)
        z=im.astype('float32')*(1-strength*.72*a[...,None])
    else:

        u=orientation(fg); along=dx*u[0]+dy*u[1];cross=-dx*u[1]+dy*u[0]
        envelope=np.exp(-.5*((along/(r*1.8))**2+(cross/(r*.9))**2))
        wave=np.sin(2*np.pi*cross/max(r*.75,3))*envelope
        disp=strength*max(2,r*.30)*wave
        map_x=(xx+u[0]*disp).astype('float32');map_y=(yy+u[1]*disp).astype('float32')
        warped=cv2.remap(im,map_x,map_y,cv2.INTER_LINEAR,borderMode=cv2.BORDER_REFLECT).astype('float32')
        shading=wave*strength*23
        z=warped+shading[...,None]

        z=im.astype('float32')*(1-envelope[...,None])+z*envelope[...,None]
    return finish(im,z,fg)





MAX_SYNTH_REDRAWS = 24

def train_generate(im,fg,method,sev,rng,mode='fruit'):
    for _ in range(MAX_SYNTH_REDRAWS):
        try:
            return _train_generate_once(im,fg,method,sev,rng,mode)
        except ValueError as exc:
            if str(exc) != 'No visible image alteration':
                raise
    raise RuntimeError(
        f'{method} ({sev}, {mode}) produced no measurable alteration after '
        f'{MAX_SYNTH_REDRAWS} deterministic attempts; inspect this source image.'
    )

def heldout_generate(im,fg,method,sev,rng):
    for _ in range(MAX_SYNTH_REDRAWS):
        try:
            return _heldout_generate_once(im,fg,method,sev,rng)
        except ValueError as exc:
            if str(exc) != 'No visible image alteration':
                raise
    raise RuntimeError(
        f'{method} ({sev}) produced no measurable alteration after '
        f'{MAX_SYNTH_REDRAWS} deterministic attempts; inspect this source image.'
    )


In [ ]:
"""Raw source inventory, approximate duplicate screening and new group-stratified split."""
from pathlib import Path
from collections import defaultdict,Counter
import cv2,numpy as np,pandas as pd
from skimage.metrics import structural_similarity as ssim
from tqdm.auto import tqdm


EXT={'.jpg','.jpeg','.png','.bmp','.tiff','.tif','.webp'}

class DSU:
    def __init__(self,n):self.p=list(range(n))
    def find(self,x):
        while self.p[x]!=x:self.p[x]=self.p[self.p[x]];x=self.p[x]
        return x
    def union(self,a,b):self.p[self.find(a)]=self.find(b)

def hashes(im):
    gray=cv2.cvtColor(im,cv2.COLOR_RGB2GRAY)
    x=cv2.resize(gray,(9,8),interpolation=cv2.INTER_AREA)
    bits=(x[:,1:]>x[:,:-1]).reshape(-1)
    dh=sum(int(q)<<i for i,q in enumerate(bits))
    z=cv2.dct(cv2.resize(gray,(32,32)).astype('float32'))[:8,:8]
    b=(z>np.median(z[1:,:])).reshape(-1)
    ph=sum(int(q)<<i for i,q in enumerate(b))
    return dh,ph

def pairs_from_hashes(vals):

    buckets=defaultdict(list);pairs=set()
    for i,v in enumerate(vals):
        for k in range(8):
            bk=(k,(int(v)>>(8*k))&255)
            for j in buckets[bk]:pairs.add((j,i))
            buckets[bk].append(i)
    return pairs



def embedding_candidates(images,topk=5,cosine_floor=.992):
    """Independent ImageNet-pretrained ResNet18 retrieval of near-duplicate candidates.
    Retrieval only; it NEVER automatically declares a duplicate without geometric evidence.
    """
    import torch
    import torch.nn.functional as F
    from torchvision.models import resnet18,ResNet18_Weights
    device='cuda' if torch.cuda.is_available() else 'cpu'
    net=resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
    net.fc=torch.nn.Identity();net.eval().to(device)
    parts=[];indices=[]
    mean=torch.tensor([.485,.456,.406],device=device)[None,:,None,None]
    std=torch.tensor([.229,.224,.225],device=device)[None,:,None,None]
    with torch.no_grad():
        for start in tqdm(range(0,len(images),32),desc="P0: image embeddings",unit="batch"):
            sub=[(i,im) for i,im in enumerate(images[start:start+32],start) if im is not None]
            if not sub:continue
            tensor=torch.from_numpy(np.stack([im for _,im in sub]).transpose(0,3,1,2)).float().to(device)/255.
            embeddings=F.normalize(net((tensor-mean)/std).float(),dim=1)
            parts.append(embeddings.cpu());indices.extend(i for i,_ in sub)
    if not parts:return set(),{}
    z=torch.cat(parts,0).to(device); candidates=set();similarities={}
    for i in tqdm(range(0,len(z),256),desc="P0: embedding neighbours",unit="batch"):
        sim=z[i:i+256]@z.T
        for row in range(len(sim)):
            sim[row,i+row]=-1
        vals,nn=torch.topk(sim,min(topk,len(z)-1),dim=1)
        for row in range(len(nn)):
            for j,cos in zip(nn[row].tolist(),vals[row].tolist()):
                if cos<cosine_floor:continue
                a,b=sorted([indices[i+row],indices[j]])
                candidates.add((a,b));similarities[(a,b)]=max(similarities.get((a,b),-1),float(cos))
    del z,net
    return candidates,similarities

def orb_verified(img1,img2):
    """RANSAC geometric inliers: a transform of the same photographed image, not just similar fruit."""
    a=cv2.cvtColor(img1,cv2.COLOR_RGB2GRAY);b=cv2.cvtColor(img2,cv2.COLOR_RGB2GRAY)
    orb=cv2.ORB_create(nfeatures=750,fastThreshold=8)
    p1,d1=orb.detectAndCompute(a,None);p2,d2=orb.detectAndCompute(b,None)
    if d1 is None or d2 is None or len(p1)<10 or len(p2)<10:return 0,0.
    matches=cv2.BFMatcher(cv2.NORM_HAMMING).knnMatch(d1,d2,k=2)
    good=[m for pair in matches if len(pair)==2 for m,n in [pair] if m.distance<.70*n.distance]
    if len(good)<10:return len(good),0.
    src=np.float32([p1[m.queryIdx].pt for m in good]);dst=np.float32([p2[m.trainIdx].pt for m in good])
    H,inliers=cv2.findHomography(src,dst,cv2.RANSAC,3.)
    if H is None or inliers is None:return len(good),0.
    return len(good),float(inliers.mean())

def scan(root,max_images=None,embedding_screen=False):
    root=Path(root).resolve()
    paths=sorted(p for p in root.rglob('*') if p.suffix.lower() in EXT and p.is_file())
    if max_images:paths=paths[:max_images]
    if not paths:raise RuntimeError(f'No source images in {root}. Provide the ORIGINAL collection root.')
    rows=[];images=[]
    for p in tqdm(paths,desc="P0: inspect original images",unit="image"):
        rel=p.relative_to(root);parts=rel.parts
        try:
            from PIL import Image
            with Image.open(p) as original_image:
                raw_width,raw_height=original_image.size
            im=image_read(p); f,status=foreground(im)
            if np.var(im)<4:status='nearly_constant'
            dh,ph=hashes(im)
        except Exception as e:
            im=None;dh=ph=None;raw_width=raw_height=None;status='unreadable:'+str(e)[:100]
        rows.append(dict(source_id=idhash(str(rel),digest(p))[:20],relative_path=str(rel),
                         file_sha256=digest(p),variety=parts[0] if len(parts)>=3 else 'UNKNOWN',
                         native_size_label=parts[1] if len(parts)>=3 else 'UNKNOWN',
                         mask_status=status, dhash=dh,phash=ph,
                         width=raw_width,height=raw_height))
        images.append(im)
    df=pd.DataFrame(rows);d=DSU(len(df));ambiguous=set();ledger=[]
    for _,ids in df.groupby('file_sha256',sort=True).groups.items():
        ids=list(ids)
        for j in ids[1:]:d.union(ids[0],j)
    valid=[i for i,x in enumerate(images) if x is not None]
    cand=set()
    for col in ['dhash','phash']:
        vals=[df.loc[i,col] for i in valid]
        for a,b in pairs_from_hashes(vals):
            ia,ib=valid[a],valid[b]
            if ia==ib:continue
            dist=(int(df.loc[ia,col])^int(df.loc[ib,col])).bit_count()
            if dist<=7:cand.add((ia,ib))
    embedding_similarity={}
    if embedding_screen:
        print("P0: finding visually similar photographs (ResNet-18 embeddings)",flush=True)
        extra,embedding_similarity=embedding_candidates(images)
        cand.update(extra)
    for a,b in tqdm(sorted(cand),desc="P0: verify duplicate candidates",unit="pair"):
        if d.find(a)==d.find(b):continue
        im1,im2=images[a],images[b]
        g1=cv2.cvtColor(im1,cv2.COLOR_RGB2GRAY);g2=cv2.cvtColor(im2,cv2.COLOR_RGB2GRAY)
        sim=ssim(g1,g2,data_range=255)
        mae=float(np.abs(im1.astype('float32')-im2.astype('float32')).mean())

        orbm,orbf=(0,0.)
        if sim>=.80 or embedding_similarity.get((a,b),0)>.996:
            orbm,orbf=orb_verified(im1,im2)
        geometry=orbm>=15 and orbf>=.70
        if (sim>=.94 and mae<=18) or geometry:status='linked';d.union(a,b)
        elif (sim>=.82 and mae<=33) or (orbm>=10 and orbf>=.45):status='ambiguous';ambiguous.add(a);ambiguous.add(b)
        else:status='not_linked'
        ledger.append(dict(source_a=df.loc[a,'source_id'],source_b=df.loc[b,'source_id'],
                           ssim=sim,mae=mae,orb_good_matches=orbm,orb_ransac_inlier_fraction=orbf,
                           embedding_cosine=embedding_similarity.get((a,b),np.nan),decision=status))
    df['source_group']=[str(d.find(i)) for i in range(len(df))]

    quarantined={d.find(x) for x in ambiguous}
    df['eligibility']= [ 'eligible' if (row.mask_status=='ok' and d.find(i) not in quarantined)
                          else ('ambiguous_near_duplicate' if d.find(i) in quarantined else row.mask_status)
                          for i,row in df.iterrows()]
    return df,pd.DataFrame(ledger)

def allocate(df,seed=40721):
    groups=[]
    for grp,g in df[df.eligibility=='eligible'].groupby('source_group'):
        strata=Counter(zip(g.variety,g.native_size_label))
        groups.append((grp,len(g),strata))
    if len(groups)<20:raise RuntimeError('Not enough independent eligible source groups')
    rng=np.random.default_rng(seed)
    order=rng.permutation(len(groups))
    groups=[groups[i] for i in order]
    groups.sort(key=lambda g:-g[1])
    fractions={'train':.60,'development':.15,'calibration':.10,'locked_test':.15}
    strata_count=Counter()
    for _,_,s in groups:strata_count.update(s)
    state={x:Counter() for x in fractions};numbers=Counter();assignment={}
    for grp,n,strata in groups:
        scores={}
        for split,frac in fractions.items():

            penalty=0
            for st,c in strata.items():
                target=strata_count[st]*frac
                before=(state[split][st]-target)**2
                after=(state[split][st]+c-target)**2
                penalty+=(after-before)/max(strata_count[st],1)
            penalty+=.7*(((numbers[split]+1)-len(groups)*frac)**2-(numbers[split]-len(groups)*frac)**2)/len(groups)
            scores[split]=penalty
        best=min(scores,key=scores.get);assignment[grp]=best;numbers[best]+=1;state[best].update(strata)
    df=df.copy();df['split']=df.source_group.map(assignment)
    if any(numbers[s]<2 for s in fractions):raise RuntimeError('Invalid allocated groups '+str(numbers))
    assert df.groupby('source_group').split.nunique(dropna=True).le(1).all()
    return df


"""Prespecified TRAIN-ONLY generator QC: no access to held-out alterations."""
from pathlib import Path
from tqdm.auto import tqdm
import numpy as np,pandas as pd


def run_training_qc(df,root,out,n=40):
    out=Path(out);out.mkdir(parents=True,exist_ok=True)
    file=out/'training_generator_qc.csv'
    if file.exists():return pd.read_csv(file)
    rows=df[df.split=='train'].sort_values('source_id').head(n).to_dict('records')
    if len(rows)<6:raise RuntimeError('Insufficient train-only images for generator feasibility checks')
    ledger=[]
    for row in rows:
        im,fg,status=load_source(row,root)
        if status!='ok':continue
        for fam in FAMILIES_TRAIN:
            for sev in SEVERITIES:
                for mode in ['generic','fruit']:
                    rng=make_rng('train-qc',row['source_id'],fam,sev)
                    altered,mask=train_generate(im,fg,fam,sev,rng,mode)
                    ledger.append({'source_id':row['source_id'],'method':fam,'severity':sev,'mode':mode,
                                   **change_stats(im,altered,mask,fg)})
    frame=pd.DataFrame(ledger)
    if frame.empty or not frame.nonempty.all() or (frame.off_fruit_pixels>0).any():
        raise RuntimeError('TRAIN-ONLY QUALITY GATE FAILED: empty or background edits')

    balance=frame.groupby(['method','severity','mode']).agg(
        mean_support=('support_fraction','mean'),mean_change=('mean_abs_change','mean')).reset_index()
    frame.to_csv(file,index=False)
    balance.to_csv(out/'generator_treatment_balance.csv',index=False)
    flags=[]
    for (fam,sev),g in balance.groupby(['method','severity']):
        values=g.set_index('mode')
        for key in ['mean_support','mean_change']:
            lo=float(values.loc['generic',key]);hi=float(values.loc['fruit',key])
            rel=abs(lo-hi)/max((lo+hi)/2,1e-8)
            if rel>.25:flags.append({'family':fam,'severity':sev,'metric':key,'relative_difference':rel})
    write_json(out/'generator_qc_findings.json',{'n_original_train_sources':len(rows),
              'status':'PASS_WITH_WARNINGS' if flags else 'PASS',
              'balance_warnings':flags,
              'note':'A >25% difference triggers an explicit confound warning, not a hidden post-test adjustment.'})
    return frame


def build_artifact_traps(df,root,out):
    """Locked-test diagnostics, separate from the official 15-variant endpoint.
    Call only AFTER creating FINAL_TEST_USED.flag. All rows are label 0 diagnostics.
    """
    import cv2,numpy as np

    out=Path(out);saved=out/'artifact_traps_manifest.csv'
    if saved.is_file():return pd.read_csv(saved)
    rows=[]
    for r in tqdm(df[df.split=='locked_test'].to_dict('records'),desc="P6: artifact controls",unit="image"):
        im,fg,status=load_source(r,root)
        if status!='ok':raise RuntimeError('Test foreground processing drift in trap audit')
        variants={'render_roundtrip':im.copy(),
                  'jpeg_roundtrip':benign(im,1,r['source_id']+'trap')}

        edge=(cv2.dilate(fg.astype('uint8'),np.ones((3,3),np.uint8))-
              cv2.erode(fg.astype('uint8'),np.ones((3,3),np.uint8))).astype(bool)
        z=im.copy();z[edge]=np.clip(z[edge].astype('float32')*.985,0,255).astype('uint8')
        variants['boundary_pipeline_only']=z
        z=im.copy();z[fg]=np.clip(z[fg].astype('float32')*[1.015,.993,.99],0,255).astype('uint8')
        variants['global_chroma_control']=z
        save_mask(out/'foreground'/f'{r["source_id"]}.png',fg)
        for kind,z in variants.items():
            sid=r['source_id']+'__'+kind
            path=out/'images'/f'{sid}.png';mask=out/'masks'/f'{sid}.png'
            save_png(path,z);save_mask(mask,np.zeros(fg.shape,'uint8'))
            rows.append(dict(sample_id=sid,source_id=r['source_id'],source_group=r['source_group'],
                             family='none',severity='none',kind=kind,label=0,
                             image_path=str(path),mask_path=str(mask),
                             fg_path=str(out/'foreground'/f'{r["source_id"]}.png')))
    frame=pd.DataFrame(rows);frame.to_csv(saved,index=False)
    return frame


In [ ]:
"""Matched ResNet18 U-Net segmentation model and training/evaluation helpers."""
from __future__ import annotations
from pathlib import Path
import copy,os,time,random
from datetime import datetime
import pandas as pd
from tqdm.auto import tqdm
import numpy as np,cv2,torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset,DataLoader
from torchvision.models import resnet18,ResNet18_Weights


class Decoder(nn.Module):
    def __init__(self,pretrained=True):
        super().__init__()
        if pretrained:
            net=resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
        else:net=resnet18(weights=None)
        self.stem=nn.Sequential(net.conv1,net.bn1,net.relu)
        self.pool=net.maxpool
        self.layer1=net.layer1;self.layer2=net.layer2;self.layer3=net.layer3;self.layer4=net.layer4
        def up(a,b,c):return nn.Sequential(nn.Conv2d(a+b,c,3,padding=1,bias=False),nn.BatchNorm2d(c),nn.ReLU(inplace=True),
                                           nn.Conv2d(c,c,3,padding=1,bias=False),nn.BatchNorm2d(c),nn.ReLU(inplace=True))
        self.up3=up(512,256,256);self.up2=up(256,128,128);self.up1=up(128,64,64);self.up0=up(64,64,48)
        self.head=nn.Sequential(nn.Conv2d(48,32,3,padding=1),nn.ReLU(),nn.Conv2d(32,1,1))
    def forward(self,x):
        a=self.stem(x);b=self.layer1(self.pool(a));c=self.layer2(b);d=self.layer3(c);e=self.layer4(d)
        z=self.up3(torch.cat([F.interpolate(e,size=d.shape[-2:],mode='bilinear',align_corners=False),d],1))
        z=self.up2(torch.cat([F.interpolate(z,size=c.shape[-2:],mode='bilinear',align_corners=False),c],1))
        z=self.up1(torch.cat([F.interpolate(z,size=b.shape[-2:],mode='bilinear',align_corners=False),b],1))
        z=self.up0(torch.cat([F.interpolate(z,size=a.shape[-2:],mode='bilinear',align_corners=False),a],1))
        return F.interpolate(self.head(z),size=x.shape[-2:],mode='bilinear',align_corners=False)

class Samples(Dataset):
    def __init__(self,df,raw_root,mode='generic',split='train',seed=0,epoch=0,max_items=None,as_bytes=False):
        self.rows=df[df.split==split].to_dict('records')
        if max_items:self.rows=self.rows[:max_items]
        self.raw_root=Path(raw_root);self.mode=mode;self.seed=seed;self.epoch=epoch;self.split=split
        self.as_bytes=as_bytes
        self.factor=3 if split=='train' else 1+2+len(FAMILIES_TRAIN)*len(SEVERITIES)
    def __len__(self):return len(self.rows)*self.factor
    def __getitem__(self,idx):
        row=self.rows[idx//self.factor];j=idx%self.factor
        im,fg,status=load_source(row,self.raw_root)
        if status!='ok':raise RuntimeError(f'Mask changed {row["source_id"]}: {status}')
        rng=make_rng('training',self.seed,self.epoch,row['source_id'],j)
        target=np.zeros(fg.shape,np.float32)
        if self.split=='train':
            kind=j
            if kind==1:
                method=FAMILIES_TRAIN[int(rng.integers(len(FAMILIES_TRAIN)))];sev=SEVERITIES[int(rng.integers(3))]
                im,mask=train_generate(im,fg,method,sev,rng,self.mode);target=mask.astype('float32')
            elif kind==2:
                im=benign(im,int(rng.integers(0,3)),row['source_id']+'train'+str(self.epoch))
        else:
            if j in [1,2]:im=benign(im,j-1,row['source_id']+'dev')
            elif j>=3:
                n=j-3;method=FAMILIES_TRAIN[n//3];sev=SEVERITIES[n%3]
                im,mask=train_generate(im,fg,method,sev,rng,self.mode)
                target=mask.astype('float32')
        x=(torch.from_numpy(np.ascontiguousarray(im.transpose(2,0,1))) if self.as_bytes else
           torch.from_numpy(np.ascontiguousarray(im.transpose(2,0,1).astype('float32')/255)))
        return x,torch.from_numpy(target[None]),torch.from_numpy(fg[None].astype('float32'))

def seg_loss(logits,y,fg):
    prob=torch.sigmoid(logits)
    bce=(F.binary_cross_entropy_with_logits(logits,y,reduction='none')*fg).sum()/fg.sum().clamp_min(1)
    inter=(prob*y*fg).sum((2,3));union=((prob+y)*fg).sum((2,3))
    dice=(1-(2*inter+1)/(union+1)).mean()
    return bce+dice

def score_batch(prob,fg):
    out=[]
    for z,f in zip(prob,fg):
        scores=z[0][f[0]>.5].flatten()
        out.append(float(torch.quantile(scores,.99)) if scores.numel()>100 else float('nan'))
    return out

def seed_everything(seed):
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)
    if torch.cuda.is_available():torch.cuda.manual_seed_all(seed)



def cpu_quota():
    """Honor container CPU quotas rather than relying only on os.cpu_count()."""
    available=len(os.sched_getaffinity(0)) if hasattr(os,'sched_getaffinity') else (os.cpu_count() or 2)
    try:
        quota,period=Path('/sys/fs/cgroup/cpu.max').read_text().split()[:2]
        if quota!='max':available=min(available,max(1,int(float(quota)/float(period))))
    except (OSError,ValueError):pass
    return max(1,available)







V4_LOADER_WORKERS=0
V4_PREFETCH_MODE='background thread (bounded queue, no shared memory)'
V4_PIN_MEMORY=torch.cuda.is_available()
V4_GPU_VALIDATION=True
V4_STATUS_LAST=0.
V4_STATUS={}

def live_update(force=False,**kwargs):
    """Atomic small status JSON, readable by a second Jupyter kernel."""
    global V4_STATUS_LAST
    V4_STATUS.update(kwargs)
    now=time.monotonic()
    if not force and now-V4_STATUS_LAST<2.0:return
    V4_STATUS_LAST=now
    payload=dict(V4_STATUS,updated_at=datetime.now().astimezone().isoformat(),
                 status_writer='GPU-aware V4',writer_pid=os.getpid())
    write_json(OUT/'live_progress.json',payload)

class ThreadPrefetchLoader:
    """Prefetch CPU batches concurrently with CUDA compute, with no shm.

    The underlying DataLoader has num_workers=0; its sampler and collation are
    unchanged. Only when batches are *prepared* differs. Queue capacity two
    keeps host-memory use bounded. All synthetic random draws are keyed to
    source, seed and epoch, so the examples are identical to the unprefetched
    pipeline. Exception propagation is explicit instead of silently hanging.
    """
    def __init__(self, loader, buffer_batches=2):
        self.loader=loader
        self.buffer_batches=buffer_batches

    def __len__(self):
        return len(self.loader)

    def __iter__(self):
        from queue import Queue, Full
        from threading import Thread, Event
        import traceback as _traceback
        q=Queue(maxsize=self.buffer_batches)
        stop=Event()

        def enqueue(kind,payload):
            while not stop.is_set():
                try:
                    q.put((kind,payload),timeout=.2)
                    return
                except Full:
                    continue

        def producer():
            try:


                cv2.setNumThreads(1)
                for batch in self.loader:
                    if stop.is_set():break
                    enqueue('batch',batch)
            except BaseException as exc:
                enqueue('error',(exc,_traceback.format_exc()))
            finally:
                enqueue('done',None)

        worker=Thread(target=producer,name='datepalm-v4-cpu-prefetch',daemon=True)
        worker.start()
        try:
            while True:
                kind,value=q.get()
                if kind=='done':break
                if kind=='error':
                    exc,tb=value
                    raise RuntimeError('CPU preprocessing/prefetch failed:\n'+tb) from exc
                yield value
        finally:
            stop.set()
            worker.join(timeout=2)

def v4_loader(dataset,batch_size,shuffle=False,seed=None):
    """Deterministic, shm-free DataLoader overlapped with GPU computation."""
    settings=dict(batch_size=batch_size,shuffle=shuffle,
                  num_workers=0,pin_memory=V4_PIN_MEMORY)
    if seed is not None:settings['generator']=torch.Generator().manual_seed(seed)
    return ThreadPrefetchLoader(DataLoader(dataset,**settings),buffer_batches=2)

def v4_worker_init(_):

    cv2.setNumThreads(1)

@torch.no_grad()
def gpu_quantile_99(prob,fg):
    """Batch-exact torch.quantile(.99) using GPU top-k inside valid foreground.

    Works on each image's *variable* number of fruit pixels, with default
    linear quantile interpolation. This is not an approximation to the 99th
    percentile and does not use the background in the ranking.
    """
    pixels=prob[:,0].flatten(1)
    valid=fg[:,0].flatten(1).bool()
    counts=valid.sum(dim=1)
    if (counts<=100).any():
        raise RuntimeError('Insufficient foreground for quantitative validation')
    fractional=(counts-1).to(torch.float64)*.99
    lower=torch.floor(fractional).long()
    upper=torch.ceil(fractional).long()
    k=int((counts-lower).max().item())
    values=pixels.masked_fill(~valid,-float('inf')).topk(k,dim=1,sorted=True).values
    lo=values.gather(1,(counts-1-lower)[:,None]).flatten()
    hi=values.gather(1,(counts-1-upper)[:,None]).flatten()
    frac=(fractional-lower).to(prob.dtype)
    return (lo*(1-frac)+hi*frac)

def validate_gpu_quantile():
    """Fail fast if GPU scoring fails to agree with original CPU scoring."""
    if not torch.cuda.is_available():return
    generator=torch.Generator().manual_seed(14321)
    p=torch.rand((7,1,24,28),generator=generator)
    masks=torch.rand((7,1,24,28),generator=generator)>.2
    reference=torch.tensor(score_batch(p,masks.float()))
    actual=gpu_quantile_99(p.cuda(),masks.cuda()).cpu()
    if not torch.allclose(reference,actual,rtol=0,atol=3e-6):
        raise AssertionError('GPU scoring disagrees with original per-image torch.quantile')

@torch.no_grad()
def prepare_gpu_dev_cache(df,root,seed,device,max_images=None):
    """Render the SAME fixed development examples once per seed, not every epoch.

    Cache byte images and foreground masks in GPU memory (~3.3 GiB for 835
    original images x 15 variants), and convert to float only per forward batch.
    Deterministic source IDs and seeds are unchanged. No held-out images used.
    """
    ds=Samples(df,root,'fruit','development',seed,max_items=max_images,as_bytes=True)
    n=len(ds)
    required=n*(3*SHAPE[0]*SHAPE[1]+SHAPE[0]*SHAPE[1])
    if str(device).startswith('cuda'):
        free,_=torch.cuda.mem_get_info()

        if required+4*1024**3>free:
            print(f'  GPU cache skipped (need {required/1024**3:.2f} GiB + reserve; available {free/1024**3:.2f} GiB)',flush=True)
            live_update(force=True,subtask='Validation streaming fallback: GPU memory reserve')
            return None
    else:return None
    live_update(force=True,subtask='Building fixed development cache on GPU',
                cache_samples_done=0,cache_samples_total=n)
    try:
        images=torch.empty((n,3,*SHAPE),dtype=torch.uint8,device=device)
        foregrounds=torch.empty((n,1,*SHAPE),dtype=torch.uint8,device=device)
    except torch.cuda.OutOfMemoryError:


        import gc
        if 'images' in locals():del images
        if 'foregrounds' in locals():del foregrounds
        gc.collect()
        torch.cuda.empty_cache()
        print('  GPU dev cache unavailable: continuing with equivalent streaming validation',flush=True)
        live_update(force=True,subtask='Validation streaming fallback: insufficient GPU memory')
        return None
    loader=v4_loader(ds,24,False)
    offset=0;clock=time.monotonic()
    try:
        for x,_,fg in loader:
            size=len(x)
            images[offset:offset+size].copy_(x.to(device,non_blocking=True))
            foregrounds[offset:offset+size].copy_(fg.to(device,non_blocking=True).to(torch.uint8))
            offset+=size
            live_update(subtask='Building fixed development cache on GPU',
                        cache_samples_done=offset,cache_samples_total=n,
                        cache_elapsed_sec=round(time.monotonic()-clock,1))
    except Exception:
        del images,foregrounds
        raise
    if offset!=n:raise RuntimeError('Incomplete GPU development cache')
    torch.cuda.synchronize()
    print(f'  GPU VALIDATION CACHE: {n:,} exact deterministic samples, {required/1024**3:.2f} GiB; built in {(time.monotonic()-clock)/60:.1f} min',flush=True)
    live_update(force=True,subtask='Development examples cached on GPU',
                cache_samples_done=n,cache_samples_total=n)
    return (images,foregrounds,ds.factor)

@torch.no_grad()
def dev_metric(model,df,root,mode,seed,device,max_images=None,gpu_cache=None):
    from sklearn.metrics import roc_auc_score
    model.eval();groups={};score=[]
    if gpu_cache is not None:
        images,foregrounds,factor=gpu_cache
        total=len(images)
        for start in range(0,total,24):
            stop=min(start+24,total)
            x=images[start:stop].float().div_(255.)
            p=torch.sigmoid(model(x))
            part=gpu_quantile_99(p,foregrounds[start:stop]).cpu().tolist()
            score.extend(part)
            live_update(subtask='Development evaluation (GPU-cached images)',
                        dev_samples_done=stop,dev_samples_total=total)
    else:
        ds=Samples(df,root,'fruit','development',seed,max_items=max_images)
        factor=ds.factor
        loader=v4_loader(ds,16,False)
        total=len(ds);finished=0
        for x,y,fg in loader:
            p=torch.sigmoid(model(x.to(device,non_blocking=True))).cpu()
            score.extend(score_batch(p,fg))
            finished+=len(x)
            live_update(subtask='Development evaluation (streamed images)',
                        dev_samples_done=finished,dev_samples_total=total)
    for i,si in enumerate(score):
        j=i%factor
        if j<=2:groups.setdefault(('negative',0),[]).append(si)
        else:groups.setdefault(((j-3)//3,(j-3)%3),[]).append(si)
    neg=groups[('negative',0)];aucs=[]
    for k in range(4):
        for sev in range(3):
            pos=groups[(k,sev)]
            aucs.append(roc_auc_score([0]*len(neg)+[1]*len(pos),neg+pos))
    return float(np.mean(aucs))

def atomic_torch_save(obj, path):
    """Avoid leaving a corrupt checkpoint when the kernel is interrupted."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + '.partial')
    torch.save(obj, temporary)
    os.replace(temporary, path)


def fit_matched(df, raw_root, out, variant, seed, config, device='cpu',
                epochs_override=None, max_images=None):
    out = Path(out)
    out.mkdir(parents=True, exist_ok=True)
    best_file, last_file, done = out/'model.pt', out/'last_epoch.pt', out/'COMPLETE.json'
    if done.exists():
        if not best_file.exists():
            raise RuntimeError('Completed model is missing its checkpoint: ' + str(out))
        print(f'  RESUME: {variant} seed={seed} already finished', flush=True)
        return best_file

    seed_everything(seed)
    model = Decoder(pretrained=config['pretrained']).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=.01)
    mode = 'fruit' if variant in ['C10','C11'] else 'generic'
    use_consistency = variant in ['C01','C11']
    lam = float(config['lambda_cons']) if use_consistency else 0.
    max_epochs = epochs_override or config['max_epochs']
    start_ep, best, wait, history = 0, -1e9, 0, []
    if last_file.exists():
        state = torch.load(last_file, map_location='cpu', weights_only=False)
        if state['variant'] != variant or state['seed'] != seed or state['max_epochs'] != max_epochs:
            raise RuntimeError('Incompatible interrupted training checkpoint: ' + str(last_file))
        model.load_state_dict(state['model'])
        optimizer.load_state_dict(state['optimizer'])
        for v in optimizer.state.values():
            for k, x in v.items():
                if torch.is_tensor(x):
                    v[k] = x.to(device)
        start_ep, best, wait = int(state['epoch']), float(state['best']), int(state['wait'])
        history = state['history']
        print(f'  RESUME: {variant} seed={seed}, from epoch {start_ep+1}', flush=True)

    start_clock = time.monotonic()
    gpu_cache = None
    if start_ep < max_epochs and V4_GPU_VALIDATION and str(device).startswith('cuda'):
        validate_gpu_quantile()
        gpu_cache=prepare_gpu_dev_cache(df,raw_root,seed,device,max_images=max_images)
    live_update(force=True,stage='P2-training',variant=variant,seed=seed,
                current_epoch=start_ep,epochs_total=max_epochs,
                batches_done=0,batches_total=0,subtask='Preparing next training epoch')
    for ep in range(start_ep, max_epochs):
        seed_everything(seed*10000 + ep)
        ds = Samples(df, raw_root, mode, 'train', seed, ep, max_items=max_images)
        loader = v4_loader(ds,config['batch_size'],True,seed+ep)
        model.train()
        total = 0.
        live_update(force=True,stage='P2-training',variant=variant,seed=seed,
                    current_epoch=ep+1,epochs_total=max_epochs,batches_done=0,
                    batches_total=len(loader),subtask='Training (CPU prefetch + CUDA forward/backward)')
        bar = tqdm(loader, desc=f'{variant} / seed {seed} / epoch {ep+1}/{max_epochs}', leave=False)
        for batch_i, (x, y, fg) in enumerate(bar):
            x=x.to(device,non_blocking=True)
            y=y.to(device,non_blocking=True)
            fg=fg.to(device,non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            logits = model(x)
            loss = seg_loss(logits, y, fg)
            if use_consistency:

                torch.manual_seed(seed*100000+ep*1000+batch_i)
                factors = torch.empty((len(x),3,1,1),device=device).uniform_(.94,1.06)
                z = (x*factors).clamp(0,1)
                logits2 = model(z)
                loss = loss + lam*((torch.sigmoid(logits)-torch.sigmoid(logits2)).abs()*fg).sum()/fg.sum().clamp_min(1)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 5)
            optimizer.step()
            total += float(loss.item())
            bar.set_postfix(loss=f'{float(loss.item()):.3f}')
            live_update(subtask='Training (CPU prefetch + CUDA forward/backward)',
                        batches_done=batch_i+1,batches_total=len(loader),
                        train_loss_running=round(total/(batch_i+1),6))
        live_update(force=True,subtask='Development evaluation',dev_samples_done=0,
                    dev_samples_total=(len(gpu_cache[0]) if gpu_cache is not None else
                                       len(df[df.split=='development'])*(3+len(FAMILIES_TRAIN)*len(SEVERITIES))))
        val = dev_metric(model, df, raw_root, mode, seed, device,
                         max_images=max_images,gpu_cache=gpu_cache)
        record = dict(epoch=ep+1, train_loss=total/max(len(loader),1), dev_macro_auroc=val)
        history.append(record)
        if val > best + 1e-5:
            best, wait = val, 0
            atomic_torch_save({'state_dict':model.state_dict(), 'seed':seed, 'variant':variant,
                               'epoch':ep+1, 'dev_macro_auroc':best, 'config':config}, best_file)
        else:
            wait += 1
        atomic_torch_save({'model':model.state_dict(), 'optimizer':optimizer.state_dict(),
                           'epoch':ep+1, 'best':best, 'wait':wait, 'history':history,
                           'variant':variant, 'seed':seed, 'max_epochs':max_epochs}, last_file)
        pd.DataFrame(history).to_csv(out/'history.csv', index=False)
        live_update(force=True,subtask='Epoch checkpoint written',current_epoch=ep+1,
                    last_dev_macro_auroc=round(val,6),best_dev_macro_auroc=round(best,6),
                    last_train_loss=round(record['train_loss'],6))
        print(f'  {variant} seed={seed} epoch={ep+1}/{max_epochs} '
              f'loss={record["train_loss"]:.4f} dev_AUROC={val:.4f} '
              f'best={best:.4f} elapsed={(time.monotonic()-start_clock)/60:.1f}m', flush=True)
        if wait >= config['patience']:
            print(f'  Early stopping at epoch {ep+1} (patience {wait})',flush=True)
            break
    if not best_file.exists():
        raise RuntimeError('Training stopped without saving a valid best model')
    write_json(done, {'variant':variant,'seed':seed,'best_dev_auroc':best,
                      'epochs_finished':len(history)})
    live_update(force=True,subtask='Matched-model seed completed',variant=variant,seed=seed,
                current_epoch=len(history),epochs_total=max_epochs)
    del gpu_cache
    if str(device).startswith('cuda'):torch.cuda.empty_cache()
    return best_file


def load_model(checkpoint, device):
    ck = torch.load(checkpoint, map_location='cpu', weights_only=False)
    model = Decoder(pretrained=False)
    model.load_state_dict(ck['state_dict'], strict=True)
    return model.eval().to(device)


In [ ]:
"""Published baseline runner using Anomalib v2.2.0 implementations.

NOTE: Anomalib has SuperSimpleNet (a DIFFERENT method from original SimpleNet).
The prospective configuration records this protocol amendment and must never call it SimpleNet.
No approximate homemade baselines are represented as published implementations.
"""
from pathlib import Path
from tqdm.auto import tqdm
import os, numpy as np, pandas as pd, cv2, torch


V4_ENABLE_EFFICIENTAD_ACCELERATION = True
V4_ENABLE_TIE_EXACT_FAST_BOOTSTRAP = True



V4_ENABLE_DRAEM_SAFE_ACCELERATION = True
V4_DRAEM_DTD_CACHE_MIB = 128


BASELINES=['Patchcore','Padim','EfficientAd','Draem','Supersimplenet']





BASELINE_FIT_CONFIG = {
    'Patchcore': {'train_batch_size': 8, 'eval_batch_size': 8, 'max_epochs': 1},
    'Padim': {'train_batch_size': 8, 'eval_batch_size': 8, 'max_epochs': 1},
    'EfficientAd': {'train_batch_size': 1, 'eval_batch_size': 8,
                    'max_epochs': 1000, 'max_steps': 70000},
    'Draem': {'train_batch_size': 8, 'eval_batch_size': 8, 'max_epochs': 60, 'check_val_every_n_epoch': 5},
    'Supersimplenet': {'train_batch_size': 8, 'eval_batch_size': 8, 'max_epochs': 60, 'check_val_every_n_epoch': 5},
}

def baseline_fit_spec(name):
    """Return and validate reproducible method-specific training settings."""
    if name not in BASELINE_FIT_CONFIG:
        raise ValueError('Unsupported baseline method: '+str(name))
    spec = dict(BASELINE_FIT_CONFIG[name])
    if name=='EfficientAd' and spec['train_batch_size']!=1:
        raise RuntimeError('EfficientAD requires training batch size 1')
    if spec['max_epochs']<1 or spec['train_batch_size']<1:
        raise RuntimeError('Invalid baseline training budget')
    return spec


def requirements_check():
    try:
        import anomalib
        from importlib.metadata import version
        ver=version('anomalib')
        if ver!='2.2.0':raise RuntimeError(f'anomalib==2.2.0 required; found {ver}')
        from anomalib import models
        missing=[n for n in BASELINES if not hasattr(models,n)]
        if missing:raise RuntimeError('Missing anomaly library models: '+str(missing))
        from anomalib.data import Folder
        from anomalib.engine import Engine
        return ver
    except ImportError as e:
        raise RuntimeError('Install pinned anomalib==2.2.0 and its compatible dependencies. '+str(e))




def build_training_tree(df,raw_root,base,seed,max_items=None):
    """Build/reuse baseline images from TRAIN+DEVELOPMENT only, never the test."""
    base=Path(base)
    if max_items is None:
        shared=OUT/'data'/'baseline_shared'
        legacy=OUT/'models'/'Patchcore'/'1049'/'datamodule'


        if (shared/'_prepared.ok').is_file():
            base=shared
        elif (legacy/'_prepared.ok').is_file():
            base=legacy
            print('  BASELINE DATA: reusing legacy prepared tree for all baselines',flush=True)
        else:
            base=shared
    marker=base/'_prepared.ok'
    if marker.is_file():return base
    rows_train=df[df.split=='train'].to_dict('records')
    rows_dev=df[df.split=='development'].to_dict('records')
    if max_items:
        rows_train=rows_train[:max_items];rows_dev=rows_dev[:max_items]
    for row in tqdm(rows_train,desc='Prepare shared baseline training data',unit='image',leave=False):
        save_png(base/'train/good'/f'{row["source_id"]}.png',image_read(Path(raw_root)/row['relative_path']))
    for row in tqdm(rows_dev,desc='Prepare shared baseline development data',unit='image',leave=False):
        im,fg,status=load_source(row,raw_root)
        if status!='ok':raise RuntimeError(f'Foreground mask mismatch: {row["source_id"]}')
        save_png(base/'eval/good'/f'{row["source_id"]}.png',im)
        for fam in FAMILIES_TRAIN:
            for sev in SEVERITIES:
                rng=make_rng('anomalib-val',row['source_id'],fam,sev)
                z,m=train_generate(im,fg,fam,sev,rng,'fruit')
                fn=f'{row["source_id"]}_{fam}_{sev}.png'
                save_png(base/'eval/defect'/fn,z)
                save_mask(base/'eval/mask'/fn,m)
    marker.parent.mkdir(parents=True,exist_ok=True)
    marker.write_text('Source-partition safe, only train and development images used.\n')
    return base









PATCHCORE_CANDIDATES_PER_IMAGE = 16
PATCHCORE_CORESET_RATIO = 0.10
PATCHCORE_AMENDMENT = (
    'Resource-bounded PatchCore variant: deterministically select 16 normal '
    'training patches per original image before the library 10%-ratio '
    'k-center-greedy coreset. No source images omitted; coreset selection, '
    'pretrained backbone and inference otherwise remain Anomalib 2.2.0. '
    'This is not an unmodified published PatchCore configuration.'
)

class PatchCandidateStore(list):
    """Bounded GPU store satisfying Anomalib 2.2.0 embedding_store API.

    No candidate augmentation or label information; per-image locations
    are sampled uniformly without replacement from existing model features.
    The resulting tensors are exactly FP32 as in the original implementation.
    """
    def __init__(self,per_image,seed):
        super().__init__()
        self.per_image=int(per_image)
        self.rng=np.random.default_rng(int(seed))
        self.batch_size=None
        self.images_seen=0
        self.candidates_seen=0
        self.patches_per_image=None
        self.channels=None

    def append(self,embedding):
        if self.batch_size is None or self.batch_size<1:
            raise RuntimeError('PatchCore training batch size was not supplied')
        batch=int(self.batch_size)
        if embedding.ndim!=2 or embedding.shape[0]%batch:
            raise RuntimeError('PatchCore feature layout changed: cannot split by image')
        n=embedding.shape[0]//batch
        d=int(embedding.shape[1])
        k=min(self.per_image,n)
        if k<1:
            raise RuntimeError('Empty PatchCore feature map')
        if self.patches_per_image is not None and self.patches_per_image!=n:
            raise RuntimeError('PatchCore training patch geometry changed mid-run')
        self.patches_per_image=n
        self.channels=d
        ids=np.stack([self.rng.choice(n,size=k,replace=False)
                      for _ in range(batch)]).astype(np.int64)
        positions=torch.from_numpy(ids).to(embedding.device)
        subset=torch.gather(embedding.reshape(batch,n,d),1,
                            positions.unsqueeze(-1).expand(-1,-1,d))
        super().append(subset.reshape(batch*k,d).contiguous())
        self.images_seen+=batch
        self.candidates_seen+=batch*k


def enable_budgeted_patchcore(model,seed):
    """Train-only instrumentation; serialized Lightning state_dict is unchanged."""
    from types import MethodType
    store=PatchCandidateStore(PATCHCORE_CANDIDATES_PER_IMAGE,seed)
    original_forward=model.model.forward
    def budgeted_forward(self,input_tensor):


        if self.training:
            store.batch_size=int(input_tensor.shape[0])
        return original_forward(input_tensor)
    model.model.embedding_store=store
    model.model.forward=MethodType(budgeted_forward,model.model)
    if abs(float(model.coreset_sampling_ratio)-PATCHCORE_CORESET_RATIO)>1e-10:
        raise RuntimeError('Unexpected PatchCore coreset ratio; audit required')
    return store


def baseline_resource_preflight(name,device,training_images):
    """Explicit RAM/GPU budget; fail before extraction rather than after it."""
    import shutil
    disk_free=shutil.disk_usage(OUT).free
    if disk_free<4*1024**3:
        raise RuntimeError('Less than 4 GiB free disk space. Preserve outputs and expand disk before resuming.')
    if not str(device).startswith('cuda'):
        raise RuntimeError('Official baseline run requires CUDA')
    torch.cuda.empty_cache()
    free,total=torch.cuda.mem_get_info()
    if free<6*1024**3:
        raise RuntimeError(f'Only {free/1024**3:.1f} GiB GPU free before {name}; '
                           'another job may be holding memory. Restart the failed kernel.')
    if name=='Padim':


        padim_store=int(training_images)*100*64*64*4
        estimate=2*padim_store+4*1024**3
        print(f'  PADIM PREFLIGHT: estimated aggregation peak '
              f'{estimate/1024**3:.1f} GiB (including 4 GiB overhead); '
              f'GPU free={free/1024**3:.1f} GiB',flush=True)
        if free<estimate:
            raise RuntimeError('Unmodified PaDiM aggregation predicted to exceed GPU '
                               'memory; will not silently change its feature distribution.')
    if name=='Patchcore':
        expected_candidates=int(training_images)*PATCHCORE_CANDIDATES_PER_IMAGE
        print(f'  PATCHCORE PREFLIGHT: {training_images:,} train images × '
              f'{PATCHCORE_CANDIDATES_PER_IMAGE} candidate patches = '
              f'{expected_candidates:,} pre-coreset patches; '
              f'GPU free={free/1024**3:.1f} GiB',flush=True)
        live_update(force=True,subtask='PatchCore candidate-pool budget preflight',
                    patchcore_candidates_expected=expected_candidates)

class BaselineProgressCallback:
    """Factory returning a true Lightning callback without a module-level import."""
    @staticmethod
    def create(name,seed):
        from lightning.pytorch.callbacks import Callback
        class _Callback(Callback):


            def __init__(self):
                import time
                self.last_end=None
                self.batch_start=None
                self.gap_samples=[]
                self.step_samples=[]
                self.rate_start=None
                self.rate_step=None
            def on_train_batch_start(self,trainer,pl_module,batch,batch_idx):
                import time
                now=time.perf_counter()
                if self.rate_start is None:
                    self.rate_start=now
                    self.rate_step=int(trainer.global_step)
                if self.last_end is not None:
                    self.gap_samples.append(max(0,now-self.last_end))
                self.batch_start=now
            def on_train_batch_end(self,trainer,pl_module,outputs,batch,batch_idx):
                import time
                now=time.perf_counter()
                if self.batch_start is not None:
                    self.step_samples.append(max(0,now-self.batch_start))
                self.last_end=now
                if len(self.step_samples)>=200:
                    live_update(force=True,
                        baseline_mean_batch_step_wall_ms=round(1000*sum(self.step_samples)/len(self.step_samples),2),
                        baseline_mean_interbatch_gap_ms=(round(1000*sum(self.gap_samples)/len(self.gap_samples),2)
                            if self.gap_samples else None))
                    if name=='Draem' and V4_ENABLE_DRAEM_SAFE_ACCELERATION:
                        cache=getattr(_v4_draem_safe_cache_status,'cache',None)
                        if cache is not None:
                            live_update(force=True,
                                draem_texture_cache_hits=int(cache.hits),
                                draem_texture_cache_misses=int(cache.misses),
                                draem_texture_cache_mib=round(cache.bytes/1048576,1))

                    current_step=int(trainer.global_step)
                    delta=max(0,current_step-self.rate_step)
                    duration=max(1e-9,now-self.rate_start)
                    rate=delta/duration
                    target=int(trainer.max_steps) if trainer.max_steps is not None else -1
                    live_update(force=True,
                        baseline_measured_steps_per_sec=round(rate,3),
                        baseline_estimated_remaining_hours=(round(max(0,target-current_step)/rate/3600,2)
                            if target>0 and rate>0 else None))
                    self.rate_start=now
                    self.rate_step=current_step
                    self.step_samples.clear()
                    self.gap_samples.clear()
                max_steps=int(trainer.max_steps) if trainer.max_steps is not None else -1
                live_update(stage='P3-baseline',variant=name,seed=seed,
                            subtask='Baseline feature extraction / training',
                            batches_done=int(batch_idx)+1,
                            batches_total=int(trainer.num_training_batches),
                            baseline_steps_done=int(trainer.global_step),
                            baseline_steps_total=max_steps if max_steps>0 else 0,
                            baseline_epoch_number=int(trainer.current_epoch)+1,
                            baseline_epochs_total=int(trainer.max_epochs))
            def on_validation_start(self,trainer,pl_module):
                nv=trainer.num_val_batches
                if isinstance(nv,(tuple,list)):nv=sum(nv)
                live_update(force=True,stage='P3-baseline',variant=name,seed=seed,
                            baseline_val_batches_done=0,baseline_val_batches_total=int(nv or 0),
                            subtask=('PatchCore k-center coreset selection'
                                     if name=='Patchcore' else 'Baseline fitting / validation'))
            def on_validation_batch_end(self,trainer,pl_module,outputs,batch,batch_idx,dataloader_idx=0):
                live_update(stage='P3-baseline',variant=name,seed=seed,
                            subtask='Baseline validation',baseline_val_batches_done=batch_idx+1)
        return _Callback()







from contextlib import contextmanager

@contextmanager
def quiet_patchcore_coreset_progress(name,seed):
    import importlib
    import time as _time
    sampler_module=importlib.import_module('anomalib.models.components.sampling.k_center_greedy')
    if not callable(getattr(sampler_module,'tqdm',None)):
        raise RuntimeError('Anomalib k-center tqdm location changed; review adapter before fitting')
    previous=sampler_module.tqdm

    def file_only_progress(iterable,*args,**kwargs):

        total=int(kwargs.get('total') or (len(iterable) if hasattr(iterable,'__len__') else 0))
        start=_time.monotonic()
        last=start
        live_update(force=True,stage='P3-baseline',variant=name,seed=seed,
                    subtask='PatchCore k-center coreset selection',
                    baseline_coreset_done=0,baseline_coreset_total=total,
                    baseline_coreset_eta_seconds=None)
        for i,value in enumerate(iterable,1):
            yield value
            now=_time.monotonic()
            if now-last>=2.5 or i==total:
                elapsed=max(now-start,1e-6)
                eta=(total-i)*elapsed/i if total and i else None
                live_update(force=True,stage='P3-baseline',variant=name,seed=seed,
                            subtask='PatchCore k-center coreset selection',
                            baseline_coreset_done=i,baseline_coreset_total=total,
                            baseline_coreset_eta_seconds=(round(eta) if eta is not None else None))
                last=now

    sampler_module.tqdm=file_only_progress
    try:
        yield
    finally:
        sampler_module.tqdm=previous


def build_model(name,extra):
    from anomalib import models
    cls=getattr(models,name)
    if name=='Draem':
        if not extra.get('dtd_dir') or not Path(extra['dtd_dir']).exists():
            raise RuntimeError('DRAEM requires local Describable Textures Dataset path --dtd-dir')
        return cls(dtd_dir=extra['dtd_dir'])
    if name=='EfficientAd':
        if not extra.get('imagenette_dir') or not Path(extra['imagenette_dir']).exists():
            raise RuntimeError('EfficientAD requires local auxiliary ImageNette data --imagenette-dir')
        return cls(imagenet_dir=extra['imagenette_dir'])
    return cls()





BASELINE_RECOVERY_INTERVAL_STEPS = 4000

def baseline_recovery_callback(base, name, seed):
    from lightning.pytorch.callbacks import Callback
    checkpoint = Path(base) / 'in_progress_full_state.ckpt'
    class RecoveryCallback(Callback):
        def on_train_batch_end(self, trainer, pl_module, outputs, batch, batch_idx):
            if name not in {'EfficientAd','Draem','Supersimplenet'}:
                return
            current=int(trainer.global_step)
            if current <= 0 or current % BASELINE_RECOVERY_INTERVAL_STEPS:
                return


            temp=checkpoint.with_suffix('.ckpt.incomplete')
            trainer.save_checkpoint(str(temp),weights_only=False)
            os.replace(temp,checkpoint)
            live_update(force=True,stage='P3-baseline',variant=name,seed=seed,
                        subtask='Saved rotating full-state recovery checkpoint',
                        baseline_recovery_step=current)
    return RecoveryCallback()

def fit_baseline(df,raw_root,base,name,seed,device,extra,max_items=None):
    requirements_check()
    from anomalib.data import Folder
    from anomalib.engine import Engine
    import gc
    base=Path(base)
    ckpt=base/'baseline.ckpt';base.mkdir(parents=True,exist_ok=True)
    marker=base/'COMPLETE.json'
    if marker.exists():
        if not ckpt.exists():raise RuntimeError('Completed baseline missing checkpoint: '+str(ckpt))
        print(f'  RESUME: baseline {name} seed={seed} already completed',flush=True)
        return ckpt
    root=build_training_tree(df,raw_root,base/'datamodule',seed,max_items)
    train_size=(min(max_items,len(df[df.split=='train'])) if max_items
                else len(df[df.split=='train']))
    baseline_resource_preflight(name,device,train_size)
    seed_everything(seed)
    fit_spec=baseline_fit_spec(name)
    print(f'  BASELINE CONFIG: {name}, train_batch_size={fit_spec["train_batch_size"]}, '
          f'eval_batch_size={fit_spec["eval_batch_size"]}, '
          f'max_epochs={fit_spec["max_epochs"]}, '
          f'max_steps={fit_spec.get("max_steps", "not set")}',flush=True)

    module=Folder(name='date_v4_'+name,root=root,normal_dir='train/good',
                  normal_test_dir='eval/good',abnormal_dir='eval/defect',
                  mask_dir='eval/mask',normal_split_ratio=0,
                  test_split_mode='from_dir',val_split_mode='from_test',
                  val_split_ratio=.5,
                  train_batch_size=fit_spec['train_batch_size'],
                  eval_batch_size=fit_spec['eval_batch_size'],
                  num_workers=0,seed=seed)
    if name=='EfficientAd' and module.train_batch_size != 1:
        raise RuntimeError('EfficientAD datamodule batch size must equal 1')
    model=build_model(name,extra)
    patchstore=None
    if name=='EfficientAd' and V4_ENABLE_EFFICIENTAD_ACCELERATION:
        enable_v4_efficientad_acceleration(model, seed)
    if name=='Draem' and V4_ENABLE_DRAEM_SAFE_ACCELERATION:
        enable_v4_draem_safe_acceleration(model,seed)
    if name=='Patchcore':
        patchstore=enable_budgeted_patchcore(model,seed)
        print('  PROTOCOL AMENDMENT: '+PATCHCORE_AMENDMENT,flush=True)
    engine_args={'default_root_dir':str(base/'engine'),
                 'accelerator':'gpu' if str(device).startswith('cuda') else 'cpu',
                 'devices':1,'logger':False,'enable_checkpointing':True,
                 'enable_progress_bar':False,
                 'callbacks':[BaselineProgressCallback.create(name,seed),
                              baseline_recovery_callback(base,name,seed)],
                 'max_epochs':fit_spec['max_epochs']}
    if 'max_steps' in fit_spec:
        engine_args['max_steps']=fit_spec['max_steps']
    if 'check_val_every_n_epoch' in fit_spec:
        engine_args['check_val_every_n_epoch']=fit_spec['check_val_every_n_epoch']
    engine=Engine(**engine_args)
    live_update(force=True,stage='P3-baseline',variant=name,seed=seed,
                subtask='Starting Anomalib baseline',batches_done=0,
                baseline_val_batches_done=0)
    recovery=base/'in_progress_full_state.ckpt'
    resume_path=(str(recovery) if recovery.exists() and
                 name in {'EfficientAd','Draem','Supersimplenet'} else None)
    if resume_path:
        print(f'  RESUME FROM FULL STATE: {resume_path}',flush=True)
    if name=='Patchcore':
        with quiet_patchcore_coreset_progress(name,seed):
            engine.fit(model=model,datamodule=module)
    else:
        engine.fit(model=model,datamodule=module,ckpt_path=resume_path)
    audit={'method':name,'seed':seed,'model_checkpoint_type':'Anomalib-2.2.0',
           'fit_spec':fit_spec,
           'accelerated_execution':bool(name=='EfficientAd' and V4_ENABLE_EFFICIENTAD_ACCELERATION),
           'acceleration_changes_math':False,
           'acceleration_scope':'remove GPU synchronizations in per-step diagnostic logging; cache identical constants and fixed boolean state',
           'draem_safe_acceleration_enabled':bool(name=='Draem' and V4_ENABLE_DRAEM_SAFE_ACCELERATION),
           'recovery_interval_optimizer_steps':BASELINE_RECOVERY_INTERVAL_STEPS,
           'validation_frequency_epochs':fit_spec.get('check_val_every_n_epoch',1),
           'training_budget_note':('EfficientAD 70k-step recipe; DRAEM and SuperSimpleNet '
                                   'capped at 60 epochs for the V4 compute budget. '
                                   'Method-specific training schedules are not identical.')}
    if patchstore is not None:
        coreset=int(model.model.memory_bank.shape[0])
        expected=train_size*min(PATCHCORE_CANDIDATES_PER_IMAGE,patchstore.patches_per_image)
        if patchstore.images_seen!=train_size or patchstore.candidates_seen!=expected:
            raise RuntimeError('PatchCore sampled training image count mismatch: '
                               f'images={patchstore.images_seen} expected={train_size}; '
                               f'candidates={patchstore.candidates_seen} expected={expected}')
        if coreset<=0 or coreset>patchstore.candidates_seen:
            raise RuntimeError('PatchCore coreset cardinality invalid')
        audit.update({'protocol_amendment':PATCHCORE_AMENDMENT,
                      'candidate_patches_per_image':PATCHCORE_CANDIDATES_PER_IMAGE,
                      'original_patches_per_image':patchstore.patches_per_image,
                      'total_candidate_patches':patchstore.candidates_seen,
                      'coreset_sampling_ratio':PATCHCORE_CORESET_RATIO,
                      'coreset_patches':coreset})
        write_json(base/'patchcore_resource_audit.json',audit)
        print(f'  PATCHCORE COMPLETE: {patchstore.candidates_seen:,} '
              f'candidate patches -> {coreset:,} coreset patches',flush=True)
    tmp=ckpt.with_suffix('.ckpt.partial')
    engine.trainer.save_checkpoint(str(tmp))
    os.replace(tmp,ckpt)
    audit['model_checkpoint_sha256']=digest(ckpt)
    write_json(marker,audit)
    recovery.unlink(missing_ok=True)
    live_update(force=True,stage='P3-baseline',variant=name,seed=seed,
                subtask='Baseline checkpoint complete')
    del engine,model,module,patchstore
    gc.collect()
    if torch.cuda.is_available():torch.cuda.empty_cache()
    return ckpt

def predict_from_checkpoint(ckpt,name,image_folder,device,extra,keep_maps=True):
    requirements_check()
    from anomalib.engine import Engine
    model=build_model(name,extra)



    engine=Engine(default_root_dir=str(Path(ckpt).parent/'prediction_engine'),
                  accelerator='gpu' if str(device).startswith('cuda') else 'cpu',devices=1,
                  logger=False,enable_checkpointing=True,enable_progress_bar=False)
    pred=engine.predict(model=model,ckpt_path=str(ckpt),data_path=str(image_folder),return_predictions=True)
    if pred is None:raise RuntimeError('Anomalib Engine returned no predictions')
    answers={}
    for batch in pred:
        paths=batch.image_path
        maps=batch.anomaly_map
        scores=batch.pred_score
        if isinstance(paths,(str,Path)):paths=[str(paths)]
        for i,path in enumerate(paths):
            name0=Path(path).stem
            if name0 in answers:raise RuntimeError('Duplicate prediction basename '+name0)
            amap=maps[i].detach().cpu().numpy() if (keep_maps and maps is not None) else None
            if amap is not None:
                amap=np.squeeze(amap);amap=cv2.resize(amap.astype('float32'),(256,256))
                if not np.isfinite(amap).all():raise RuntimeError('Nonfinite anomaly map')
            scalar=float(scores[i].detach().cpu().item())
            if not np.isfinite(scalar):raise RuntimeError('Nonfinite anomaly score')
            answers[name0]={'score':scalar,'map':amap if keep_maps else None}
    return answers


"""Optional download of independently public auxiliary baseline-training datasets.
These are not date-fruit evaluation data; archive digests recorded after download.
"""
from pathlib import Path
import argparse,tarfile,urllib.request
from tqdm.auto import tqdm

URLS={'dtd':('https://www.robots.ox.ac.uk/~vgg/data/dtd/download/dtd-r1.0.1.tar.gz','dtd-r1.0.1.tar.gz'),
      'imagenette':('https://s3.amazonaws.com/fast-ai-imageclas/imagenette2-160.tgz','imagenette2-160.tgz')}

def extract_safe(tarpath,where):
    where=Path(where).resolve();where.mkdir(parents=True,exist_ok=True)
    with tarfile.open(tarpath,'r:gz') as f:
        for entry in f.getmembers():
            target=(where/entry.name).resolve()
            if not (str(target)==str(where) or str(target).startswith(str(where)+'/')):
                raise RuntimeError('Unsafe path in external dataset archive')
            if entry.issym() or entry.islnk():raise RuntimeError('Symbolic links not allowed in external archives')
        f.extractall(where,filter='data')

def download_assets(out):
    root=Path(out).resolve();root.mkdir(parents=True,exist_ok=True)
    info={}
    for name,(url,filename) in URLS.items():
        file=root/filename
        if not file.is_file():
            print(f'Downloading auxiliary {name}: {url}',flush=True)
            partial=file.with_suffix(file.suffix+'.partial')
            def on_chunk(n, chunk, total):
                if n==0:
                    print(f'  Expected compressed bytes: {total:,}' if total>0 else '  Archive size not supplied',flush=True)
                if total>0 and (n%1200==0):
                    print(f'  {min(n*chunk/total,1)*100:.0f}% downloaded',flush=True)
            urllib.request.urlretrieve(url,partial,reporthook=on_chunk)
            partial.replace(file)
        info[name]={'source_url':url,'downloaded_sha256':digest(file),'archive_bytes':file.stat().st_size}
        if name=='dtd':
            directory=root/'dtd'
            if not (directory/'images').is_dir():extract_safe(file,root)
            if not (directory/'images').is_dir():raise RuntimeError('DTD archive layout unexpected: '+str(root))
        else:
            directory=root/'imagenette2-160'
            if not (directory/'train').is_dir():extract_safe(file,root)
            if not (directory/'train').is_dir():raise RuntimeError('Imagenette archive layout unexpected: '+str(root))
    write_json(root/'auxiliary_sources.json',info)
    return {'dtd_dir':str(root/'dtd'),'imagenette_dir':str(root/'imagenette2-160')}












def _v4_build_fast_efficientad_step(original):
    """Transform only four diagnostic log arguments; verify AST reversibility."""
    import ast, copy, inspect, textwrap
    source = textwrap.dedent(inspect.getsource(original))
    tree = ast.parse(source)
    if len(tree.body)!=1 or not isinstance(tree.body[0], ast.FunctionDef):
        raise RuntimeError('EfficientAD training_step structure changed: acceleration refused')
    reference=copy.deepcopy(tree)
    targets={'train_st':'loss_st','train_ae':'loss_ae',
             'train_stae':'loss_stae','train_loss':'loss'}
    seen={}
    class Transform(ast.NodeTransformer):
        def visit_Call(self,node):
            self.generic_visit(node)
            if (isinstance(node.func,ast.Attribute) and node.func.attr=='log' and
                isinstance(node.func.value,ast.Name) and node.func.value.id=='self' and
                len(node.args)>=2 and isinstance(node.args[0],ast.Constant) and
                node.args[0].value in targets):
                name=node.args[0].value
                second=node.args[1]
                if not (isinstance(second,ast.Call) and not second.args and not second.keywords and
                        isinstance(second.func,ast.Attribute) and second.func.attr=='item' and
                        isinstance(second.func.value,ast.Name) and
                        second.func.value.id==targets[name]):
                    raise RuntimeError('Unexpected EfficientAD loss logging: acceleration refused')
                if name in seen:raise RuntimeError('Duplicate EfficientAD log key')
                seen[name]=True


                second.func.attr='detach'
            return node
    transformed=Transform().visit(tree)
    if set(seen)!=set(targets):
        raise RuntimeError('EfficientAD logging source changed; acceleration refused')

    class Reverse(ast.NodeTransformer):
        def visit_Call(self,node):
            self.generic_visit(node)
            if (isinstance(node.func,ast.Attribute) and node.func.attr=='log' and
                isinstance(node.func.value,ast.Name) and node.func.value.id=='self' and
                node.args and isinstance(node.args[0],ast.Constant) and
                node.args[0].value in targets):
                node.args[1].func.attr='item'
            return node
    rebuilt=Reverse().visit(copy.deepcopy(transformed))
    if ast.dump(rebuilt,include_attributes=False)!=ast.dump(reference,include_attributes=False):
        raise RuntimeError('The compiled EfficientAD fast step differs beyond diagnostic logging')
    ast.fix_missing_locations(transformed)
    namespace={}
    exec(compile(transformed,'<anomalib-2.2.0-v4-logger-acceleration>','exec'),
         original.__globals__, namespace)
    return namespace['training_step'],source


def _v4_fast_efficientad_parity(original,accelerated):
    """Compare reference and accelerated gradient and logs on identical CPU inputs."""
    from types import SimpleNamespace
    class Toy(torch.nn.Module):
        def __init__(self):
            super().__init__(); self.p=torch.nn.Parameter(torch.tensor(1.25))
        def forward(self,batch,batch_imagenet):
            return (self.p*batch.mean(),self.p**2*batch_imagenet.mean(),
                    self.p*(batch**2).mean())
    class Host:
        def __init__(self):
            self.model=Toy()
            self.device='cpu'
            self.imagenet_iterator=iter([(torch.tensor([[.1,.8]]),None)])
            self.imagenet_loader=[]
            self.logs={}
        def log(self,key,val,**kwargs):
            self.logs[key]=float(val.detach().item() if isinstance(val,torch.Tensor) else val)
    example=SimpleNamespace(image=torch.tensor([[.25,.6]]))
    a,b=Host(),Host()
    ya=original(a,example)['loss']
    yb=accelerated(b,example)['loss']
    ya.backward();yb.backward()
    if not (torch.equal(ya,yb) and torch.equal(a.model.p.grad,b.model.p.grad) and a.logs==b.logs):
        raise RuntimeError('EfficientAD logging optimization did not preserve toy losses/gradients')
    return True


def _v4_install_cached_imagenet_normalization():
    """Cache exactly identical ImageNet constants per CUDA device, with bitwise test."""
    import inspect, hashlib, functools
    import anomalib.models.image.efficient_ad.torch_model as em
    original = em.imagenet_norm_batch
    if getattr(original,'_v4_accelerated',False):
        return getattr(original,'_v4_original_sha256','already-patched')
    src=inspect.getsource(original)

    checks=[
        'torch.tensor([0.485, 0.456, 0.406])',
        'torch.tensor([0.229, 0.224, 0.225])',
        'return (x - mean) / std',
    ]
    if any(x not in src for x in checks) or src.count('torch.tensor(')!=2:
        raise RuntimeError('Unexpected EfficientAD normalization source; optimization refused')

    @functools.lru_cache(maxsize=6)
    def constants(device_key):
        device=torch.device(device_key)
        mean=torch.tensor([0.485,0.456,0.406])[None,:,None,None].to(device)
        std=torch.tensor([0.229,0.224,0.225])[None,:,None,None].to(device)
        return mean,std
    def cached(x):
        mean,std=constants(str(x.device))
        return (x-mean)/std

    devices=['cpu']+(['cuda:0'] if torch.cuda.is_available() else [])
    for device in devices:
        for dtype in (torch.float32,torch.float16):
            test=torch.tensor([.0,.1,.5,.9,1.0],device=device,dtype=dtype)
            test=test.reshape(1,1,1,5).expand(1,3,4,5).clone()
            ref=original(test)
            got=cached(test)
            if not torch.equal(ref,got):
                raise RuntimeError(f'ImageNet normalization not bitwise identical for {device}/{dtype}')
    cached._v4_accelerated=True
    cached._v4_original_sha256=hashlib.sha256(src.encode()).hexdigest()
    em.imagenet_norm_batch=cached
    return cached._v4_original_sha256


def enable_v4_efficientad_acceleration(model,seed):
    """Enable only measured-overhead removals with runtime source and math guards."""
    import inspect, hashlib
    from types import MethodType
    from importlib.metadata import version
    if version('anomalib')!='2.2.0' or type(model).__name__!='EfficientAd':
        raise RuntimeError('Optimized code only supports official Anomalib EfficientAd 2.2.0')
    original=model.training_step.__func__
    fast,source=_v4_build_fast_efficientad_step(original)
    _v4_fast_efficientad_parity(original,fast)
    norm_sha=_v4_install_cached_imagenet_normalization()
    model.training_step=MethodType(fast,model)



    start_original=model.on_train_start.__func__
    def fast_start(self):
        start_original(self)
        old=self.model.is_set
        normal=self.model.mean_std
        if not bool(old(normal)):
            raise RuntimeError('EfficientAD teacher statistics unexpectedly unset')
        def safe_is_set(d):
            if d is normal:return True
            return old(d)
        self.model.is_set=safe_is_set
    model.on_train_start=MethodType(fast_start,model)
    audit={'method':'EfficientAd','seed':int(seed),'anomalib_version':'2.2.0',
           'training_step_original_source_sha256':hashlib.sha256(source.encode()).hexdigest(),
           'normalization_original_source_sha256':norm_sha,
           'patched_logger_keys':['train_st','train_ae','train_stae','train_loss'],
           'learning_budget_unchanged':True,'effective_batch_size_unchanged':True,
           'loss_formula_unchanged':True,'teacher_stats_computation_unchanged':True,
           'imagenette_sampling_unchanged':True,'validation_and_score_protocol_unchanged':True,
           'safety_tests':['AST reversible transform','identical toy gradients',
                           'bitwise normalization equality on supported dtypes'],
           'important_note':'Execution optimization; resumed interrupted runs are not guaranteed bitwise-identical to uninterrupted runs.'}
    out=OUT/'models'/'EfficientAd'/str(seed)
    out.mkdir(parents=True,exist_ok=True)
    write_json(out/'acceleration_audit.json',audit)
    print('  VERIFIED EFFICIENTAD ACCELERATION: four loss .item() fences removed, '
          'constant GPU tensors cached, redundant fixed-state checks removed',flush=True)
    return audit








_v4_draem_safe_cache_status=type('_DraemCacheStatus',(),{})()
_v4_draem_safe_cache_status.cache=None

def _v4_build_fast_draem_step(original):
    """Change ONLY the diagnostic logging argument; verify reversible AST."""
    import inspect, ast, textwrap, copy, hashlib
    source=textwrap.dedent(inspect.getsource(original))
    original_tree=ast.parse(source)
    tree=copy.deepcopy(original_tree)
    changes=[]
    class DiagnosticLoggingOnly(ast.NodeTransformer):
        def visit_Call(self,node):
            self.generic_visit(node)
            if (isinstance(node.func,ast.Attribute) and node.func.attr=='log'
                and isinstance(node.func.value,ast.Name) and node.func.value.id=='self'
                and len(node.args)>=2 and isinstance(node.args[0],ast.Constant)
                and node.args[0].value=='train_loss'):
                val=node.args[1]
                if not (isinstance(val,ast.Call) and isinstance(val.func,ast.Attribute)
                        and val.func.attr=='item' and not val.args and not val.keywords
                        and isinstance(val.func.value,ast.Name) and val.func.value.id=='loss'):
                    raise RuntimeError('Unexpected DRAEM train_loss logging; adapter refused')
                val.func.attr='detach'
                changes.append('train_loss')
            return node
    tree=DiagnosticLoggingOnly().visit(tree)
    if changes!=['train_loss']:
        raise RuntimeError('DRAEM logging source differs from pinned Anomalib 2.2.0')
    restored=copy.deepcopy(tree)
    class Undo(ast.NodeTransformer):
        def visit_Call(self,node):
            self.generic_visit(node)
            if (isinstance(node.func,ast.Attribute) and node.func.attr=='log'
                and isinstance(node.func.value,ast.Name) and node.func.value.id=='self'
                and node.args and isinstance(node.args[0],ast.Constant)
                and node.args[0].value=='train_loss'):
                node.args[1].func.attr='item'
            return node
    restored=Undo().visit(restored)
    if ast.dump(original_tree,include_attributes=False)!=ast.dump(restored,include_attributes=False):
        raise RuntimeError('DRAEM adapter changes source beyond scalar logging; refused')
    ast.fix_missing_locations(tree)
    ns={}
    exec(compile(tree,'<datepalm-v4-draem-logger>','exec'),original.__globals__,ns)
    return ns['training_step'],hashlib.sha256(source.encode()).hexdigest()

def _v4_draem_fast_logging_parity(original,fast):
    """Toy gradient + scalar log parity, without touching experimental data."""
    from types import SimpleNamespace
    import torch
    class Host:
        def __init__(self):
            self.weight=torch.nn.Parameter(torch.tensor(1.75))
            self.augmenter=lambda x:(x*0.75,torch.ones_like(x))
            self.model=lambda x:(self.weight*x,self.weight*x.square())
            self.loss=lambda x,reconstruction,mask,prediction: (
                (reconstruction-x).square().mean()+prediction.square().mean())
            self.sspcab=False
            self.log_values={}
        def log(self,name,value,**kwargs):
            self.log_values[name]=float(value.detach().item() if isinstance(value,torch.Tensor) else value)
    batch=SimpleNamespace(image=torch.tensor([[0.2,0.5,0.8]]))
    a,b=Host(),Host()
    out_a=original(a,batch)['loss'];out_b=fast(b,batch)['loss']
    out_a.backward();out_b.backward()
    if not (torch.equal(out_a,out_b) and torch.equal(a.weight.grad,b.weight.grad)
            and a.log_values==b.log_values):
        raise RuntimeError('DRAEM loss/gradient/log parity failed; optimization refused')
    return True

def _v4_install_draem_texture_cache(model,capacity_mib=128):
    """Cache decoded original DTD images on CPU only; no random transformations."""
    import inspect, hashlib
    from collections import OrderedDict
    import anomalib.data.utils.generators.perlin as perlin
    from importlib.metadata import version
    if version('anomalib')!='2.2.0':
        raise RuntimeError('Texture cache only validated for anomalib==2.2.0')
    source=inspect.getsource(perlin.PerlinAnomalyGenerator.generate_perturbation)
    if ('io.read_image(str(anomaly_source_path), mode=io.ImageReadMode.RGB)' not in source
        or 'anomaly_source_img' not in source or '.float().to(device)' not in source):
        raise RuntimeError('DRAEM DTD reader source changed; refusing cache')
    old=perlin.io
    if getattr(old,'_v4_draem_cache',False):
        _v4_draem_safe_cache_status.cache=old
        return old,hashlib.sha256(source.encode()).hexdigest()
    expected=getattr(model.augmenter,'anomaly_source_paths',[])
    if not expected:
        raise RuntimeError('DRAEM has no DTD texture files; cannot validate cache')
    capacity=int(capacity_mib)*1024**2
    if capacity<1:
        raise ValueError('DTD CPU cache capacity must be positive')
    class CachedTextureIO:
        _v4_draem_cache=True
        def __init__(self,delegate):
            self.delegate=delegate
            self.buffer=OrderedDict()
            self.bytes=0
            self.hits=0
            self.misses=0
        def __getattr__(self,key):
            return getattr(self.delegate,key)
        def read_image(self,path,mode=None,**kwargs):
            from pathlib import Path
            p=Path(path)
            st=p.stat()
            key=(str(p),str(mode),int(st.st_mtime_ns),int(st.st_size))
            hit=self.buffer.get(key)
            if hit is not None:
                self.hits+=1
                self.buffer.move_to_end(key)
                return hit
            self.misses+=1
            out=self.delegate.read_image(path,mode=mode,**kwargs)
            if not (out.device.type=='cpu' and out.dtype==torch.uint8):
                raise RuntimeError('DRAEM texture reader unexpectedly changed dtype/device')
            n=int(out.numel()*out.element_size())
            if n<=capacity:
                while self.buffer and self.bytes+n>capacity:
                    _,removed=self.buffer.popitem(last=False)
                    self.bytes-=int(removed.numel()*removed.element_size())
                self.buffer[key]=out
                self.bytes+=n
            return out
    proxy=CachedTextureIO(old)

    sample=str(expected[0]);fmt=old.ImageReadMode.RGB
    a=old.read_image(sample,mode=fmt)
    b=proxy.read_image(sample,mode=fmt)
    c=proxy.read_image(sample,mode=fmt)
    if not (torch.equal(a,b) and torch.equal(a,c) and b.dtype==a.dtype
            and b.shape==a.shape and b.device==a.device):
        raise RuntimeError('DRAEM DTD image-cache source parity failed')
    perlin.io=proxy
    _v4_draem_safe_cache_status.cache=proxy
    return proxy,hashlib.sha256(source.encode()).hexdigest()

def enable_v4_draem_safe_acceleration(model,seed):
    """Source-guarded, side-effect-limited training accelerator for DRAEM."""
    from importlib.metadata import version
    from types import MethodType
    if version('anomalib')!='2.2.0' or type(model).__name__!='Draem':
        raise RuntimeError('DRAEM accelerator supports only official Anomalib 2.2.0')
    original=model.training_step.__func__
    fast,source_sha=_v4_build_fast_draem_step(original)
    _v4_draem_fast_logging_parity(original,fast)
    proxy,reader_sha=_v4_install_draem_texture_cache(model,V4_DRAEM_DTD_CACHE_MIB)
    model.training_step=MethodType(fast,model)
    audit={
        'method':'Draem','seed':int(seed),'anomalib_version':'2.2.0',
        'training_loss_logging_original_source_sha256':source_sha,
        'texture_reader_original_source_sha256':reader_sha,
        'cpu_dtd_cache_capacity_mib':int(V4_DRAEM_DTD_CACHE_MIB),
        'cache_decoded_original_textures_only':True,
        'cache_augmented_images_or_perlin_masks':False,
        'loss_formula_optimizer_architecture_precision_budget_unchanged':True,
        'train_batch_size_unchanged':True,'validation_frequency_unchanged':True,
        'safety_checks':['reversible AST source transformation','identical toy gradients and logs',
                         'exact DTD decoded-image byte comparison'],
        'limitation':'Runtime GPU speedup and exact stochastic resumed-run equivalence are unverified.',
    }
    dest=OUT/'models'/'Draem'/str(seed)
    dest.mkdir(parents=True,exist_ok=True)
    write_json(dest/'acceleration_audit.json',audit)
    print('  DRAEM FAST SAFE: diagnostic GPU logging fence removed; '
          'CPU original-texture decode LRU enabled, capacity=',V4_DRAEM_DTD_CACHE_MIB,'MiB',flush=True)
    return audit


In [ ]:
"""Frozen split evaluation and source-aware statistical analysis."""
from __future__ import annotations
from pathlib import Path
import math,time,os
from tqdm.auto import tqdm
import numpy as np,pandas as pd,cv2,torch
from sklearn.metrics import roc_auc_score,average_precision_score,roc_curve
from scipy.stats import rankdata




V4_EVAL_RELEASE = '2026-10-15-postprocessed-streaming-parity-v2'
V4_PIXEL_PARITY_ABS_TOL = 2e-11
V4_LEGACY_SPOT_CHECK_COUNT = 7
_V4_CKPT_HASH_CACHE = {}
_V4_MANIFEST_HASH_CACHE = {}
_V4_MASK_CACHE = {}


def v4_file_sha_cached(path):
    path=Path(path).resolve()
    stat=path.stat()
    key=(str(path),int(stat.st_size),int(stat.st_mtime_ns))
    if key not in _V4_CKPT_HASH_CACHE:
        _V4_CKPT_HASH_CACHE.clear() if len(_V4_CKPT_HASH_CACHE)>100 else None
        _V4_CKPT_HASH_CACHE[key]=digest(path)
    return _V4_CKPT_HASH_CACHE[key]


def v4_manifest_hash(samples):

    key=id(samples)
    if key not in _V4_MANIFEST_HASH_CACHE:
        fields=[x for x in ['sample_id','source_id','source_group','family','severity',
                            'label','kind','image_path','image_sha256','mask_path',
                            'mask_sha256','fg_path'] if x in samples]
        raw=samples[fields].to_csv(index=False).encode('utf-8')
        _V4_MANIFEST_HASH_CACHE[key]=(samples,hashlib.sha256(raw).hexdigest())
    return _V4_MANIFEST_HASH_CACHE[key][1]


def v4_metric_release_signature():
    funcs=['matched_predict','baseline_predict','predict_from_checkpoint',
           '_v4_prediction_collator','v4_fast_pixel_auc_ap',
           'global_edit_support_aupro','collect_and_score',
           'image_read','load_model','build_model']
    parts={name:_stable_code(globals()[name].__code__) for name in funcs}
    return hashlib.sha256(json.dumps(parts,sort_keys=True,default=str).encode()).hexdigest()


def v4_score_contract(checkpoint,samples,output,do_localization):
    return {'schema_version':2,'release':V4_EVAL_RELEASE,
        'checkpoint_sha256':v4_file_sha_cached(checkpoint),
        'benchmark_sha256':v4_manifest_hash(samples),
        'scoring_code_sha256':v4_metric_release_signature(),
        'localization':bool(do_localization),
        'score_file_name':Path(output).name}


def v4_migrate_verified_p4_matched_cache(metadata,contract,output,name):
    """Preserve verified V1 matched-model scores across BASELINE-only callback patch.

    V1 score provenance exists for 20 matched development scores. Its global
    `scoring_code_sha256` included even the unrelated baseline prediction
    transport, so correcting baseline callback order would otherwise trigger
    unnecessary repeated matched-model scoring. Migration is permitted ONLY
    pre-lock for Cxx scores, only when the old SHA256 file/checkpoint/benchmark
    bindings still match and the score-file SHA256 still matches the CSV.
    The original V1 metadata is copied intact for a permanent audit trail.
    This does not retroactively prove the provenance of previously adopted
    legacy CSVs; the original limitation is retained in the new record.
    """
    metadata=Path(metadata);output=Path(output)
    if not name.startswith('C'):
        return False
    if (OUT/'frozen'/'protocol_lock.json').exists() or (OUT/'FINAL_TEST_USED.flag').exists():
        return False
    if not output.resolve().is_relative_to((OUT/'development').resolve()):
        return False
    old=read_json(metadata)
    if old.get('release')!='2026-10-15-audited-p4-p7-v1':
        return False
    for key in ['schema_version','checkpoint_sha256','benchmark_sha256',
                'localization','score_file_name']:
        if old.get(key)!=contract.get(key):
            return False
    if old.get('localization') is not False:
        return False
    prior_signature=old.get('scoring_code_sha256')
    if not isinstance(prior_signature,str) or len(prior_signature)!=64:
        return False
    if old.get('score_csv_sha256')!=digest(output):
        return False
    import shutil
    backup=metadata.with_name(metadata.name+'.original_v1')
    if backup.exists():
        if digest(backup)!=digest(metadata):
            raise RuntimeError('Original V1 provenance backup differs; manual audit required: '+str(backup))
    else:
        shutil.copy2(metadata,backup)
        if digest(backup)!=digest(metadata):
            raise RuntimeError('Failed to preserve original P4 provenance metadata')
    amended=dict(old)
    amended.update(contract)
    amended.update({'score_csv_sha256':old['score_csv_sha256'],
                    'cache_origin':old.get('cache_origin','unknown'),
                    'prelock_transport_release_migration_from':old['release'],
                    'v1_original_provenance_sha256':digest(backup),
                    'migration_basis':'matched_predict and matched scoring helpers AST-identical to V1; '
                                      'only baseline streaming transport changed',
                    'unverified_legacy_cache_remains_unverified':
                        old.get('cache_origin')=='legacy_spot_checked_unverifiable_full_provenance'})
    write_json(metadata,amended)
    print('  VERIFIED V1 MATCHED-SCORE CACHE RETAINED WITHOUT GPU RECOMPUTATION: '
          +output.name,flush=True)
    return True


def v4_verify_score_table(cached,samples,localization):
    if (len(cached)!=len(samples) or not cached.sample_id.is_unique or
        set(cached.sample_id.astype(str))!=set(samples.sample_id.astype(str))):
        raise RuntimeError('Score CSV missing, duplicated, or incorrectly identified samples')
    if not np.isfinite(pd.to_numeric(cached.score,errors='coerce')).all():
        raise RuntimeError('Cached image scores include nonfinite values')
    aligned=cached.set_index('sample_id').loc[samples.sample_id.astype(str)]
    expected=samples.set_index('sample_id').loc[samples.sample_id.astype(str)]
    for column in ['source_id','source_group','family','severity','kind','label']:
        if column not in aligned or column not in expected:
            raise RuntimeError('Required score provenance column missing: '+column)
        left=aligned[column].fillna('NA').astype(str).to_numpy()
        right=expected[column].fillna('NA').astype(str).to_numpy()
        if not np.array_equal(left,right):
            raise RuntimeError('Cached scientific metadata differs from benchmark: '+column)
    if localization and not {'edit_pixel_auroc','edit_pixel_ap',
                             'edit_support_aupro_fpr0p30','aupro_count_regions'}.issubset(cached.columns):
        raise RuntimeError('Cached test scores lack mandatory localization metrics')


def v4_adopt_legacy_score_cache(checkpoint,name,samples,device,extra,output,localization,expected):



    root=Path(output)
    if localization or root.parent.resolve().is_relative_to((OUT/'results').resolve()):
        raise RuntimeError('Legacy post-lock cache lacks checkpoint provenance: '+str(root)+
                           '. Do not discard the lock or silently reuse these scores.')
    cached=pd.read_csv(root,dtype={'sample_id':str,'source_id':str,'source_group':str})
    v4_verify_score_table(cached,samples,False)
    locations=np.unique(np.linspace(0,len(samples)-1,min(V4_LEGACY_SPOT_CHECK_COUNT,len(samples)),dtype=int))
    chosen=samples.iloc[locations].copy()
    if name.startswith('C'):
        predictions=matched_predict(checkpoint,chosen,device,keep_maps=False)
        check={r['sample_id']:r['score'] for r in predictions}
    else:
        import tempfile,shutil
        with tempfile.TemporaryDirectory(prefix='v4_legacy_verify_',dir=OUT/'data') as tmp:
            folder=Path(tmp)
            for row in chosen.itertuples():
                src=Path(row.image_path);dest=folder/src.name
                try:dest.symlink_to(src)
                except OSError:shutil.copy2(src,dest)
            check={sid:r['score'] for sid,r in
                   predict_from_checkpoint(checkpoint,name,folder,device,extra,keep_maps=False).items()}
    lookup=cached.set_index('sample_id').score
    for sid,val in check.items():
        if sid not in lookup or not np.isclose(float(lookup[sid]),val,rtol=1e-4,atol=1e-5):
            raise RuntimeError('Legacy score differs from checkpoint on '+str(sid)+
                               '; refusing reuse: '+str(output))
    record=dict(expected,cache_origin='legacy_spot_checked_unverifiable_full_provenance',
                checked_samples=[str(x) for x in check],
                audit_warning='Historical CSV has no original checkpoint/benchmark manifest. '+
                'Seven reference predictions match, but full provenance cannot be reconstructed.',
                score_csv_sha256=digest(root))
    write_json(root.with_suffix('.csv.provenance.json'),record)
    print('  LEGACY CACHE ADOPTED AFTER CHECKPOINT SPOT CHECK; provenance limitation recorded: '+root.name,flush=True)
    return cached


def v4_runtime_versions():
    from importlib.metadata import version
    packages=['torch','torchvision','lightning','anomalib','numpy',
              'scikit-learn','scipy','pandas']
    return {name:version(name) for name in packages} | {'opencv':str(cv2.__version__)}


def v4_fast_pixel_auc_ap(y,pred):
    """Single stable score sort computes sklearn-equivalent AUROC and AP."""
    y=np.asarray(y).reshape(-1)
    pred=np.asarray(pred).reshape(-1)
    if y.size!=pred.size or not y.size or not np.isfinite(pred).all():
        raise ValueError('Pixel scores must be finite and match labels')
    npos=int(np.count_nonzero(y))
    if npos==0 or npos==len(y):
        raise ValueError('Pixel AUROC requires positives and negatives')
    order=np.argsort(pred,kind='mergesort')
    p=pred[order]; labels=y[order].astype(np.int64,copy=False)
    starts=np.r_[0,np.flatnonzero(p[1:]!=p[:-1])+1]
    positives=np.add.reduceat(labels,starts)
    counts=np.diff(np.r_[starts,len(labels)])
    tp=np.cumsum(positives[::-1],dtype=np.float64)
    fp=np.cumsum((counts-positives)[::-1],dtype=np.float64)
    tpr=np.r_[0.,tp/npos]
    fpr=np.r_[0.,fp/(len(y)-npos)]
    auc=float(np.trapezoid(tpr,fpr))
    ap=float(np.sum((positives[::-1]/npos)*(tp/(tp+fp))))
    return auc,ap


class V4PackedMaskCache:


    def __init__(self,samples):
        self.samples=samples
        self.identity=v4_manifest_hash(samples)
        self.idx={str(v):i for i,v in enumerate(samples.sample_id.astype(str))}
        if len(self.idx)!=len(samples):raise RuntimeError('Nonunique packed-mask IDs')
        self.packed_bytes=256*256//8
        self.folder=OUT/'data'/'evaluation_mask_cache'/self.identity
        self.folder.mkdir(parents=True,exist_ok=True)
        self.meta=self.folder/'validated.json'
        self.fg_file=self.folder/'foreground.npy'
        self.gt_file=self.folder/'ground_truth.npy'
        expected={'schema':1,'benchmark_sha256':self.identity,
                  'n':len(samples),'shape':[256,256], 'bitorder':'big'}
        if self.meta.exists():
            saved=read_json(self.meta)
            if any(saved.get(k)!=v for k,v in expected.items()):
                raise RuntimeError('Packed-mask metadata mismatch; refusing stale masks')
            if not self.fg_file.is_file() or not self.gt_file.is_file():
                raise RuntimeError('Packed-mask files incomplete; remove only the incomplete cache directory')
            fg=np.load(self.fg_file,mmap_mode='r');gt=np.load(self.gt_file,mmap_mode='r')
            if fg.shape!=(len(samples),self.packed_bytes) or gt.shape!=fg.shape:
                raise RuntimeError('Packed-mask arrays malformed')
            if saved.get('fg_sha256')!=digest(self.fg_file) or saved.get('gt_sha256')!=digest(self.gt_file):
                raise RuntimeError('Packed-mask cache checksum mismatch')
            self.fg,self.gt=fg,gt
        else:
            for path in [self.fg_file,self.gt_file]:
                if path.exists():path.unlink()
            from numpy.lib.format import open_memmap
            fg=open_memmap(self.fg_file,mode='w+',dtype=np.uint8,shape=(len(samples),self.packed_bytes))
            gt=open_memmap(self.gt_file,mode='w+',dtype=np.uint8,shape=(len(samples),self.packed_bytes))
            for i,row in enumerate(tqdm(samples.itertuples(),total=len(samples),desc='Cache evaluation binary masks',unit='image')):
                a=cv2.imread(row.fg_path,cv2.IMREAD_GRAYSCALE)
                b=cv2.imread(row.mask_path,cv2.IMREAD_GRAYSCALE)
                if a is None or b is None or a.shape!=(256,256) or b.shape!=(256,256):
                    raise RuntimeError('Missing or unexpected binary evaluation mask at '+str(row.sample_id))
                if hasattr(row,'mask_sha256') and isinstance(row.mask_sha256,str) and row.mask_sha256:
                    if digest(row.mask_path)!=row.mask_sha256:
                        raise RuntimeError('Benchmark mask SHA256 mismatch: '+str(row.sample_id))
                fg[i]=np.packbits((a>127).reshape(-1))
                gt[i]=np.packbits((b>127).reshape(-1))
            fg.flush();gt.flush();del fg,gt
            meta=dict(expected,fg_sha256=digest(self.fg_file),gt_sha256=digest(self.gt_file),
                      written_at=time.strftime('%Y-%m-%dT%H:%M:%SZ',time.gmtime()))
            write_json(self.meta,meta)
            self.fg=np.load(self.fg_file,mmap_mode='r')
            self.gt=np.load(self.gt_file,mmap_mode='r')
    def masks(self,sid):
        i=self.idx[str(sid)]
        fg=np.unpackbits(self.fg[i],count=256*256).reshape(256,256).astype(bool)
        gt=np.unpackbits(self.gt[i],count=256*256).reshape(256,256).astype(bool)
        return fg,gt


def v4_pixel_metric_preflight():
    """Fail before P6 if accelerated image metrics disagree with sklearn."""
    rng=np.random.default_rng(715001)
    for i in range(75):
        n=128 if i%4 else 4096
        labels=np.zeros(n,np.uint8)
        labels[rng.choice(n,size=int(rng.integers(1,n)),replace=False)]=1
        scores=(rng.integers(0,8,size=n)/8.).astype('float32') if i%3==0 else rng.normal(size=n).astype('float32')
        a,b=v4_fast_pixel_auc_ap(labels,scores)
        if abs(a-roc_auc_score(labels,scores))>V4_PIXEL_PARITY_ABS_TOL or            abs(b-average_precision_score(labels,scores))>V4_PIXEL_PARITY_ABS_TOL:
            raise RuntimeError('Pixel AUROC/AP optimization failed reference parity')

    import tempfile
    with tempfile.TemporaryDirectory(prefix='v4_aupro_parity_') as temp:
        root=Path(temp);rows=[];preds=[]
        for i in range(4):
            fg=(rng.random((256,256))>.35)
            gt=np.zeros((256,256),dtype=bool)
            gt[20+i*3:43+i*3,25:48]=True
            gt[80:100,100+i*6:125+i*6]=True
            gt[10,10]=True
            m=np.round(rng.normal(size=(256,256)).astype('float32')*4).astype('float32')
            ff=root/f'fg{i}.png';mf=root/f'gt{i}.png'
            cv2.imwrite(str(ff),fg.astype('uint8')*255)
            cv2.imwrite(str(mf),gt.astype('uint8')*255)
            rows.append({'sample_id':f'parity{i}','fg_path':str(ff),'mask_path':str(mf),
                         'label':1 if i%2 else 0})
            preds.append({'sample_id':f'parity{i}','map':m})
        data=pd.DataFrame(rows)
        a,n=v4_reference_global_aupro(preds,data)
        b,m=global_edit_support_aupro(preds,data)
        if n!=m or not np.isclose(a,b,rtol=0,atol=V4_PIXEL_PARITY_ABS_TOL):
            raise RuntimeError(f'AUPRO vectorization fails reference: {a},{b} regions {n},{m}')
    return True


def v4_performance_event(event):
    folder=OUT/'reports';folder.mkdir(parents=True,exist_ok=True)
    event=dict(event,when=time.strftime('%Y-%m-%dT%H:%M:%SZ',time.gmtime()))
    with (folder/'performance_events.jsonl').open('a',encoding='utf-8') as file:
        file.write(json.dumps(event,sort_keys=True,default=str)+'\n')






def v4_verify_benchmark_materialization(samples,phase):
    if not {'image_sha256','mask_sha256'}.issubset(samples.columns):
        raise RuntimeError('Official benchmark lacks image/mask SHA256 evidence')
    t0=time.perf_counter()
    for i,row in enumerate(tqdm(samples.itertuples(),total=len(samples),
                                desc=f'Verify {phase} benchmark SHA256',unit='image'),1):
        for filename,expect in [(row.image_path,row.image_sha256),(row.mask_path,row.mask_sha256)]:
            if not isinstance(expect,str) or not Path(filename).exists() or digest(filename)!=expect:
                raise RuntimeError('Benchmark image/mask has been altered: '+str(filename))
        if i%256==0:
            live_update(subtask=f'Verifying {phase} benchmark PNG integrity',
                        benchmark_integrity_done=i,benchmark_integrity_total=len(samples))
    v4_performance_event({'stage':phase,'operation':'full_image_and_mask_sha256',
                         'images':len(samples),'seconds':time.perf_counter()-t0})
    print(f'  BENCHMARK INTEGRITY PASS: {phase}: {len(samples):,} images and masks hashed.',flush=True)
    return True


def build_benchmark(df, raw_root, out, split, limit=None):
    """Persist progress after every original source; safely continue partial generation."""
    out=Path(out); out.mkdir(parents=True,exist_ok=True)
    done=out/'manifest.csv'
    if done.exists():
        existing=pd.read_csv(done,dtype={'sample_id':str,'source_id':str,'source_group':str})
        expected=df[df.split==split].source_id.nunique()
        if limit: expected=min(expected,limit)
        factor=3 if split=='calibration' else 15
        if len(existing)!=expected*factor or not existing.image_path.map(lambda p:Path(p).exists()).all():
            raise RuntimeError('Incomplete/corrupt cached benchmark manifest: '+str(done))
        print(f'  RESUME: {split} benchmark ({len(existing):,} image variants) reused',flush=True)
        return existing
    rows=df[df.split==split].to_dict('records')
    if limit: rows=rows[:limit]
    family=FAMILIES_TEST if split=='locked_test' else ([] if split=='calibration' else FAMILIES_TRAIN)
    parts=out/'completed_sources';parts.mkdir(exist_ok=True)
    all_rows=[]
    live_update(force=True,subtask=f'Generate {split} benchmark',benchmark_done=0,benchmark_total=len(rows))
    for source_i,r in enumerate(tqdm(rows,desc=f'Generate {split} benchmark',unit='source'),1):
        cache=parts/(r['source_id']+'.csv')
        if cache.exists():
            frame=pd.read_csv(cache,dtype={'sample_id':str,'source_id':str,'source_group':str})
            if all(Path(p).exists() for p in frame.image_path) and all(Path(p).exists() for p in frame.mask_path):
                all_rows.extend(frame.to_dict('records'))
                live_update(subtask=f'Generate {split} benchmark',benchmark_done=source_i,benchmark_total=len(rows))
                continue
        im,fg,status=load_source(r,raw_root)
        if status!='ok':raise RuntimeError('Foreground QC inconsistent '+r['source_id'])
        originals=[('original','none','none',im,np.zeros_like(fg,dtype=np.uint8))]
        for i in (0,1):
            originals.append((f'benign_{i}','none','none',benign(im,i,r['source_id']+split),np.zeros_like(fg,dtype=np.uint8)))
        for fam in family:
            for severity in SEVERITIES:
                rng=make_rng('fixed-eval',split,r['source_id'],fam,severity)
                z,m=(heldout_generate(im,fg,fam,severity,rng) if split=='locked_test'
                     else train_generate(im,fg,fam,severity,rng,'fruit'))
                originals.append((f'{fam}_{severity}',fam,severity,z,m))
        fgfile=out/'foreground'/f'{r["source_id"]}.png';save_mask(fgfile,fg)
        entries=[]
        for variant,fam,sev,z,m in originals:
            sid=r['source_id']+'__'+variant
            fn=out/'images'/(sid+'.png');mf=out/'masks'/(sid+'.png')
            save_png(fn,z);save_mask(mf,m)
            stats=(change_stats(im,z,m,fg) if fam!='none' else
                   {'support_fraction':0.,'off_fruit_pixels':0,'mean_abs_change':0.,'nonempty':False})
            entries.append(dict(sample_id=sid,source_id=r['source_id'],source_group=r['source_group'],
                                source_variety=r['variety'],source_size=r['native_size_label'],
                                family=fam,severity=sev,kind=variant,label=int(fam!='none'),
                                image_path=str(fn),mask_path=str(mf),fg_path=str(fgfile),
                                image_sha256=digest(fn),mask_sha256=digest(mf),**stats))
        pd.DataFrame(entries).to_csv(cache.with_suffix('.tmp'),index=False)
        os.replace(cache.with_suffix('.tmp'),cache)
        all_rows.extend(entries)
        live_update(subtask=f'Generate {split} benchmark',benchmark_done=source_i,benchmark_total=len(rows))
    result=pd.DataFrame(all_rows)
    expected=len(rows)*(3 if split=='calibration' else 15)
    if len(result)!=expected:raise RuntimeError('Benchmark incomplete '+str(len(result))+' != '+str(expected))
    tmp=done.with_suffix('.csv.tmp')
    result.to_csv(tmp,index=False);os.replace(tmp,done)
    print(f'  {split}: {len(rows)} original sources → {len(result)} variants',flush=True)
    return result

class BenchmarkImageDataset(torch.utils.data.Dataset):
    """Preserve benchmark order, but decode PNGs in background CPU workers."""
    def __init__(self,samples):
        self.rows=samples.to_dict('records')
    def __len__(self):return len(self.rows)
    def __getitem__(self,i):
        row=self.rows[i]
        image=image_read(row['image_path'])
        fg=cv2.imread(row['fg_path'],cv2.IMREAD_GRAYSCALE)
        if fg is None:raise RuntimeError('Missing benchmark foreground '+row['fg_path'])
        return (torch.from_numpy(np.ascontiguousarray(image.transpose(2,0,1))),
                torch.from_numpy(np.ascontiguousarray(fg>127)),row['sample_id'])

@torch.no_grad()
def matched_predict(ckpt,samples,device,batch_size=24,keep_maps=True,store_path=None):
    model=load_model(ckpt,device);model.eval();result=[]
    ds=BenchmarkImageDataset(samples)
    loader=v4_loader(ds,batch_size,False)


    stored_maps=None
    if keep_maps and store_path is not None:
        from shutil import disk_usage
        store_path=Path(store_path)
        store_path.parent.mkdir(parents=True,exist_ok=True)
        needed=len(ds)*256*256*4
        if disk_usage(store_path.parent).free < needed+2*1024**3:
            raise RuntimeError('Insufficient disk space for bounded matched-model localization')
        stored_maps=np.lib.format.open_memmap(store_path,mode='w+',
                                  dtype=np.float32,shape=(len(ds),256,256))
    processed=0
    live_update(force=True,subtask='Matched benchmark inference (GPU)',
                scoring_samples_done=0,scoring_samples_total=len(ds))
    for images,fg,sample_ids in tqdm(loader,desc='Neural predictions',unit='batch',leave=False):
        xx=images.to(device,non_blocking=True).float().div_(255.)
        maps=torch.sigmoid(model(xx)).float().cpu().numpy()[:,0]
        foreground=fg.numpy()
        for k,sid in enumerate(sample_ids):
            vals=maps[k][foreground[k]]
            if len(vals)<100:raise RuntimeError('Insufficient fruit pixels')
            if keep_maps and stored_maps is not None:
                if maps[k].shape!=(256,256):
                    raise RuntimeError('Unexpected matched-model anomaly map shape')
                stored_maps[processed+k]=maps[k]
            result.append({'sample_id':sid,'score':float(np.quantile(vals,.99)),
                           'map':(stored_maps[processed+k] if stored_maps is not None
                                  else (maps[k] if keep_maps else None))})
        processed+=len(sample_ids)
        live_update(subtask='Matched benchmark inference (GPU)',
                    scoring_samples_done=processed,scoring_samples_total=len(ds))
    if stored_maps is not None:stored_maps.flush()
    del model
    if str(device).startswith('cuda'):torch.cuda.empty_cache()
    return result

def baseline_predict(ckpt,name,samples,device,extra,keep_maps=True):


    image_dir=Path(samples.iloc[0].image_path).parent
    if V4_ENABLE_STREAMING_BASELINE_INFERENCE:
        batch=_v4_verify_inference_parity(ckpt,name,samples,device,extra)

        mapfile=OUT/'data'/'streaming_pixel_maps'/(f'{name}_{Path(ckpt).parent.name}_{samples.iloc[0]["sample_id"].split("__")[0]}.npy')
        mapfile.parent.mkdir(exist_ok=True,parents=True)
        predicted=_v4_prediction_collator(ckpt,name,image_dir,device,extra,
                     keep_maps=keep_maps,expected_ids=samples.sample_id.astype(str).tolist(),
                     map_store_path=mapfile if keep_maps else None,eval_batch=batch)
    else:
        predicted=predict_from_checkpoint(ckpt,name,image_dir,device,extra,keep_maps=keep_maps)
    result=[]
    for r in samples.to_dict('records'):
        v=predicted.get(r['sample_id'])
        if v is None:raise RuntimeError(f'{name} missing prediction '+r['sample_id'])
        result.append({'sample_id':r['sample_id'],'score':float(v['score']),'map':v['map']})
    return result

def v4_reference_global_aupro(preds,samples,bins=96,maximum_fpr=.30):
    """Area under per-region overlap vs false-positive pixel rate, normalized to [0,1].

    The regions are connected components of algorithmic edit-support masks, not real defects.
    Pixel evaluation is restricted to the fruit foreground. Tiny components <5 pixels
    are excluded and counted as a segmentation-target limitation.
    """
    lookup=samples.set_index('sample_id')
    finite_maps=[p['map'] for p in preds if p['map'] is not None]
    if not finite_maps:return np.nan,0
    lo=min(float(np.nanmin(m)) for m in finite_maps);hi=max(float(np.nanmax(m)) for m in finite_maps)
    if not np.isfinite(lo+hi) or hi<=lo:return np.nan,0
    edges=np.linspace(lo,hi,bins+1)
    normal_hist=np.zeros(bins,np.float64);pro_hist=np.zeros(bins,np.float64)
    regions=0
    live_update(force=True,subtask='Pixel AUPRO analysis',aupro_done=0,aupro_total=len(preds))
    for aupro_i,p in enumerate(tqdm(preds,desc='Pixel AUPRO',unit='map',leave=False),1):
        if aupro_i%20==0:live_update(subtask='Pixel AUPRO analysis',aupro_done=aupro_i,aupro_total=len(preds))
        m=p['map']
        if m is None:continue
        r=lookup.loc[p['sample_id']]
        fg=cv2.imread(r['fg_path'],cv2.IMREAD_GRAYSCALE)>127
        gt=cv2.imread(r['mask_path'],cv2.IMREAD_GRAYSCALE)>127
        normal_hist+=np.histogram(m[fg&(~gt)],bins=edges)[0]
        if r['label']!=1:continue
        components,labels,stats,_=cv2.connectedComponentsWithStats(gt.astype('uint8'),8)
        for cid in range(1,components):
            area=int(stats[cid,cv2.CC_STAT_AREA])
            if area<5:continue
            h=np.histogram(m[labels==cid],bins=edges)[0].astype('float64')
            pro_hist+=h/max(h.sum(),1)
            regions+=1
    if regions<1 or normal_hist.sum()==0:return np.nan,regions

    fpr=np.r_[0.,np.cumsum(normal_hist[::-1])/normal_hist.sum()]
    pro=np.r_[0.,np.cumsum(pro_hist[::-1])/regions]
    keep=fpr<maximum_fpr
    xx=np.r_[fpr[keep],maximum_fpr]
    yy=np.r_[pro[keep],np.interp(maximum_fpr,fpr,pro)]
    return float(np.trapezoid(yy,xx)/maximum_fpr),regions


def global_edit_support_aupro(preds,samples,bins=96,maximum_fpr=.30,mask_cache=None):
    """Same AUPRO definition, vectorized per-region histogram accumulation.

    Reference still available as v4_reference_global_aupro; only a local
    numerical-parity test can authorize this evaluator.
    """
    lookup=samples.set_index('sample_id')
    finite_maps=[p['map'] for p in preds if p['map'] is not None]
    if not finite_maps:return np.nan,0
    lo=min(float(np.nanmin(m)) for m in finite_maps)
    hi=max(float(np.nanmax(m)) for m in finite_maps)
    if not np.isfinite(lo+hi) or hi<=lo:return np.nan,0
    edges=np.linspace(lo,hi,bins+1)
    normal_hist=np.zeros(bins,np.float64)
    pro_hist=np.zeros(bins,np.float64)
    regions=0
    live_update(force=True,subtask='Pixel AUPRO analysis',aupro_done=0,aupro_total=len(preds))
    for aupro_i,p in enumerate(tqdm(preds,desc='Pixel AUPRO',unit='map',leave=False),1):
        if aupro_i%30==0:
            live_update(subtask='Pixel AUPRO analysis',aupro_done=aupro_i,aupro_total=len(preds))
        m=p['map']
        if m is None:continue
        r=lookup.loc[p['sample_id']]
        if mask_cache is None:
            fg=cv2.imread(r['fg_path'],cv2.IMREAD_GRAYSCALE)>127
            gt=cv2.imread(r['mask_path'],cv2.IMREAD_GRAYSCALE)>127
        else:fg,gt=mask_cache.masks(p['sample_id'])

        normal_hist+=np.histogram(m[fg&(~gt)],bins=edges)[0]
        if r['label']!=1:continue
        count,labels,stats,_=cv2.connectedComponentsWithStats(gt.astype('uint8'),8)
        if count<2:continue
        valid_ids=np.flatnonzero(stats[:,cv2.CC_STAT_AREA]>=5)
        valid_ids=valid_ids[valid_ids!=0]
        if not len(valid_ids):continue
        valid_region=np.isin(labels,valid_ids)
        ids=labels[valid_region]
        if not ids.size:continue
        values=np.asarray(m)[valid_region]
        buckets=np.searchsorted(edges,values,side='right')-1
        np.clip(buckets,0,bins-1,out=buckets)
        weights=np.zeros(count,np.float64)
        weights[valid_ids]=1./stats[valid_ids,cv2.CC_STAT_AREA]
        pro_hist+=np.bincount(buckets,weights=weights[ids],minlength=bins)
        regions+=len(valid_ids)
    if regions<1 or normal_hist.sum()==0:return np.nan,regions
    fpr=np.r_[0.,np.cumsum(normal_hist[::-1])/normal_hist.sum()]
    pro=np.r_[0.,np.cumsum(pro_hist[::-1])/regions]
    keep=fpr<maximum_fpr
    xx=np.r_[fpr[keep],maximum_fpr]
    yy=np.r_[pro[keep],np.interp(maximum_fpr,fpr,pro)]
    return float(np.trapezoid(yy,xx)/maximum_fpr),regions


def collect_and_score(checkpoint,name,samples,device,extra,output,do_localization=True):
    """Validated prediction cache plus exact-reference pixel metrics."""
    output=Path(output)
    output.parent.mkdir(parents=True,exist_ok=True)
    metadata=output.with_suffix('.csv.provenance.json')
    contract=v4_score_contract(checkpoint,samples,output,do_localization)
    if metadata.exists() and not output.exists():
        raise RuntimeError('Score metadata exists without its CSV; manual audit required: '+str(output))
    if output.is_file() and not metadata.exists() and        output.resolve().is_relative_to((OUT/'results').resolve()):
        quarantine=OUT/'data'/'orphan_locked_test_csv'
        quarantine.mkdir(parents=True,exist_ok=True)
        location=quarantine/f'{output.parent.name}_{output.stem}_{digest(output)[:16]}.csv'
        if location.exists() and digest(location)!=digest(output):
            raise RuntimeError('Quarantined orphan name collision: '+str(location))
        if location.exists():
            output.unlink()
        else:
            os.replace(output,location)
        print('  RECOVERY: orphan CSV quarantined; safely recomputing one locked result '+str(output),flush=True)
    if output.is_file():
        cached=pd.read_csv(output,dtype={'sample_id':str,'source_id':str,'source_group':str})
        v4_verify_score_table(cached,samples,do_localization)
        if metadata.exists():
            old=read_json(metadata)
            if any(old.get(k)!=v for k,v in contract.items()):
                if v4_migrate_verified_p4_matched_cache(metadata,contract,output,name):
                    old=read_json(metadata)
            for key,value in contract.items():
                if old.get(key)!=value:
                    raise RuntimeError('Cached scores have incompatible '+key+': '+str(output)+
                                       '. Do not reuse or overwrite locked results.')
            if old.get('score_csv_sha256')!=digest(output):
                raise RuntimeError('Cached scoring CSV mutated or incomplete: '+str(output))
        else:
            cached=v4_adopt_legacy_score_cache(checkpoint,name,samples,device,extra,
                                                output,do_localization,contract)
        print('  RESUME: provenance-checked scoring '+output.name,flush=True)
        return cached
    start=time.perf_counter()
    if torch.cuda.is_available():torch.cuda.reset_peak_memory_stats()
    mapfile=OUT/'data'/'streaming_pixel_maps'/(
        f'{name}_{Path(checkpoint).parent.name}_{samples.iloc[0]["sample_id"].split("__")[0]}.npy')
    if name.startswith('C'):
        preds=matched_predict(checkpoint,samples,device,keep_maps=do_localization,
                              store_path=mapfile if do_localization else None)
    else:
        preds=baseline_predict(checkpoint,name,samples,device,extra,keep_maps=do_localization)
    inference_elapsed=time.perf_counter()-start
    frame=samples.drop(columns=['image_path','mask_path','fg_path'],errors='ignore').merge(
        pd.DataFrame([{k:v for k,v in p.items() if k!='map'} for p in preds]),
        on='sample_id',validate='one_to_one')
    if len(frame)!=len(samples):raise RuntimeError('Missing predictions')
    if do_localization:
        identity=v4_manifest_hash(samples)
        if identity not in _V4_MASK_CACHE:
            _V4_MASK_CACHE[identity]=V4PackedMaskCache(samples)
        mask_cache=_V4_MASK_CACHE[identity]
        lookup=samples.set_index('sample_id')
        scores={}
        live_update(force=True,subtask='Pixel localization scoring',pixel_done=0,pixel_total=len(preds))
        for pixel_i,p in enumerate(tqdm(preds,desc='Pixel localization',unit='image',leave=False),1):
            if pixel_i%30==0:
                live_update(subtask='Pixel localization scoring',pixel_done=pixel_i,pixel_total=len(preds))
            sid=p['sample_id'];m=p['map'];r=lookup.loc[sid]
            if r['label']!=1 or m is None:
                scores[sid]={'edit_pixel_auroc':np.nan,'edit_pixel_ap':np.nan};continue
            fg,gt=mask_cache.masks(sid)
            if fg.sum()<100 or (fg&gt).sum()<1 or (fg&(~gt)).sum()<1:
                scores[sid]={'edit_pixel_auroc':np.nan,'edit_pixel_ap':np.nan};continue
            y=gt[fg].astype('uint8')
            pred=np.asarray(m,dtype='float32')[fg]
            auc,ap=v4_fast_pixel_auc_ap(y,pred)
            scores[sid]={'edit_pixel_auroc':float(auc),'edit_pixel_ap':float(ap)}
        frame=frame.merge(pd.DataFrame([{'sample_id':k,**v} for k,v in scores.items()]),
                          on='sample_id',validate='one_to_one')
        aupro,regions=global_edit_support_aupro(preds,samples,mask_cache=mask_cache)
        frame['edit_support_aupro_fpr0p30']=aupro
        frame['aupro_count_regions']=regions
    elapsed=time.perf_counter()-start
    frame['evaluation_seconds_total']=float(inference_elapsed)
    frame['mean_inference_ms_including_io_and_loading']=float(inference_elapsed)*1000./len(frame)
    tmp=output.with_suffix('.csv.partial')
    frame.to_csv(tmp,index=False)
    os.replace(tmp,output)
    record=dict(contract,score_csv_sha256=digest(output),cache_origin='provenance_verified',
                inference_seconds=float(inference_elapsed),
                localization_seconds=float(elapsed-inference_elapsed),
                total_seconds=float(elapsed),
                peak_gpu_memory_bytes=(int(torch.cuda.max_memory_allocated()) if
                                       torch.cuda.is_available() else None))
    write_json(metadata,record)
    v4_performance_event({'stage':output.parent.name,'method':name,
                         'seed':Path(checkpoint).parent.name,'images':len(samples),
                         'inference_seconds':inference_elapsed,
                         'pixel_metric_seconds':elapsed-inference_elapsed,
                         'gpu_peak_gib':(record['peak_gpu_memory_bytes']/1024**3 if
                                         record['peak_gpu_memory_bytes'] is not None else None)})
    if do_localization and (name.startswith('C') or V4_ENABLE_STREAMING_BASELINE_INFERENCE):
        import gc
        del preds
        gc.collect()
        mapfile.unlink(missing_ok=True)
    return frame


def macro(frame):
    """Mean of four families each with three separate severities; same 3 negatives per source."""
    neg=frame[frame.label==0]
    families=FAMILIES_TEST if frame.family.astype(str).str.startswith('HO_').any() else FAMILIES_TRAIN
    values=[]
    for fam in families:
        for sev in SEVERITIES:
            pos=frame[(frame.family==fam)&(frame.severity==sev)]
            if len(pos)==0:raise ValueError(f'Absent positive family {fam}/{sev}')
            scores=pd.concat([neg.score,pos.score]);labels=np.r_[np.zeros(len(neg)),np.ones(len(pos))]
            values.append(float(roc_auc_score(labels,scores.to_numpy())))
    return float(np.mean(values))

def subtable(frame):
    neg=frame[frame.label==0];rows=[]
    for (family,sev),pos in frame[frame.label==1].groupby(['family','severity']):
        x=pd.concat([neg,pos]); y=x.label.to_numpy();v=x.score.to_numpy()
        rows.append({'family':family,'severity':sev,'AUROC':roc_auc_score(y,v),
                     'n_source_groups':x.source_group.nunique(),
                     'n_positive_variants':len(pos)})
    return pd.DataFrame(rows)

def calibrate(frame,quantile=.95):
    neg=frame[frame.label==0]
    if len(neg)==0:raise RuntimeError('Calibration missing negatives')

    if not neg.groupby('source_id').size().eq(3).all():raise RuntimeError('Calibration negatives not source-balanced')
    return float(np.quantile(neg.score.to_numpy(),quantile,method='higher'))

def response(frame,threshold):
    neg=frame[frame.label==0];pos=frame[frame.label==1]
    d={'threshold':threshold,'original_fpr':float((neg[neg.kind=='original'].score>threshold).mean()),
       'benign_fpr':float((neg[neg.kind!='original'].score>threshold).mean()),
       'overall_sensitivity':float((pos.score>threshold).mean())}
    for fam,sub in pos.groupby('family'):
        d[f'recall_{fam}']=float((sub.score>threshold).mean())
    return d


def _v4_reference_resampled_auc(frame,weights):
    """Original sklearn estimator retained as the scientific reference implementation."""
    neg=frame[frame.label==0]
    families=FAMILIES_TEST
    nweight=neg.source_group.astype(str).map(weights).fillna(0).to_numpy()
    score=[]
    for fam in families:
        for sev in SEVERITIES:
            pos=frame[(frame.family==fam)&(frame.severity==sev)]
            labels=np.r_[np.zeros(len(neg)),np.ones(len(pos))]
            values=np.r_[neg.score.to_numpy(),pos.score.to_numpy()]
            sample_weight=np.r_[nweight,pos.source_group.astype(str).map(weights).fillna(0).to_numpy()]
            if sample_weight[:len(neg)].sum()==0 or sample_weight[len(neg):].sum()==0:return np.nan
            score.append(roc_auc_score(labels,values,sample_weight=sample_weight))
    return float(np.mean(score))


def _v4_rank_cache(frame):
    """Precompute sorted score/tie groups once; source-group weights vary by draw."""
    groups=tuple(sorted(frame.source_group.astype(str).unique()))
    group_id={g:i for i,g in enumerate(groups)}
    neg=frame[frame.label==0]
    cache=[]
    for fam in FAMILIES_TEST:
        for sev in SEVERITIES:
            pos=frame[(frame.family==fam)&(frame.severity==sev)]
            if len(pos)==0:raise RuntimeError('Missing bootstrap family/severity stratum')
            labs=np.r_[np.zeros(len(neg),np.int8),np.ones(len(pos),np.int8)]
            vals=np.r_[neg.score.to_numpy(),pos.score.to_numpy()]
            g=np.r_[neg.source_group.astype(str).map(group_id).to_numpy(),
                    pos.source_group.astype(str).map(group_id).to_numpy()].astype(np.int32)
            order=np.argsort(vals,kind='mergesort')
            sorted_vals=vals[order]
            tie_starts=np.r_[0,np.flatnonzero(sorted_vals[1:]!=sorted_vals[:-1])+1]
            cache.append((g[order],labs[order],tie_starts))
    return groups,cache


def _v4_fast_resampled_auc(frame,weights):
    """Weighted Mann–Whitney tie formula; same ROC AUC as sklearn in fp tolerance."""
    if '_v4_auroc_sorted_cache' not in frame.attrs:
        frame.attrs['_v4_auroc_sorted_cache']=_v4_rank_cache(frame)
    groups,tables=frame.attrs['_v4_auroc_sorted_cache']
    group_weights=np.asarray([weights.get(k,0) for k in groups],dtype=np.float64)
    if not np.isfinite(group_weights).all() or (group_weights<0).any():
        raise ValueError('Invalid bootstrap weights')
    values=[]
    for grp,labs,ties in tables:
        w=group_weights[grp]
        n=np.add.reduceat(np.where(labs==0,w,0.0),ties)
        p=np.add.reduceat(np.where(labs==1,w,0.0),ties)
        sn=float(np.sum(n));sp=float(np.sum(p))
        if sn==0 or sp==0:return np.nan
        rank_score=float(np.sum(p*(np.cumsum(n)-n+0.5*n)))/(sn*sp)
        values.append(rank_score)
    return float(np.mean(values))


def v4_test_fast_bootstrap():
    """Reject optimization unless weighted/tied AUC matches sklearn numerically."""
    if not V4_ENABLE_TIE_EXACT_FAST_BOOTSTRAP:
        return False

    rng=np.random.default_rng(100904)
    rows=[]
    for g in range(7):
        for fam in ['none']+list(FAMILIES_TEST):
            for sev in (['none'] if fam=='none' else list(SEVERITIES)):
                rows.append({'source_group':f'group{g}', 'source_id':f'group{g}',
                             'family':fam,'severity':sev,'label':int(fam!='none'),
                             'score':float(rng.integers(0,12))/11.0})
    test=pd.DataFrame(rows)
    for rep in range(16):
        w={f'group{g}':int(rng.integers(0,5)) for g in range(7)}
        if sum(w.values())==0:continue
        reference=_v4_reference_resampled_auc(test,w)
        trial=_v4_fast_resampled_auc(test,w)
        if not (np.isnan(reference) and np.isnan(trial)) and not np.isclose(reference,trial,rtol=0,atol=2e-12):
            raise RuntimeError(f'Fast bootstrap not equivalent to sklearn: {reference} vs {trial}')
    return True


def resampled_auc(frame,weights):
    return (_v4_fast_resampled_auc(frame,weights) if V4_ENABLE_TIE_EXACT_FAST_BOOTSTRAP
            else _v4_reference_resampled_auc(frame,weights))



def v4_array_macro_spec(frame):
    neg=np.flatnonzero(frame.label.to_numpy()==0)
    fam=frame.family.astype(str).to_numpy()
    sev=frame.severity.astype(str).to_numpy()
    output=[]
    for family in FAMILIES_TEST:
        for severity in SEVERITIES:
            pos=np.flatnonzero((fam==family)&(sev==severity))
            if not len(pos):raise RuntimeError('Missing family/severity for fast swap')
            index=np.r_[neg,pos]
            labels=np.r_[np.zeros(len(neg),np.uint8),np.ones(len(pos),np.uint8)]
            output.append((index,labels))
    return output


def v4_array_macro(values,spec):
    return float(np.mean([v4_fast_pixel_auc_ap(labels,values[index])[0]
                          for index,labels in spec]))


def v4_permutation_reference_preflight():
    rng=np.random.default_rng(20261015)
    rows=[]
    for group in range(5):
        for kind in ['original','benign_0','benign_1']:
            rows.append({'source_group':str(group),'sample_id':f'{group}_{kind}',
                         'family':'none','severity':'none','label':0,'kind':kind,
                         'score':float(rng.integers(0,20))/20.})
        for family in FAMILIES_TEST:
            for severity in SEVERITIES:
                rows.append({'source_group':str(group),'sample_id':f'{group}_{family}_{severity}',
                             'family':family,'severity':severity,'label':1,
                             'kind':family+'_'+severity,'score':float(rng.integers(0,20))/20.})
    a=pd.DataFrame(rows);b=a.copy()
    b['score']=rng.integers(0,20,size=len(b))/20.
    spec=v4_array_macro_spec(a)
    for i in range(18):
        switches={str(g):int(rng.integers(0,2)) for g in range(5)}
        x=a.copy();y=b.copy()
        chosen=a.source_group.map(switches).to_numpy().astype(bool)
        x.loc[chosen,'score']=b.loc[chosen,'score'].to_numpy()
        y.loc[chosen,'score']=a.loc[chosen,'score'].to_numpy()
        reference=macro(x)-macro(y)
        fast=(v4_array_macro(x.score.to_numpy(),spec)-v4_array_macro(y.score.to_numpy(),spec))
        if abs(reference-fast)>2e-12:
            raise RuntimeError('Fast paired permutation fails sklearn reference')
    return True


def bootstrap_comparison(all_predictions,method_a,method_b,B=2000,seed=20261107,permutations=999):
    """Paired source-cluster bootstrap + paired source-group swap randomization.
    Aggregate across model seeds WITHIN every resample; do not treat seed as a fruit specimen.
    """
    preds_a=[v for (m,s),v in all_predictions.items() if m==method_a]
    preds_b=[v for (m,s),v in all_predictions.items() if m==method_b]
    if len(preds_a)!=len(preds_b):raise RuntimeError('Seed mismatch in paired analysis')
    groups=sorted(set(preds_a[0].source_group.astype(str)))
    rng=np.random.default_rng(seed)
    observed=float(np.mean([macro(a)-macro(b) for a,b in zip(preds_a,preds_b)]))
    boot=[];invalid=0
    for _ in tqdm(range(B),desc="Source-group bootstrap",unit="replicate",leave=False):
        counts=pd.Series(rng.choice(groups,len(groups),replace=True)).value_counts().to_dict()
        scores=[]
        for a,b in zip(preds_a,preds_b):
            x=resampled_auc(a,counts);y=resampled_auc(b,counts)
            if not (np.isfinite(x) and np.isfinite(y)):invalid+=1;continue
            scores.append(x-y)
        boot.append(float(np.mean(scores)) if len(scores)==len(preds_a) else np.nan)
    bb=np.array(boot);good=bb[np.isfinite(bb)]
    if len(good)<.99*B:raise RuntimeError('Excess invalid cluster bootstrap replicates')

    swaps=[]
    group_index={g:i for i,g in enumerate(groups)}
    paired=[]
    for a,b in zip(preds_a,preds_b):
        key=['source_group','sample_id','family','severity','kind','label']
        if not a[key].reset_index(drop=True).equals(b[key].reset_index(drop=True)):
            raise RuntimeError('Cannot permute unmatched prediction rows')
        ids=a.source_group.astype(str).map(group_index).to_numpy(dtype=np.int32)
        paired.append((a.score.to_numpy(dtype=np.float64),
                       b.score.to_numpy(dtype=np.float64),ids,v4_array_macro_spec(a)))
    for _ in tqdm(range(permutations),desc="Paired permutation",unit="replicate",leave=False):
        switch_group=rng.integers(0,2,len(groups))
        ds=[]
        for ascore,bscore,ids,spec in paired:
            switch=switch_group[ids].astype(bool)
            x=np.where(switch,bscore,ascore)
            y=np.where(switch,ascore,bscore)
            ds.append(v4_array_macro(x,spec)-v4_array_macro(y,spec))
        swaps.append(float(np.mean(ds)))
    p=(1+sum(abs(x)>=abs(observed) for x in swaps))/(1+permutations)
    return {'comparison':method_a+' minus '+method_b,'observed_delta':observed,
            'ci_low':float(np.quantile(good,.025)),'ci_high':float(np.quantile(good,.975)),
            'bootstrap_B':B,'invalid_bootstrap':int(B-len(good)),
            'paired_group_swap_p':float(p),'permutations':permutations,
            'ci_method':'percentile_cluster_bootstrap','groups':len(groups)}


def fixed_prior_pr_auc(frame,prevalence):
    """ROC-reweighted PR integral under ASSUMED prevalence, not empirical synthetic prevalence."""
    neg=frame[frame.label==0]; estimates=[]
    for (fam,sev),pos in frame[frame.label==1].groupby(['family','severity']):
        data=pd.concat([neg,pos]);fpr,tpr,_=roc_curve(data.label.to_numpy(),data.score.to_numpy())
        denom=prevalence*tpr+(1-prevalence)*fpr
        precision=np.divide(prevalence*tpr,denom,out=np.ones_like(tpr),where=denom>0)

        estimates.append(float(np.trapezoid(precision,tpr)))
    if len(estimates)!=12:raise RuntimeError('Fixed-prevalence PR metric requires exactly twelve family-severity strata')
    return float(np.mean(estimates))

def subgroup_metrics(frame,group_col):
    """Descriptive strata; not confirmatory because subgroup sizes may differ."""
    rows=[]
    for name,g in frame.groupby(group_col,dropna=False):
        if g.source_group.nunique()<8:continue
        try:val=macro(g)
        except ValueError:continue
        rows.append({group_col:name,'n_source_groups':g.source_group.nunique(),'macro_auroc':val})
    return pd.DataFrame(rows)






V4_ENABLE_STREAMING_BASELINE_INFERENCE = True
V4_REQUESTED_INFERENCE_BATCH = 8
V4_PREDICTION_PARITY_ATOL = 1e-7
V4_PREDICTION_PARITY_RTOL = 1e-6
_V4_EVAL_BATCH_BY_METHOD = {}


def _v4_prediction_collator(checkpoint, name, image_folder, device, extra,
                            keep_maps=False, expected_ids=None, map_store_path=None,
                            eval_batch=1):
    """Bounded prediction transport; no output tensors accumulated by Lightning."""
    from anomalib.data import PredictDataset
    from anomalib.engine import Engine
    from lightning.pytorch.callbacks import Callback
    import gc, shutil
    from torch.utils.data import DataLoader

    folder=Path(image_folder)
    dataset=PredictDataset(folder)
    loader=DataLoader(dataset,collate_fn=dataset.collate_fn,
                      batch_size=eval_batch,num_workers=0,pin_memory=True)
    if expected_ids is None:
        expected_ids=[p.stem for p in sorted(folder.glob('*.png'))]
    expected_ids=list(expected_ids)
    if len(set(expected_ids))!=len(expected_ids):
        raise RuntimeError('Duplicate expected image IDs in the streaming predictor')
    index={s:i for i,s in enumerate(expected_ids)}
    expected_n=len(index)
    if len(dataset)!=expected_n:
        raise RuntimeError('Prediction folder has a different number of images than benchmark; abort')
    if keep_maps and map_store_path is None:
        raise ValueError('Streaming localization requires a temporary map store path')
    if keep_maps:
        needed=expected_n*256*256*4
        disk=shutil.disk_usage(Path(map_store_path).parent)
        if disk.free<needed+2*1024**3:
            raise RuntimeError(f'Insufficient disk space to stream pixel maps; need >{(needed+2*1024**3)/1024**3:.1f} GiB')
        maps_out=np.lib.format.open_memmap(map_store_path,mode='w+',dtype=np.float32,
                                           shape=(expected_n,256,256))
    else:
        maps_out=None
    answers={}
    class Collector(Callback):
        def on_predict_epoch_start(self, trainer, pl_module):






            callbacks = trainer.callbacks
            if callbacks.count(self) != 1:
                raise RuntimeError('Streaming collector has incorrect callback registration')
            if callbacks[-1] is not self:
                trainer.callbacks = [c for c in callbacks if c is not self] + [self]
            if trainer.callbacks[-1] is not self:
                raise RuntimeError('Unable to place streaming collector after post-processing')
            postprocessors = [c for c in trainer.callbacks[:-1]
                              if 'postprocessor' in type(c).__name__.lower()]
            if not postprocessors:
                raise RuntimeError('No Anomalib post-processor callback found before streaming collector; '
                                   'cannot guarantee equivalent normalized predictions')
            print(f'  POST-PROCESSED STREAMING: {name}: collector positioned after '
                  f'{len(postprocessors)} Anomalib post-processor(s)',flush=True)

        def on_predict_batch_end(self,trainer,pl_module,outputs,batch,batch_idx,dataloader_idx=0):
            if trainer.callbacks[-1] is not self:
                raise RuntimeError('Streaming collector was reordered ahead of model post-processing')
            if outputs is None:
                raise RuntimeError('Anomalib prediction callback received no output')
            paths=outputs.image_path
            if isinstance(paths,(str,Path)):paths=[str(paths)]


            scores=outputs.pred_score.detach().cpu().numpy().reshape(-1)
            maps=outputs.anomaly_map
            maps_cpu=(maps.detach().cpu().numpy() if keep_maps and maps is not None else None)
            for k,path in enumerate(paths):
                sid=Path(path).stem
                if sid not in index:raise RuntimeError('Unexpected prediction basename: '+sid)
                if sid in answers:raise RuntimeError('Repeated prediction basename: '+sid)
                score=float(scores[k])
                if not np.isfinite(score):raise RuntimeError('Nonfinite image anomaly score')
                if keep_maps:
                    if maps_cpu is None:raise RuntimeError('No pixel maps returned for localization')
                    m=np.squeeze(maps_cpu[k])
                    m=cv2.resize(m.astype('float32'),(256,256))
                    if not np.isfinite(m).all():raise RuntimeError('Nonfinite anomaly map')
                    maps_out[index[sid]]=m
                answers[sid]={'score':score}
            live_update(subtask=f'Streaming baseline inference: {name}',
                        scoring_samples_done=len(answers),scoring_samples_total=expected_n)
    model=build_model(name,extra)


    engine=Engine(default_root_dir=str(Path(checkpoint).parent/'prediction_engine'),
                  accelerator='gpu' if str(device).startswith('cuda') else 'cpu',
                  devices=1,logger=False,enable_checkpointing=True,
                  enable_progress_bar=False,callbacks=[Collector()])
    live_update(force=True,subtask=f'Bounded prediction batches ({name}, batch={eval_batch})',
                scoring_samples_done=0,scoring_samples_total=expected_n)


    returned=engine.predict(model=model,ckpt_path=str(checkpoint),
                            dataloaders=loader,return_predictions=False)
    if returned not in (None,[]):
        raise RuntimeError('Expected no accumulated predictions in streaming mode')
    if set(answers)!=set(index):
        raise RuntimeError(f'Streamed only {len(answers)}/{expected_n} predictions')
    if maps_out is not None:
        maps_out.flush()
        for key in answers:
            answers[key]['map']=maps_out[index[key]]
    else:
        for key in answers:answers[key]['map']=None
    del engine,model,loader,dataset
    gc.collect()
    if str(device).startswith('cuda'):torch.cuda.empty_cache()
    return answers


def _v4_verify_inference_parity(ckpt, name, samples, device, extra):
    """Authorize bounded streaming only when final POST-PROCESSED values match Anomalib.

    Every method/checkpoint gets a development-only reference comparison. Streaming
    batch 1 must match the original Engine.predict() BEFORE any larger batch is tried.
    Try 8, 4 and 2 and select the fastest plausible (largest) batch whose scores
    AND FULL anomaly maps match to the original tolerances. Never use locked test
    predictions for parity selection. No normalization or training changes.
    """
    import tempfile, gc
    checkpoint_sha=v4_file_sha_cached(ckpt)
    approval_key=f'{name}_{checkpoint_sha[:24]}'
    output=OUT/'data'/'inference_parity'/f'{approval_key}.json'
    if approval_key in _V4_EVAL_BATCH_BY_METHOD:
        return _V4_EVAL_BATCH_BY_METHOD[approval_key]
    if output.is_file():
        approved_doc=read_json(output)
        for k,v in {'method':name,'checkpoint_sha256':checkpoint_sha,
                    'requested_batch_size':V4_REQUESTED_INFERENCE_BATCH,
                    'tolerance_atol':V4_PREDICTION_PARITY_ATOL,
                    'tolerance_rtol':V4_PREDICTION_PARITY_RTOL}.items():
            if approved_doc.get(k)!=v:
                raise RuntimeError('Stored inference parity approval mismatches '+k+': '+str(output))
        if approved_doc.get('capture_stage')!='after_anomalib_postprocessing_v2':
            raise RuntimeError('Old streaming parity approval lacks post-processing proof; '
                               'must revalidate on development data before locked access')
        batch=int(approved_doc['selected_batch_size'])
        if batch not in (1,2,4,V4_REQUESTED_INFERENCE_BATCH):
            raise RuntimeError('Invalid persisted inference batch size')
        _V4_EVAL_BATCH_BY_METHOD[approval_key]=batch
        return batch
    if (OUT/'frozen'/'protocol_lock.json').exists() or (OUT/'FINAL_TEST_USED.flag').exists():
        raise RuntimeError('Missing VERIFIED pre-lock development inference parity approval for '+approval_key+
                           '; refusing to select inference batch after lock')
    if samples.iloc[0].get('family','') in FAMILIES_TEST:
        raise RuntimeError('Inference transport not verified before locked test; run development first')

    def parity_stats(reference,got,ids):
        if set(got)!=set(reference):
            return False, {'reason':'sample ID mismatch','reference_ids':len(reference),'streamed_ids':len(got)}
        okay=True; max_map=0.; max_score=0.; worst_id=None; first_reason=None
        for sid in ids:
            a,b=reference[sid],got[sid]
            score_diff=abs(float(a['score'])-float(b['score']))
            map_diff=float(np.max(np.abs(a['map']-b['map'])))
            if max(score_diff,map_diff)>max(max_score,max_map):worst_id=sid
            max_score=max(max_score,score_diff)
            max_map=max(max_map,map_diff)
            score_ok=bool(np.isclose(a['score'],b['score'],rtol=V4_PREDICTION_PARITY_RTOL,
                                     atol=V4_PREDICTION_PARITY_ATOL))
            map_ok=bool(np.allclose(a['map'],b['map'],rtol=V4_PREDICTION_PARITY_RTOL,
                                     atol=V4_PREDICTION_PARITY_ATOL,equal_nan=False))
            if not (score_ok and map_ok):
                okay=False
                if first_reason is None:first_reason={'sample_id':sid,'score_ok':score_ok,'map_ok':map_ok}
        return okay,{'max_abs_score_difference':max_score,'max_abs_map_difference':max_map,
                     'worst_sample_id':worst_id,'first_mismatch':first_reason}

    with tempfile.TemporaryDirectory(prefix='v4_postprocess_parity_',dir=OUT/'data') as scratch:
        root=Path(scratch)
        chosen=samples.iloc[:min(24,len(samples))]
        for row in chosen.to_dict('records'):
            src=Path(row['image_path']);dest=root/src.name
            if dest.exists():raise RuntimeError('Nonunique prediction smoke-test filenames')
            try:dest.symlink_to(src)
            except OSError:
                import shutil
                shutil.copy2(src,dest)
        reference=predict_from_checkpoint(ckpt,name,root,device,extra,keep_maps=True)
        ids=chosen.sample_id.astype(str).tolist()
        approved=1; checks=[]
        trials=[1]+[b for b in (V4_REQUESTED_INFERENCE_BATCH,4,2)
                    if b>1 and b<=V4_REQUESTED_INFERENCE_BATCH]
        for trial in trials:
            trial_maps=root/f'smoke_batch_{trial}_maps.npy'
            got=None
            try:
                got=_v4_prediction_collator(ckpt,name,root,device,extra,
                                            keep_maps=True,expected_ids=ids,
                                            map_store_path=trial_maps,eval_batch=trial)
                okay,stats=parity_stats(reference,got,ids)
                checks.append({'batch_size':trial,'passed':bool(okay),**stats})
                if trial==1 and not okay:
                    raise RuntimeError('Batch-1 streaming is NOT numerically equivalent to original '
                                       f'Anomalib post-processed prediction: {stats}. '
                                       'No baseline scoring was performed; preserve all trained checkpoints.')
                if okay:
                    approved=trial
                    print(f'  INFERENCE PARITY PASS: {name} batch={trial}; '
                          f'max score diff={stats["max_abs_score_difference"]:.3g}; '
                          f'max map diff={stats["max_abs_map_difference"]:.3g}',flush=True)
                    if trial!=1:break
                else:
                    print(f'  INFERENCE PARITY: {name} batch={trial} differs; '
                          f'trying smaller batch; {stats}',flush=True)
            except (RuntimeError,ValueError) as exc:
                if 'out of memory' not in str(exc).lower():raise
                if trial==1:raise RuntimeError('Even batch-1 baseline prediction is OOM') from exc
                checks.append({'batch_size':trial,'passed':False,'reason':'out of memory'})
                print(f'  INFERENCE BATCH OOM: {name} batch={trial}; trying smaller',flush=True)
                torch.cuda.empty_cache()
            finally:
                if got is not None:del got
                gc.collect()
                trial_maps.unlink(missing_ok=True)
        print(f'  INFERENCE APPROVAL: {name}, selected batch={approved} after final-value parity',flush=True)
    output.parent.mkdir(parents=True,exist_ok=True)
    write_json(output,
               {'method':name,'checkpoint_sha256':checkpoint_sha,'source_stage':'development',
                'requested_batch_size':V4_REQUESTED_INFERENCE_BATCH,
                'selected_batch_size':approved,'dev_reference_batch_size':1,
                'tolerance_atol':V4_PREDICTION_PARITY_ATOL,'tolerance_rtol':V4_PREDICTION_PARITY_RTOL,
                'capture_stage':'after_anomalib_postprocessing_v2','comparisons':checks,
                'no_locked_test_used_to_select_inference_batch':True})
    _V4_EVAL_BATCH_BY_METHOD[approval_key]=approved
    return approved


In [ ]:
def render_figures(summary,per_family,out,source_split,response_table,comparisons):
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    figs=out/'figures';figs.mkdir(exist_ok=True)
    split_order=['train','development','calibration','locked_test']
    splits=pd.read_csv(source_split)
    counts=splits[ splits.eligibility=='eligible'].groupby('split').source_group.nunique()
    plt.figure(figsize=(7,4))
    plt.bar(split_order,[counts.get(x,0) for x in split_order])
    plt.ylabel('Independent screened source groups');plt.title('New source-group partition')
    plt.tight_layout();plt.savefig(figs/'figure_1_source_splits.png',dpi=170);plt.close()
    x=summary.sort_values('macro_auroc_mean',ascending=False)
    plt.figure(figsize=(10,4.5));plt.bar(x.method,x.macro_auroc_mean)
    plt.ylabel('Held-out family/severity macro AUROC');plt.xticks(rotation=55,ha='right');plt.ylim(0,1)
    plt.tight_layout();plt.savefig(figs/'figure_2_macro_auroc.png',dpi=170);plt.close()
    a=per_family.groupby(['method','family']).AUROC.mean().unstack()
    plt.figure(figsize=(8,5));plt.imshow(a.to_numpy(),vmin=0,vmax=1,cmap='viridis',aspect='auto')
    plt.xticks(range(len(a.columns)),a.columns);plt.yticks(range(len(a.index)),a.index)
    plt.colorbar(label='AUROC');plt.tight_layout();plt.savefig(figs/'figure_3_family_auroc.png',dpi=170);plt.close()
    x=per_family[per_family.method=='C11'].groupby(['family','severity']).AUROC.mean().unstack()
    x=x.reindex(columns=['low','medium','high'])
    plt.figure(figsize=(7,4));plt.imshow(x.to_numpy(),vmin=0,vmax=1,cmap='viridis',aspect='auto')
    plt.xticks(range(3),['low','medium','high']);plt.yticks(range(len(x.index)),x.index)
    plt.colorbar(label='AUROC');plt.title('Proposed model: held-out severity')
    plt.tight_layout();plt.savefig(figs/'figure_4_severity_by_family.png',dpi=170);plt.close()
    p=response_table.groupby('method')[['benign_fpr','overall_sensitivity']].mean().reset_index()
    plt.figure(figsize=(6.5,5));plt.scatter(p.benign_fpr,p.overall_sensitivity)
    for r in p.itertuples():plt.annotate(r.method,(r.benign_fpr,r.overall_sensitivity),xytext=(3,3),textcoords='offset points',fontsize=8)
    plt.xlabel('Benign false rejection at locked threshold');plt.ylabel('Synthetic recall')
    plt.tight_layout();plt.savefig(figs/'figure_5_fixed_operating_point.png',dpi=170);plt.close()
    p=summary.sort_values('pixel_ap_mean',ascending=False)
    plt.figure(figsize=(10,4.5));plt.bar(p.method,p.pixel_ap_mean)
    plt.ylabel('Mean synthetic edit-support pixel AP');plt.xticks(rotation=55,ha='right')
    plt.tight_layout();plt.savefig(figs/'figure_6_edit_localization.png',dpi=170);plt.close()
    labels=[q['comparison'] for q in comparisons];values=[q['observed_delta'] for q in comparisons]
    ci_lo=[v-q['ci_low'] for v,q in zip(values,comparisons)];ci_hi=[q['ci_high']-v for v,q in zip(values,comparisons)]
    plt.figure(figsize=(8,3.6));plt.errorbar(values,range(len(values)),xerr=[ci_lo,ci_hi],fmt='o',capsize=4)
    plt.yticks(range(len(labels)),labels);plt.axvline(0,color='black',linestyle='--')
    plt.xlabel('Paired macro AUROC difference (source-group 95% CI)')
    plt.tight_layout();plt.savefig(figs/'figure_7_paired_effects.png',dpi=170);plt.close()
    p=summary.sort_values('image_inference_latency_ms')
    plt.figure(figsize=(8,5));plt.scatter(p.image_inference_latency_ms,p.macro_auroc_mean)
    for r in p.itertuples():plt.annotate(r.method,(r.image_inference_latency_ms,r.macro_auroc_mean),xytext=(3,3),textcoords='offset points',fontsize=8)
    plt.xscale('log');plt.xlabel('Mean timing per image, incl. model load and I/O (ms; not hardware-normalized)')
    plt.ylabel('Held-out macro AUROC')
    plt.tight_layout();plt.savefig(figs/'figure_8_performance_timing.png',dpi=170);plt.close()

def final_report(out,predictions,thresholds,comparator,smoke):
    methods=sorted({m for m,s in predictions});rows=[];family_rows=[];response_rows=[];subgroup_rows=[]
    for (method,seed),frame in predictions.items():
        rows.append({'method':method,'seed':seed,'macro_auroc':macro(frame),
                     'assumed_10pct_pr_auc':fixed_prior_pr_auc(frame,.10),
                     'assumed_1pct_pr_auc':fixed_prior_pr_auc(frame,.01),
                     'mean_inference_ms_including_io':float(frame.mean_inference_ms_including_io_and_loading.iloc[0]),
                     'edit_support_aupro':float(frame.edit_support_aupro_fpr0p30.iloc[0]) if 'edit_support_aupro_fpr0p30' in frame else np.nan,
                     'pixel_auroc_mean':float(frame.edit_pixel_auroc.mean()) if 'edit_pixel_auroc' in frame else np.nan,
                     'pixel_ap_mean':float(frame.edit_pixel_ap.mean()) if 'edit_pixel_ap' in frame else np.nan})
        t=subtable(frame);t['method']=method;t['seed']=seed;family_rows.append(t)
        z=response(frame,thresholds[f'{method}_{seed}']);z.update(method=method,seed=seed);response_rows.append(z)
        for column in ['source_variety','source_size']:
            sg=subgroup_metrics(frame,column)
            if not sg.empty:
                sg['method']=method;sg['seed']=seed;sg['stratum_type']=column
                sg=sg.rename(columns={column:'stratum_value'});subgroup_rows.append(sg)
    ss=pd.DataFrame(rows);ss.to_csv(out/'reports'/'all_seed_metrics.csv',index=False)
    family=pd.concat(family_rows,ignore_index=True);family.to_csv(out/'reports'/'family_severity_scores.csv',index=False)
    split_df=pd.read_csv(out/'data'/'source_split.csv')
    split_df.groupby(['split','variety','native_size_label'],dropna=False).agg(
        original_images=('source_id','size'),unique_source_groups=('source_group','nunique')
    ).reset_index().to_csv(out/'reports'/'source_split_strata_table.csv',index=False)
    bench=pd.read_csv(out/'results'/'fixed_benchmark'/'manifest.csv')
    bench[bench.label==1].groupby(['family','severity']).agg(
        generated_variants=('sample_id','size'),unique_source_groups=('source_group','nunique'),
        median_support_fraction=('support_fraction','median'),
        mean_support_fraction=('support_fraction','mean'),
        mean_abs_rgb_change=('mean_abs_change','mean'),
        off_fruit_pixels=('off_fruit_pixels','sum')
    ).reset_index().to_csv(out/'reports'/'synthetic_operator_severity_table.csv',index=False)
    pd.DataFrame(response_rows).to_csv(out/'reports'/'calibration_fixed_operating_points.csv',index=False)
    if subgroup_rows:pd.concat(subgroup_rows,ignore_index=True).to_csv(out/'reports'/'source_subgroup_metrics.csv',index=False)
    aggregate=ss.groupby('method').agg(macro_auroc_mean=('macro_auroc','mean'),
                                      macro_auroc_sd=('macro_auroc','std'),
                                      assumed_10pct_pr_auc=('assumed_10pct_pr_auc','mean'),
                                      assumed_1pct_pr_auc=('assumed_1pct_pr_auc','mean'),
                                      image_inference_latency_ms=('mean_inference_ms_including_io','mean'),
                                      edit_support_aupro=('edit_support_aupro','mean'),
                                      pixel_auroc_mean=('pixel_auroc_mean','mean'),
                                      pixel_ap_mean=('pixel_ap_mean','mean')).reset_index()
    aggregate.to_csv(out/'reports'/'publication_primary_table.csv',index=False)
    if smoke:
        write_json(out/'reports'/'FINAL_DECISION.json',{'status':'SMOKE_ONLY_NO_SCIENTIFIC_CONCLUSIONS'})
        return
    comparisons=[]
    for m in ['C00',comparator]:


        key={'comparison':['C11',m], 'B':2000,'seed':20261107,'permutations':999,
             'inputs':{f'{model}_{seed}':digest(out/'results'/f'{model}_{seed}.csv')
                       for (model,seed) in predictions if model in ('C11',m)},
             'code':hashlib.sha256(repr(_stable_code(bootstrap_comparison.__code__)).encode()).hexdigest()}
        cache=out/'reports'/f'paired_stats_C11_vs_{m}.json'
        if cache.exists():
            previous=read_json(cache)
            if previous.get('contract')!=key:
                raise RuntimeError('Cached paired test provenance mismatch: '+str(cache))
            comps=previous['result']
            print('  RESUME: paired-source-group comparison '+m,flush=True)
        else:
            comps=bootstrap_comparison(predictions,'C11',m,B=2000,seed=20261107,permutations=999)
            write_json(cache,{'contract':key,'result':comps})
        comparisons.append(comps)
    order=np.argsort([x['paired_group_swap_p'] for x in comparisons]);ps=np.array([x['paired_group_swap_p'] for x in comparisons])
    adjusted=np.empty(2,float);running=0.
    for rank,i in enumerate(order):
        running=max(running,(2-rank)*ps[i]);adjusted[i]=min(1.,running)
    for r,p in zip(comparisons,adjusted):r['holm_p']=float(p)
    compdf=pd.DataFrame(comparisons);compdf.to_csv(out/'reports'/'primary_paired_tests.csv',index=False)
    render_figures(aggregate,family,out,out/'data'/'source_split.csv',pd.DataFrame(response_rows),comparisons)
    claim=all((r['ci_low']>0 and r['holm_p']<.05) for r in comparisons)
    write_json(out/'reports'/'FINAL_DECISION.json',{
        'status':'SCIENTIFIC_RUN_FINISHED',
        'supports_registered_superiority':claim,
        'strongest_competitor_preselected_on_development':comparator,
        'all_mandatory_baselines_present':set(BASELINES).issubset(methods),
        'five_seed_coverage':all(sum(m==m2 for m2,s in predictions)==5 for m in methods),
        'limitations':['No real-defect generalization established','No untouched raw dataset history',
                       'Unverified specimen-level grouping','SuperSimpleNet is a prospective comparator amendment'],
        'note':'A completed experiment can legitimately have null or negative results; no guarantee of novelty.'})



In [ ]:

import time, traceback
VARIANTS = ['C00','C10','C01','C11']


def status(stage, message):
    timestamp=time.strftime('%Y-%m-%d %H:%M:%S')
    line=f'[{timestamp}] {stage} | {message}'
    print('\n'+line,flush=True)
    with (OUT/'pipeline_progress.log').open('a') as f:
        f.write(line+'\n')
    live_update(force=True,stage=stage,stage_message=message,subtask=message)


def source_and_split():
    folder=OUT/'data';folder.mkdir(exist_ok=True)
    csv=folder/'source_split.csv'
    if csv.exists():
        status('P0','Reusing existing source-group CSV; no duplicate scan repeated')
        df=pd.read_csv(csv,dtype={'source_id':str,'source_group':str,'relative_path':str})
        for r in tqdm(df.to_dict('records'),desc='Verify original image SHA256',unit='file'):
            fp=RAW/r['relative_path']
            if not fp.is_file() or digest(fp)!=r['file_sha256']:
                raise RuntimeError('Original image missing or altered since split CSV: '+str(fp))
    else:
        status('P0','Reading original photos and screening duplicate groups')
        raw,ledger=scan(RAW,embedding_screen=True)
        raw.to_csv(folder/'raw_inventory.csv',index=False)
        ledger.to_csv(folder/'duplicate_candidate_ledger.csv',index=False)
        df=allocate(raw,seed=SEED_SPLIT)
        tmp=csv.with_suffix('.csv.tmp')
        df.to_csv(tmp,index=False)
        os.replace(tmp,csv)
    if not {'source_id','source_group','split','eligibility','relative_path','file_sha256'}.issubset(df.columns):
        raise RuntimeError('Source split CSV missing mandatory columns')
    eligible=df[df.eligibility=='eligible'].copy()
    if not eligible.groupby('source_group')['split'].nunique().le(1).all():
        raise RuntimeError('LEAKAGE: duplicate group assigned to multiple partitions')
    counts=eligible.groupby('split').agg(images=('source_id','size'),groups=('source_group','nunique'))
    for name in ['train','development','calibration','locked_test']:
        if name not in counts.index:raise RuntimeError('Required split empty: '+name)
    summary={'source_images':len(df),'eligible_images':len(eligible),
             'groups':int(eligible.source_group.nunique()),
             'quarantined':int((df.eligibility=='ambiguous_near_duplicate').sum()),
             'partition_counts':{str(k):int(v) for k,v in df.groupby('split',dropna=False).size().items()},
             'size_labels':{str(k):int(v) for k,v in df.native_size_label.value_counts(dropna=False).items()},
             'varieties':{str(k):int(v) for k,v in df.variety.value_counts(dropna=False).items()}}
    write_json(folder/'source_summary.json',summary)
    print('\nP0 finished. Frozen assignments in:',csv,flush=True)
    display(counts)
    if eligible.source_group.nunique()<80:
        raise RuntimeError('Too few eligible independent source groups')
    return df


def select_consistency_weight(df, device, config):
    path=OUT/'development'/'lambda_selected.json'
    if path.exists():
        chosen=float(read_json(path)['lambda_cons'])
        status('TUNING',f'Reusing development-selected consistency weight λ={chosen}')
        return chosen
    status('TUNING','Selecting nuisance-consistency weight on training/development data only')
    rows=[]
    for lam in tqdm([.05,.10,.20],desc='Consistency weight candidates'):
        trial=dict(config,lambda_cons=lam,max_epochs=12,patience=4)
        for variant in ['C01','C11']:
            ck=fit_matched(df,RAW,OUT/'development'/'consistency_search'/f'lambda_{lam}'/variant,
                           variant,SEEDS[0],trial,device,epochs_override=12)
            info=torch.load(ck,map_location='cpu',weights_only=False)
            rows.append(dict(lambda_cons=lam,variant=variant,dev_macro_auroc=info['dev_macro_auroc']))
    chosen=float(pd.DataFrame(rows).groupby('lambda_cons').dev_macro_auroc.mean().idxmax())
    write_json(path,{'lambda_cons':chosen,'grid':[.05,.10,.20],'development_results':rows})
    return chosen


def _stable_code(code):
    import types
    return (code.co_code.hex(),code.co_names,code.co_varnames,
            [_stable_code(c) if isinstance(c,types.CodeType) else repr(c) for c in code.co_consts])


def code_fingerprint():

    names=['digest','write_json','image_read','foreground','orientation','choose_point',
           'finish','benign','_train_generate_once','_heldout_generate_once','train_generate','heldout_generate','hashes','pairs_from_hashes',
           'embedding_candidates','orb_verified','scan','allocate','run_training_qc',
           'build_artifact_traps','seg_loss','fit_matched','dev_metric','load_model',
           'cpu_quota','v4_loader','v4_worker_init','gpu_quantile_99','prepare_gpu_dev_cache',
           'build_training_tree','build_model','baseline_fit_spec','baseline_recovery_callback','fit_baseline','predict_from_checkpoint',
           '_v4_prediction_collator','_v4_verify_inference_parity',
           '_v4_build_fast_efficientad_step','_v4_fast_efficientad_parity','_v4_install_cached_imagenet_normalization','enable_v4_efficientad_acceleration',
           '_v4_build_fast_draem_step','_v4_draem_fast_logging_parity','_v4_install_draem_texture_cache','enable_v4_draem_safe_acceleration',
           '_v4_reference_resampled_auc','_v4_rank_cache','_v4_fast_resampled_auc','v4_test_fast_bootstrap',
           'enable_budgeted_patchcore','baseline_resource_preflight',
           'quiet_patchcore_coreset_progress',
           'build_benchmark','matched_predict','baseline_predict','collect_and_score',
           'macro','calibrate','bootstrap_comparison','global_edit_support_aupro',
           'v4_fast_pixel_auc_ap','v4_pixel_metric_preflight',
           'v4_manifest_hash','v4_score_contract','v4_verify_score_table',
           'v4_verify_benchmark_materialization','v4_runtime_versions',
           'v4_adopt_legacy_score_cache','v4_reference_global_aupro',
           'v4_array_macro_spec','v4_array_macro','v4_permutation_reference_preflight',
           'final_report','render_figures','source_and_split','select_consistency_weight']
    content={name:_stable_code(globals()[name].__code__) for name in names}

    content['quiet_patchcore_coreset_progress.body']=_stable_code(
        quiet_patchcore_coreset_progress.__wrapped__.__code__)
    for cls in [DSU,Decoder,Samples,BenchmarkImageDataset,PatchCandidateStore,V4PackedMaskCache]:
        for name,item in vars(cls).items():
            if callable(item) and hasattr(item,'__code__'):
                content[cls.__name__+'.'+name]=_stable_code(item.__code__)
    return hashlib.sha256(json.dumps(content,sort_keys=True,default=str).encode()).hexdigest()


def freeze_or_verify(ckpts, thresholds, comparator, config):
    path=OUT/'frozen'/'protocol_lock.json'
    hashes={f'{k}_{s}':digest(p) for (k,s),p in ckpts.items()}
    setting={'source_split_sha256':digest(OUT/'data'/'source_split.csv'),
             'thresholds_sha256':digest(OUT/'calibration'/'thresholds.json'),
             'checkpoints_sha256':hashes,'comparator':comparator,
             'config':config,'training_seeds':SEEDS,
             'train_mechanisms':FAMILIES_TRAIN,'heldout_mechanisms':FAMILIES_TEST,
             'code_fingerprint':code_fingerprint(),'baseline_patchcore_budget':{'candidates_per_image':PATCHCORE_CANDIDATES_PER_IMAGE,'coreset_ratio':PATCHCORE_CORESET_RATIO,'amendment':PATCHCORE_AMENDMENT},
             'baseline_method_fit_config':BASELINE_FIT_CONFIG,
             'acceleration_enabled':V4_ENABLE_EFFICIENTAD_ACCELERATION,
             'draem_safe_acceleration_enabled':V4_ENABLE_DRAEM_SAFE_ACCELERATION,
             'draem_texture_cache_mib':V4_DRAEM_DTD_CACHE_MIB,
             'optimized_bootstrap_enabled':V4_ENABLE_TIE_EXACT_FAST_BOOTSTRAP,
             'eval_release':V4_EVAL_RELEASE,
             'runtime_versions':v4_runtime_versions(),
             'pixel_parity_tolerance':V4_PIXEL_PARITY_ABS_TOL,
             'prediction_parity_tolerances':[V4_PREDICTION_PARITY_ATOL,V4_PREDICTION_PARITY_RTOL],
             'streaming_baseline_inference_enabled':V4_ENABLE_STREAMING_BASELINE_INFERENCE,
             'requested_inference_batch_size':V4_REQUESTED_INFERENCE_BATCH}
    if path.exists():
        prior=read_json(path)
        for k,v in setting.items():
            if prior.get(k)!=v:
                raise RuntimeError('Locked experiment changed: '+k+'; refusing test evaluation.')
        status('LOCK','Resuming with verified frozen code, models, splits and thresholds')
        return prior
    setting.update({'locked_at':time.strftime('%Y-%m-%dT%H:%M:%SZ',time.gmtime()),
                    'source_citation':SOURCE_CITATION,'licence_note':LICENSE_NOTE,
                    'prior_image_source_exposure':'Original photos reused; no old learned models or predictions',
                    'limitation':'Synthetic edits are not natural defect labels',
                    'baseline_amendment':('SuperSimpleNet is not original SimpleNet. '+PATCHCORE_AMENDMENT)})
    write_json(path,setting)
    status('LOCK','Protocol, source splits, checkpoints and thresholds frozen')
    return setting


def verify_cached_scores(expected_samples, score_file):
    if not score_file.exists():return False
    tab=pd.read_csv(score_file,usecols=['sample_id'])
    if len(tab)!=len(expected_samples) or set(tab.sample_id)!=set(expected_samples.sample_id):
        raise RuntimeError('Prediction CSV does not match frozen sample IDs: '+str(score_file))
    return True


def guard_gpu_aware_restart():
    """Do not race old GPU kernels or modify an already locked experiment."""
    lock=OUT/'frozen'/'protocol_lock.json'
    flag=OUT/'FINAL_TEST_USED.flag'
    if flag.exists() and not lock.exists():
        raise RuntimeError('Locked-test access marker exists but protocol lock is missing. '+
                           'Do not recreate a missing lock.')
    if lock.exists():
        previous=read_json(lock)
        if previous.get('code_fingerprint')!=code_fingerprint():
            raise RuntimeError('This notebook does not match the ALREADY LOCKED executable. '+
                               'Do not modify the lock or repeat held-out evaluation with changed code.')
        if previous.get('runtime_versions') and previous['runtime_versions']!=v4_runtime_versions():
            raise RuntimeError('The dependency versions differ from the already locked evaluation runtime')

        for key,filename in [('source_split_sha256',OUT/'data'/'source_split.csv'),
                             ('thresholds_sha256',OUT/'calibration'/'thresholds.json')]:
            if not filename.exists() or digest(filename)!=previous.get(key):
                raise RuntimeError('Previously locked '+key+' is missing or altered')
        for key,expected_hash in previous.get('checkpoints_sha256',{}).items():
            method,seed=key.rsplit('_',1)
            path=(OUT/'models'/method/seed)
            for filename in ['baseline.ckpt','model.pt','best.pt']:
                checkpoint=path/filename
                if checkpoint.exists():break
            else:
                raise RuntimeError('Missing prior locked checkpoint for '+key)
            if digest(checkpoint)!=expected_hash:
                raise RuntimeError('Prior locked checkpoint changed: '+key)
        print('LOCKED RESUME: immutable code/split/thresholds/checkpoints verified.',flush=True)
    try:
        import subprocess
        proc=subprocess.run(['nvidia-smi','--query-compute-apps=pid',
                             '--format=csv,noheader,nounits'],
                            capture_output=True,text=True,timeout=5)
        if proc.returncode==0:
            pids={int(x.strip()) for x in proc.stdout.splitlines() if x.strip().isdigit()}
            other=pids-{os.getpid()}
            if other:
                raise RuntimeError('Another GPU compute process is still active '
                                   f'(PID(s) {sorted(other)}). Stop the OLD V4 kernel '
                                   'after a saved epoch before starting GPU-aware V4. '
                                   'Never run two experiments against the same checkpoints.')
    except (OSError,subprocess.TimeoutExpired):

        pass


def run_v4():
    guard_gpu_aware_restart()

    if V4_ENABLE_TIE_EXACT_FAST_BOOTSTRAP:
        assert v4_test_fast_bootstrap(), 'Fast-bootstrap parity preflight failed'
        print('  PERFORMANCE PREFLIGHT PASS: weighted/tie AUROC agrees with sklearn',flush=True)
    assert v4_pixel_metric_preflight(), 'Pixel-metric numerical parity preflight failed'
    assert v4_permutation_reference_preflight(), 'Paired permutation parity preflight failed'
    t0=time.monotonic()
    for sub in ['data','development','calibration','frozen','results','reports','models']:
        (OUT/sub).mkdir(exist_ok=True,parents=True)
    if not torch.cuda.is_available():
        raise RuntimeError('Official 5-seed experiment requires a CUDA GPU.')
    device='cuda'
    done=OUT/'reports'/'FINAL_DECISION.json'
    if done.exists() and read_json(done).get('status')=='SCIENTIFIC_RUN_FINISHED':
        status('COMPLETE','Experiment already finished; no repeat of the locked test')
        display(pd.read_csv(OUT/'reports'/'publication_primary_table.csv'))
        return

    status('START','DatePalm V4 GPU-aware — existing CSV, masks, epochs and checkpoints reused')
    status('GPU',f'CUDA={torch.cuda.get_device_name(0)}; CPU quota={cpu_quota()}; data loading={V4_PREFETCH_MODE}; workers={V4_LOADER_WORKERS}; cached development GPU scoring={V4_GPU_VALIDATION}')
    df=source_and_split()
    status('P0-CACHE','Caching one fruit foreground mask per eligible original (reused in every epoch)')
    eligible=df[df.eligibility=='eligible']
    new_masks=0
    for r in tqdm(eligible.to_dict('records'),desc='Foreground masks',unit='image'):
        cached=OUT/'data'/'foreground_cache'/f"{r['source_id']}.png"
        if not cached.exists():
            _,_,mask_status=load_source(r,RAW)
            if mask_status!='ok':
                raise RuntimeError('Original photo foreground status changed: '+str(r['source_id']))
            new_masks+=1
    status('P0-CACHE',f'Foreground masks ready: {len(eligible)} total, {new_masks} newly cached')
    source_fingerprint=digest(OUT/'data'/'source_split.csv')
    signature_file=OUT/'DATASET_MANIFEST_SHA256.txt'
    if signature_file.exists() and signature_file.read_text().strip()!=source_fingerprint:
        raise RuntimeError('Source split CSV changed since this experiment was initialized')
    signature_file.write_text(source_fingerprint+'\n')

    status('P1','Training-only synthetic generator QA')
    run_training_qc(df,RAW,OUT/'data'/'train_generator_qc')
    status('P1','Generator quality audit finished / reused')

    status('BASELINE DATA','Checking auxiliary public datasets for published baselines')
    paths=download_assets(OUT/'auxiliary_baseline_data')
    requirements_check()
    extra={**paths,'shared_baseline_data':str(OUT/'data'/'baseline_shared')}
    config={'max_epochs':MAX_EPOCHS,'batch_size':BATCH_SIZE,'patience':10,
            'pretrained':PRETRAINED,'lambda_cons':0.10}
    config['lambda_cons']=select_consistency_weight(df,device,config)

    write_json(OUT/'data'/'baseline_protocol_amendment.json',
               {'library':'anomalib==2.2.0', 'patchcore':PATCHCORE_AMENDMENT,
                'candidate_patches_per_image':PATCHCORE_CANDIDATES_PER_IMAGE,
                'coreset_sampling_ratio':PATCHCORE_CORESET_RATIO,
                'set_before_locked_test':True,
                'method_fit_config':BASELINE_FIT_CONFIG,
                'scientific_math_equivalence_acceleration':{
                    'efficientad_logging_only_gpu_sync_reduction':V4_ENABLE_EFFICIENTAD_ACCELERATION,
                     'draem_logger_sync_reduction_and_exact_decoded_texture_cache':V4_ENABLE_DRAEM_SAFE_ACCELERATION,
                     'draem_cpu_texture_cache_mib':V4_DRAEM_DTD_CACHE_MIB,
                    'bootstrap_exact_weighted_rank_equivalent':V4_ENABLE_TIE_EXACT_FAST_BOOTSTRAP,
                    'bounded_streamed_baseline_inference':V4_ENABLE_STREAMING_BASELINE_INFERENCE,
                    'per_checkpoint_development_inference_parity_before_lock':True,
                    'cached_binary_masks_exact_bitpacking':True,
                    'pixel_metric_sklearn_reference_parity_preflight':True,
                    'no_training_hyperparameters_or_step_counts_changed':True,
                    'baseline_prediction_maps_discarded_when_localization_not_required':True},
                'full_state_resume_every_optimizer_steps':BASELINE_RECOVERY_INTERVAL_STEPS,
                'training_validation_frequency_note':'EfficientAD retains every-epoch validation to match existing completed seeds; DRAEM and SuperSimpleNet use every-fifth-epoch validation across all 5 seeds. This may alter validation-derived score normalization; disclose.',
                'budget_note':'EfficientAD 70,000 optimizer steps / 1 image per batch; DRAEM and SuperSimpleNet 60 epochs; patchcore and padim one-pass feature methods.'})
    ckpts={}
    jobs=[(v,s) for v in VARIANTS for s in SEEDS]
    status('P2','Training four matched models × five seeds')
    for k,(variant,seed) in enumerate(jobs,1):
        status('TRAIN',f'[{k}/{len(jobs)}] {variant} / seed {seed}')
        ckpts[(variant,seed)]=fit_matched(df,RAW,OUT/'models'/variant/str(seed),
                                          variant,seed,config,device)
    status('P2','Matched model training complete')

    status('P3','Anomalib baselines × five seeds (PatchCore uses a documented bounded candidate pool)')
    baseline_jobs=[(m,s) for m in BASELINES for s in SEEDS]
    for k,(name,seed) in enumerate(baseline_jobs,1):
        status('BASELINE',f'[{k}/{len(baseline_jobs)}] {name} / seed {seed}')
        ckpts[(name,seed)]=fit_baseline(df,RAW,OUT/'models'/name/str(seed),
                                        name,seed,device,extra)
    status('P3','All five baseline methods completed')

    status('P4','Creating/reusing development benchmark and scoring all checkpoints')
    dev=build_benchmark(df,RAW,OUT/'development'/'benchmark','development')
    v4_verify_benchmark_materialization(dev,'development')
    metrics=[]
    for k,((method,seed),checkpoint) in enumerate(ckpts.items(),1):
        outfile=OUT/'development'/'scores'/f'{method}_{seed}.csv'
        status('DEV',f'[{k}/{len(ckpts)}] {method} seed {seed}'+(' [cached]' if outfile.exists() else ''))
        frame=collect_and_score(checkpoint,method,dev,device,extra,outfile,False)
        metrics.append(dict(method=method,seed=seed,macro_auroc=macro(frame)))
    table=pd.DataFrame(metrics)
    table.to_csv(OUT/'development'/'method_comparison.csv',index=False)
    competitor=table[table.method.isin(BASELINES)].groupby('method').macro_auroc.mean().idxmax()
    comparator_path=OUT/'development'/'comparator_selected.json'
    if comparator_path.exists() and read_json(comparator_path)['competitor']!=competitor:
        raise RuntimeError('Development-selected baseline changed on a resumed run')
    write_json(comparator_path,{'competitor':competitor,'development_only':True})
    status('P4','Strongest registered published baseline: '+competitor)

    status('P5','Generating/reusing calibration negatives and fixed thresholds')
    calib=build_benchmark(df,RAW,OUT/'calibration'/'benchmark','calibration')
    v4_verify_benchmark_materialization(calib,'calibration')
    thresholds={}
    for k,((method,seed),checkpoint) in enumerate(ckpts.items(),1):
        outfile=OUT/'calibration'/'scores'/f'{method}_{seed}.csv'
        status('CAL',f'[{k}/{len(ckpts)}] {method} seed {seed}'+(' [cached]' if outfile.exists() else ''))
        frame=collect_and_score(checkpoint,method,calib,device,extra,outfile,False)
        thresholds[f'{method}_{seed}']=calibrate(frame)
    threshold_path=OUT/'calibration'/'thresholds.json'
    if threshold_path.exists() and read_json(threshold_path)!=thresholds:
        raise RuntimeError('Cached calibration thresholds changed unexpectedly')
    write_json(threshold_path,thresholds)
    if V4_ENABLE_STREAMING_BASELINE_INFERENCE:
        status('P5-PARITY','Persisting per-checkpoint development-only inference approvals before test lock')
        for (method,seed),checkpoint in ckpts.items():
            if method in BASELINES:
                _v4_verify_inference_parity(checkpoint,method,dev,device,extra)
    freeze_or_verify(ckpts,thresholds,competitor,config)

    marker=OUT/'FINAL_TEST_USED.flag'
    if not marker.exists():
        marker.write_text('First locked test access. Resume ONLY with identical frozen code/data/models. Do not reset this marker.\n')
    status('P6','Resuming/starting ONE FROZEN locked-test evaluation')
    test=build_benchmark(df,RAW,OUT/'results'/'fixed_benchmark','locked_test')
    v4_verify_benchmark_materialization(test,'locked_test')
    predictions={}
    for k,((method,seed),checkpoint) in enumerate(ckpts.items(),1):
        outfile=OUT/'results'/f'{method}_{seed}.csv'
        status('TEST',f'[{k}/{len(ckpts)}] {method} seed {seed}'+(' [cached]' if outfile.exists() else ''))
        predictions[(method,seed)]=collect_and_score(checkpoint,method,test,device,extra,outfile,True)

    status('P6-QA','Evaluating artifact-only negative controls')
    traps=build_artifact_traps(df,RAW,OUT/'results'/'artifact_traps')
    trap_rows=[]
    for k,((method,seed),checkpoint) in enumerate(ckpts.items(),1):
        outfile=OUT/'results'/'artifact_trap_scores'/f'{method}_{seed}.csv'
        status('TRAP',f'[{k}/{len(ckpts)}] {method} seed {seed}'+(' [cached]' if outfile.exists() else ''))
        frame=collect_and_score(checkpoint,method,traps,device,extra,outfile,False)
        for kind,g in frame.groupby('kind'):
            trap_rows.append({'method':method,'seed':seed,'trap_type':kind,
                              'n_source_groups':int(g.source_group.nunique()),
                              'fpr_at_calibration_threshold':float((g.score>thresholds[f'{method}_{seed}']).mean())})
    pd.DataFrame(trap_rows).to_csv(OUT/'reports'/'artifact_trap_fpr.csv',index=False)

    status('P7','Statistics, paired source-group bootstrap, figures and final report')
    final_report(OUT,predictions,thresholds,competitor,False)
    status('FINISHED',f'All mandatory experiment stages complete in {(time.monotonic()-t0)/3600:.2f} hours')
    print('Final results:',OUT/'reports'/'publication_primary_table.csv',flush=True)
    display(pd.read_csv(OUT/'reports'/'publication_primary_table.csv'))




In [ ]:
RUN_PHASE='AUDIT'
AUDIT_STAGED_SHA256=False
APPROVE_DRAEM_PROMOTION=False
ALLOW_MISSING_BASELINE_TRAINING=False
APPROVE_PRELOCK_EVALUATION=False
APPROVE_PRELOCK_COMPARATOR_CORRECTION=False
APPROVE_CREATE_PROTOCOL_LOCK=False
APPROVE_LOCKED_TEST=False
LOCKED_TEST_AUTHORIZATION_PHRASE=''
APPROVE_POST_TEST_ANALYSIS=False


In [ ]:



from pathlib import Path
from datetime import datetime, timezone
import json, os, shutil, hashlib, tempfile, time, contextlib
import numpy as np
import pandas as pd

V4_SEEDS = (1049, 2053, 3067, 4093, 5051)
V4_EXPECTED_BASELINES = ('Patchcore','Padim','EfficientAd','Draem','Supersimplenet')
V4_EXPECTED_METHODS = tuple(['C00','C10','C01','C11'] + list(V4_EXPECTED_BASELINES))
V4_HUE_PROTOCOL = 'datepalm-v4-draem-hue-input-range-v1'
V4_HUE_SOURCE_HASH = 'e3a73b5759cdd374c829964911f54af0accf89a0d9d440579d0a4adb638214bd'
V4_STAGE_ROOT = OUT/'diagnostics'/'draem_hue_guard_v1_stage'
V4_CONT_ROOT = OUT/'diagnostics'/'v4_controlled_finalization_20261017'
V4_PRESERVE = V4_CONT_ROOT/'preserved_prelock_evidence'
V4_PROMOTION_LEDGER = V4_CONT_ROOT/'promotion_ledger.json'
V4_STAGE_NAMES = ('AUDIT','PROMOTE','PRELOCK','LOCK','TEST','POST_TEST','ALL_PRELOCK')



assert tuple(SEEDS) == V4_SEEDS, 'Original experimental seeds unexpectedly differ'
assert tuple(BASELINES) == V4_EXPECTED_BASELINES, 'Baseline names differ from registered V4 code'
assert tuple(VARIANTS) == ('C00','C10','C01','C11'), 'Controlled treatment groups changed'
assert V4_EVAL_RELEASE == '2026-10-15-postprocessed-streaming-parity-v2', 'Evaluation release differs from audited source'


def v4_check_no_test_started():
    lock = OUT/'frozen'/'protocol_lock.json'
    marker = OUT/'FINAL_TEST_USED.flag'
    test_csvs = list((OUT/'results').glob('*.csv')) if (OUT/'results').exists() else []
    if lock.exists() or marker.exists() or test_csvs or (OUT/'reports'/'FINAL_DECISION.json').exists():
        raise RuntimeError('A lock, test marker, or test score file already exists. Do not use pre-lock operations. '
                           'Assess the original locked executable before continuing. '
                           f'lock={lock.exists()}, marker={marker.exists()}, test_CSVs={len(test_csvs)}')


def v4_read_json_strict(path):
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(str(path))
    doc = json.loads(path.read_text(encoding='utf8'))
    if not isinstance(doc, dict):
        raise RuntimeError('Expected JSON object: '+str(path))
    return doc


def v4_checkpoint(name, seed):
    base = OUT/'models'/str(name)/str(seed)
    return base/('model.pt' if name.startswith('C') else 'baseline.ckpt')


def v4_valid_model(name, seed, hash_check=False):
    base = OUT/'models'/str(name)/str(seed)
    ck = v4_checkpoint(name, seed)
    marker = base/'COMPLETE.json'
    if not ck.is_file() or not marker.is_file() or ck.stat().st_size < 1024:
        return False, 'Missing checkpoint or COMPLETE.json'
    try:
        doc = v4_read_json_strict(marker)
    except Exception as e:
        return False, f'Unreadable completion manifest: {e}'
    if hash_check and not name.startswith('C') and doc.get('model_checkpoint_sha256') and doc['model_checkpoint_sha256'] != digest(ck):
        return False, 'Baseline checkpoint SHA256 disagrees with completion record'
    if name == 'Draem' and doc.get('protocol_amendment') != V4_HUE_PROTOCOL:
        return False, 'Legacy DRAEM has not been promoted'
    if hash_check and name == 'Draem':
        claimed = doc.get('model_checkpoint_sha256')
        if claimed != digest(ck):
            return False, 'Promoted DRAEM checkpoint hash differs from manifest'
    return True, 'Checkpoint and completion record exist'+(' (hash-checked)' if hash_check else '')


def v4_audit_draem_staged(seed, verify_tensor_metadata=True):
    """Read-only independent qualification; no model/optimizer weights allocated on GPU."""
    folder = V4_STAGE_ROOT/str(seed)
    ck = folder/'baseline.ckpt'
    record = v4_read_json_strict(folder/'COMPLETE.json')
    h = v4_read_json_strict(folder/'hue_guard_protocol.json')
    if not ck.is_file(): raise RuntimeError(f'DRAEM seed {seed}: staged checkpoint absent')
    if ck.stat().st_size < 500*1024**2:
        raise RuntimeError(f'DRAEM seed {seed}: checkpoint unexpectedly small')
    fit = record.get('fit_spec') or {}
    if record.get('seed') != seed or fit.get('max_epochs') != 60 or fit.get('train_batch_size') != 8 or fit.get('eval_batch_size') != 8:
        raise RuntimeError(f'DRAEM seed {seed}: training budget/identity differs')
    if record.get('protocol_amendment') != V4_HUE_PROTOCOL or record.get('source_snapshot_sha256') != V4_HUE_SOURCE_HASH:
        raise RuntimeError(f'DRAEM seed {seed}: repair protocol/source hash differs')
    if int(record.get('optimizer_float_tensors_checked') or 0) < 100:
        raise RuntimeError(f'DRAEM seed {seed}: optimizer finite-tensor audit missing')
    usage = record.get('hue_guard_usage') or {}
    if int(usage.get('hue_calls') or 0) < 1 or int(usage.get('corrected_calls') or 0) < 1:
        raise RuntimeError(f'DRAEM seed {seed}: hue audit missing or no recorded correction')
    claim = record.get('model_checkpoint_sha256')
    if not isinstance(claim,str) or len(claim) != 64 or h.get('checkpoint_sha256') != claim:
        raise RuntimeError(f'DRAEM seed {seed}: manifest checkpoint SHA256 disagreement')
    actual = digest(ck)
    if actual != claim:
        raise RuntimeError(f'DRAEM seed {seed}: ACTUAL checkpoint SHA256 mismatch')
    if verify_tensor_metadata:
        try:
            from torch._subclasses.fake_tensor import FakeTensorMode
            with FakeTensorMode():
                meta = torch.load(ck, weights_only=True, map_location='cpu', mmap=True)
            loops = meta.get('loops', {}).get('fit_loop', {})
            epochs = loops.get('epoch_progress',{}).get('total',{}).get('completed')

            if (int(meta.get('epoch',-1)) != 60 or epochs != 60 or
                int(meta.get('global_step',-1)) != 27060 or
                len(meta.get('state_dict',{})) < 300 or
                not meta.get('optimizer_states')):
                raise RuntimeError('Training-loop/model/optimizer metadata incomplete')
            del meta
        except Exception as ex:
            raise RuntimeError(f'DRAEM seed {seed}: metadata validation failed: {ex}') from ex
    return dict(seed=seed, checkpoint=str(ck), sha256=actual,
                bytes=ck.stat().st_size, epoch=60, global_step=27060,
                protocol_amendment=V4_HUE_PROTOCOL, source_snapshot_sha256=V4_HUE_SOURCE_HASH,
                hue_calls=int(usage['hue_calls']), corrected_calls=int(usage['corrected_calls']))


def v4_audit_inventory(full_staged_hashes=False):
    print('\n=== CANONICAL V4 MODEL INVENTORY ===')
    missing=[]
    for name in V4_EXPECTED_METHODS:
        for seed in V4_SEEDS:
            if name=='Draem':
                official_ok, note=v4_valid_model(name,seed)
                staged = (V4_STAGE_ROOT/str(seed)/'COMPLETE.json').exists()
                label=('PROMOTED' if official_ok else 'STAGED' if staged else 'MISSING')
                ok=official_ok or staged
                print(f'{name:15} {seed}: {label:10} {note}')
            else:
                ok,note=v4_valid_model(name,seed)
                print(f'{name:15} {seed}: {"COMPLETE" if ok else "MISSING":10} {note}')
            if not ok: missing.append((name,seed))
    print('\nCanonical missing/unrecognized jobs:',missing or 'none')
    for sub in ('development','calibration'):
        folder=OUT/sub/'scores'
        misses=[f'{name}_{seed}' for name in V4_EXPECTED_METHODS for seed in V4_SEEDS
                if not (folder/f'{name}_{seed}.csv').is_file()]
        print(sub,'existing:',45-len(misses),'/45; missing:',misses or 'none')
    for rel in ('frozen/protocol_lock.json','FINAL_TEST_USED.flag'):
        print('TEST GUARD:',rel, (OUT/rel).exists())
    print('Existing result CSVs:',len(list((OUT/'results').glob('*.csv'))) if (OUT/'results').exists() else 0)
    print('GPU present:',torch.cuda.is_available(),'; CPU quota:',cpu_quota(),'; GPU:',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')
    if full_staged_hashes:
        print('\nINDEPENDENT STAGED DRAEM AUDIT (each SHA256 reads ~1.1 GiB):')
        for seed in V4_SEEDS:
            v4_audit_draem_staged(seed)
            print('PASS staged DRAEM seed',seed)
    return missing


def v4_prepare_runtime():
    """Reuse original code's validated preflight, rather than changing training/model math."""
    guard_gpu_aware_restart()
    if not torch.cuda.is_available():raise RuntimeError('CUDA is required for official evaluation.')
    if V4_ENABLE_TIE_EXACT_FAST_BOOTSTRAP and not v4_test_fast_bootstrap():
        raise RuntimeError('Fast statistical kernel parity preflight failed')
    if not v4_pixel_metric_preflight() or not v4_permutation_reference_preflight():
        raise RuntimeError('Pixel/permutation numerical parity preflight failed')
    requirements_check()
    status('GPU',f'GPU={torch.cuda.get_device_name(0)}; CPU quota={cpu_quota()}; '
                 f'parity-gated streaming inference={V4_ENABLE_STREAMING_BASELINE_INFERENCE}; '
                 f'batch attempt={V4_REQUESTED_INFERENCE_BATCH}')

    extra = download_assets(OUT/'auxiliary_baseline_data')
    extra.update({'shared_baseline_data':str(OUT/'data'/'baseline_shared')})
    df = source_and_split()
    if not (OUT/'development'/'lambda_selected.json').is_file():
        raise RuntimeError('Missing development-only lambda selection; do not silently retune a completed 20-model experiment.')
    config={'max_epochs':MAX_EPOCHS,'batch_size':BATCH_SIZE,'patience':10,
            'pretrained':PRETRAINED,'lambda_cons':float(v4_read_json_strict(OUT/'development'/'lambda_selected.json')['lambda_cons'])}
    return df, 'cuda', extra, config


def v4_collect_ckpts(permit_missing_non_draem=False, df=None, device=None, extra=None):
    ckpts={}
    for name in V4_EXPECTED_METHODS:
        for seed in V4_SEEDS:
            if name == 'Draem':
                ok, why=v4_valid_model(name,seed,hash_check=True)
                if not ok:raise RuntimeError(f'Seed {seed}: corrected DRAEM not promoted ({why}); do PROMOTE first.')
            else:
                ok,why=v4_valid_model(name,seed,hash_check=True)
                if not ok:
                    if not (permit_missing_non_draem and ALLOW_MISSING_BASELINE_TRAINING
                            and name in V4_EXPECTED_BASELINES and df is not None):
                        raise RuntimeError(f'Missing {name}/{seed}: {why}; do not invent a trained model')
                    base=OUT/'models'/name/str(seed)
                    if (base/'engine').exists() or (base/'in_progress_full_state.ckpt').exists():
                        raise RuntimeError(f'Incomplete training evidence exists for {name}/{seed}; '
                                           'requires method-specific audited recovery, not blind overwrite')
                    status('P3',f'Explicitly approved original training of missing {name} seed {seed}')
                    fit_baseline(df,RAW,base,name,seed,device,extra)
                    ok,why=v4_valid_model(name,seed)
                    if not ok:raise RuntimeError(f'Training did not create complete {name}/{seed}: {why}')
            ckpts[(name,seed)] = v4_checkpoint(name,seed)
    assert len(ckpts)==45
    return ckpts


def v4_backup_file(src, area, scope):
    """Copy historical evidence to immutable content-addressed storage before a change."""
    src=Path(src)
    if not src.is_file():return None
    fingerprint=digest(src)
    target=V4_PRESERVE/area/scope/(src.name+'.'+fingerprint[:20]+'.preserved')
    if target.is_file():
        if digest(target)!=fingerprint:raise RuntimeError('Historical backup content changed: '+str(target))
        return target
    target.parent.mkdir(parents=True,exist_ok=True)
    tmp=target.with_name(target.name+'.incomplete')
    if tmp.exists():raise RuntimeError('Previous unfinished backup requires inspection: '+str(tmp))
    with src.open('rb') as inp, tmp.open('xb') as dst:
        shutil.copyfileobj(inp,dst,length=8*1024*1024)
        dst.flush();os.fsync(dst.fileno())
    if digest(tmp)!=fingerprint:
        raise RuntimeError('Copy verification failed before replace: '+str(src))
    os.replace(tmp,target)
    return target


def v4_promote_draem():
    v4_check_no_test_started()
    if not APPROVE_DRAEM_PROMOTION:
        raise RuntimeError('Promotion is locked. Review audit and explicitly set APPROVE_DRAEM_PROMOTION=True.')
    all_rows=[v4_audit_draem_staged(seed) for seed in V4_SEEDS]
    V4_CONT_ROOT.mkdir(parents=True,exist_ok=True)
    for row in all_rows:
        seed=row['seed']
        source=Path(row['checkpoint'])
        official=OUT/'models'/'Draem'/str(seed)
        official.mkdir(parents=True,exist_ok=True)
        final=official/'baseline.ckpt'
        marker=official/'COMPLETE.json'
        protocol=official/'hue_guard_protocol.json'
        existing = (digest(final) if final.is_file() else None)
        if existing != row['sha256']:

            for p in [final, marker, protocol]:
                if p.exists():v4_backup_file(p,'legacy_draem',f'seed_{seed}')
            free=shutil.disk_usage(official).free
            if free < source.stat().st_size + 512*1024**2:
                raise RuntimeError(f'Insufficient disk capacity to promote seed {seed}')
            partial=official/'baseline.ckpt.PROMOTING'
            if partial.exists():raise RuntimeError('Unfinished promotion copy found: '+str(partial))
            with source.open('rb') as i, partial.open('xb') as o:
                shutil.copyfileobj(i,o,length=8*1024*1024)
                o.flush();os.fsync(o.fileno())
            if digest(partial) != row['sha256']:
                raise RuntimeError(f'Official DRAEM seed {seed}: copy integrity failure')
            os.replace(partial,final)

        staged_record = v4_read_json_strict(V4_STAGE_ROOT/str(seed)/'COMPLETE.json')
        staged_hue = v4_read_json_strict(V4_STAGE_ROOT/str(seed)/'hue_guard_protocol.json')
        for target,payload in [(protocol,staged_hue),(marker,staged_record)]:
            temp=target.with_name(target.name+'.PROMOTING')
            if temp.exists():raise RuntimeError('Leftover manifest promotion temp: '+str(temp))
            with temp.open('x',encoding='utf8') as f:
                json.dump(payload,f,indent=2,sort_keys=True)
                f.flush();os.fsync(f.fileno())
            os.replace(temp,target)
        passed,msg=v4_valid_model('Draem',seed,hash_check=True)
        if not passed:raise RuntimeError(f'Official seed {seed} failed independent post-promotion verification: {msg}')
        status('PROMOTION',f'DRAEM {seed} independently accepted and hash-verified in official model directory')
    ledger = dict(kind='datepalm_v4_draem_hue_guard_independent_promotion_v1',
                  accepted_at=datetime.now(timezone.utc).isoformat(),
                  evidence=all_rows,original_baseline_preservation=str(V4_PRESERVE/'legacy_draem'),
                  monitored_model_dir=str(OUT/'models'/'Draem'),
                  acceptance_limits=['SHA256, Lightning metadata and documented finite/hue audit verified',
                                     'Full numerical reproduction on GPU was not rerun'])
    if V4_PROMOTION_LEDGER.exists():
        prev=v4_read_json_strict(V4_PROMOTION_LEDGER)
        if prev.get('evidence')!=all_rows:raise RuntimeError('Existing promotion ledger conflicts with this run')
        print('Promotion ledger already matches; original acceptance time preserved')
    else:
        write_json(V4_PROMOTION_LEDGER,ledger)
    print('PROMOTION VERIFIED: 5/5 official corrected DRAEM checkpoints. Monitor promotion counter should read 5/5.')


def v4_archive_prelok_score(output, reason):
    """Never discard historical unverified scores; conserve them for audit."""
    output=Path(output)
    if not output.is_file():return
    meta=output.with_suffix('.csv.provenance.json')
    for src in (output,meta):
        if src.is_file():
            v4_backup_file(src,'score_versions',output.parent.name+'_'+output.stem)

    v4_check_no_test_started()
    for src in (output,meta):
        if src.exists(): src.unlink()
    status('CACHE',f'Archived pre-lock score {output.name}: {reason}')


def v4_prelock_score(checkpoint,name,samples,device,extra,outfile,phase):

    outfile=Path(outfile)
    metadata=outfile.with_suffix('.csv.provenance.json')
    if metadata.is_file() and not outfile.is_file():
        v4_check_no_test_started()
        v4_backup_file(metadata,'orphan_prelock_metadata',outfile.parent.name+'_'+outfile.stem)
        metadata.unlink()
        status('CACHE',f'Archived orphan pre-lock metadata before rebuilding {outfile.name}')
    if outfile.is_file():
        reason=None
        if name=='Draem':

            if not metadata.is_file():
                reason='Old DRAEM score lacks source checkpoint provenance'
            else:
                rec=v4_read_json_strict(metadata)
                if rec.get('checkpoint_sha256') != digest(checkpoint):
                    reason='Legacy DRAEM checkpoint differs from corrected model'
        if reason is None and not metadata.is_file():
            reason='Legacy score has no reliable provenance; choose exact regeneration'
        if reason is None:
            rec=v4_read_json_strict(metadata)
            if rec.get('cache_origin')=='legacy_spot_checked_unverifiable_full_provenance':
                reason='Legacy seven-image spot-check is not full provenance'
            elif name.startswith('C') and rec.get('release')=='2026-10-15-audited-p4-p7-v1':
                reason=None
            else:
                contract=v4_score_contract(checkpoint,samples,outfile,False)
                if any(rec.get(k)!=v for k,v in contract.items()):
                    reason='Stored scientific evaluator/benchmark/checkpoint contract changed'
                elif rec.get('score_csv_sha256')!=digest(outfile):
                    raise RuntimeError('Existing score CSV checksum does not match provenance; audit manually: '+str(outfile))
        if reason:
            v4_archive_prelok_score(outfile,reason)
    return collect_and_score(checkpoint,name,samples,device,extra,outfile,False)


def v4_dev_stage(df,device,extra,ckpts):
    v4_check_no_test_started()
    status('P4','Generating/verifying training-mechanism development benchmark only')
    dev=build_benchmark(df,RAW,OUT/'development'/'benchmark','development')
    v4_verify_benchmark_materialization(dev,'development')
    rows=[]
    for j,((method,seed),ckpt) in enumerate(ckpts.items(),1):
        outfile=OUT/'development'/'scores'/f'{method}_{seed}.csv'
        status('DEV',f'[{j}/45] development score {method} / seed {seed}')
        frame=v4_prelock_score(ckpt,method,dev,device,extra,outfile,'P4')
        rows.append(dict(method=method,seed=seed,macro_auroc=macro(frame)))
    table=pd.DataFrame(rows)
    if len(table)!=45:raise RuntimeError('P4 requires all 45 method/seed scores')
    choice=table[table.method.isin(BASELINES)].groupby('method').macro_auroc.mean().idxmax()
    path=OUT/'development'/'comparator_selected.json'
    if path.is_file():
        previous=v4_read_json_strict(path)
        if previous.get('competitor') != choice:
            if not APPROVE_PRELOCK_COMPARATOR_CORRECTION:
                raise RuntimeError('Corrected DRAEM development scores change the previously selected baseline '
                                   f'{previous.get("competitor")!r} -> {choice!r}. '
                                   'This requires explicit, documented PRE-LOCK selection amendment approval; '
                                   'set APPROVE_PRELOCK_COMPARATOR_CORRECTION=True only after reviewing the development-only evidence.')
            v4_backup_file(path,'old_comparator','p4_reselection')
    other=OUT/'development'/'method_comparison.csv'
    if other.is_file():v4_backup_file(other,'old_comparison','p4_reselection')
    temp=other.with_suffix('.csv.partial')
    table.to_csv(temp,index=False);os.replace(temp,other)
    write_json(path,{'competitor':str(choice),'development_only':True,
                     'corrected_draem_amendment':V4_HUE_PROTOCOL,
                     'source':'45 verified corrected-checkpoint development score CSVs',
                     'method_comparison_sha256':digest(other)})
    status('P4',f'Complete 45/45 development scores; PRE-LOCK competitor={choice}')
    return dev,str(choice)


def v4_calibration_stage(df,dev,device,extra,ckpts):
    v4_check_no_test_started()

    if not (OUT/'development'/'comparator_selected.json').is_file():
        raise RuntimeError('P4 comparator missing')
    calib=build_benchmark(df,RAW,OUT/'calibration'/'benchmark','calibration')
    v4_verify_benchmark_materialization(calib,'calibration')
    thresholds={}
    status('P5','Begin 45 source-balanced calibration score jobs')
    for j,((method,seed),ckpt) in enumerate(ckpts.items(),1):
        status('CAL',f'[{j}/45] calibration {method}/{seed}')
        path=OUT/'calibration'/'scores'/f'{method}_{seed}.csv'
        frame=v4_prelock_score(ckpt,method,calib,device,extra,path,'P5')
        thresholds[f'{method}_{seed}']=calibrate(frame)
    threshold_file=OUT/'calibration'/'thresholds.json'
    if threshold_file.exists():
        prior=v4_read_json_strict(threshold_file)
        if prior!=thresholds:
            v4_backup_file(threshold_file,'old_thresholds','calibration')
    write_json(threshold_file,thresholds)
    status('P5-PARITY','Verifying baseline inference parity ON DEVELOPMENT ONLY, before lock')
    if V4_ENABLE_STREAMING_BASELINE_INFERENCE:
        for j,((method,seed),ckpt) in enumerate(ckpts.items(),1):
            if method in BASELINES:
                status('PARITY',f'[{j}/45] Anomalib original vs streamed post-processed outputs {method}/{seed}')
                _v4_verify_inference_parity(ckpt,method,dev,device,extra)
    status('P5','All 45 thresholds fixed with original quantile method; parity approvals complete')
    return thresholds


def v4_prepare_for_lock(df,device,extra,config):
    ckpts=v4_collect_ckpts()
    comparator=v4_read_json_strict(OUT/'development'/'comparator_selected.json')['competitor']
    thresholds=v4_read_json_strict(OUT/'calibration'/'thresholds.json')
    if comparator not in BASELINES:raise RuntimeError('Chosen comparator is not one of five registered baselines')
    if set(thresholds)!={f'{n}_{s}' for n in V4_EXPECTED_METHODS for s in V4_SEEDS}:
        raise RuntimeError('P5 thresholds incomplete; expected 45 values')
    if not V4_PROMOTION_LEDGER.is_file():raise RuntimeError('Official five-seed promotion ledger not found')
    for seed in V4_SEEDS:v4_audit_draem_staged(seed, verify_tensor_metadata=False)
    for phase in ('development','calibration'):
        scores=OUT/phase/'scores'
        files=list(scores.glob('*.csv'))
        if len([p for p in files if p.stem in {f'{m}_{s}' for m in V4_EXPECTED_METHODS for s in V4_SEEDS}])!=45:
            raise RuntimeError(phase+' score files incomplete')
        for (method,seed), ckpt in ckpts.items():
            csv=scores/f'{method}_{seed}.csv'
            meta=csv.with_suffix('.csv.provenance.json')
            if not meta.is_file():raise RuntimeError('Provenance absent for pre-lock score: '+str(csv))
            record=v4_read_json_strict(meta)
            if record.get('checkpoint_sha256')!=digest(ckpt) or record.get('score_csv_sha256')!=digest(csv):
                raise RuntimeError('Score provenance mismatch before lock: '+str(csv))
            if record.get('cache_origin')=='legacy_spot_checked_unverifiable_full_provenance':
                raise RuntimeError('Unverifiable old cache not acceptable before final lock: '+str(csv))


    if V4_ENABLE_STREAMING_BASELINE_INFERENCE:
        for (method,seed), ckpt in ckpts.items():
            if method not in BASELINES:continue
            approval=OUT/'data'/'inference_parity'/f'{method}_{digest(ckpt)[:24]}.json'
            record=v4_read_json_strict(approval)
            checks=record.get('comparisons') or []
            if (record.get('checkpoint_sha256')!=digest(ckpt) or
                record.get('method')!=method or record.get('source_stage')!='development' or
                record.get('capture_stage')!='after_anomalib_postprocessing_v2' or
                record.get('requested_batch_size')!=V4_REQUESTED_INFERENCE_BATCH or
                record.get('tolerance_atol')!=V4_PREDICTION_PARITY_ATOL or
                record.get('tolerance_rtol')!=V4_PREDICTION_PARITY_RTOL or
                not any(c.get('batch_size')==1 and c.get('passed') is True for c in checks)):
                raise RuntimeError('Baseline inference parity proof absent or incomplete: '+str(approval))
    status('LOCK_PREFLIGHT','Validated five promoted DRAEM checkpoints, 45 P4 and 45 P5 score provenance records')
    return ckpts,thresholds,comparator



def v4_controller_fingerprint():


    names = ['v4_audit_draem_staged','v4_valid_model','v4_collect_ckpts',
             'v4_backup_file','v4_promote_draem','v4_prelock_score',
             'v4_dev_stage','v4_calibration_stage','v4_prepare_for_lock',
             'v4_create_lock','v4_assert_locked_ready','v4_test_stage',
             'v4_post_test_stage','v4_run_controlled']
    body={name:_stable_code(globals()[name].__code__) for name in names}
    return hashlib.sha256(json.dumps(body,sort_keys=True,default=str).encode()).hexdigest()


def v4_prelock_output_hashes():
    return {f'{part}/{name}_{seed}':digest(OUT/part/'scores'/f'{name}_{seed}.csv')
            for part in ('development','calibration')
            for name in V4_EXPECTED_METHODS for seed in V4_SEEDS}

def v4_create_lock():
    if not APPROVE_CREATE_PROTOCOL_LOCK:
        raise RuntimeError('Final protocol lock requires APPROVE_CREATE_PROTOCOL_LOCK=True')
    if (OUT/'FINAL_TEST_USED.flag').exists() or list((OUT/'results').glob('*.csv')):
        raise RuntimeError('Test already opened; cannot recreate the pre-test controller lock')


    df,device,extra,config=v4_prepare_runtime()
    ckpts,thresholds,comparator=v4_prepare_for_lock(df,device,extra,config)


    lock=freeze_or_verify(ckpts,thresholds,comparator,config)
    controller={
        'controller_release':'datepalm-v4-controlled-finalization-2026-10-17-v1',
        'source_scientific_release':V4_EVAL_RELEASE,
        'original_code_fingerprint':code_fingerprint(),
        'controller_code_sha256':v4_controller_fingerprint(),
        'prelock_score_sha256':v4_prelock_output_hashes(),
        'original_protocol_lock_sha256':digest(OUT/'frozen'/'protocol_lock.json'),
        'promotion_ledger_sha256':digest(V4_PROMOTION_LEDGER),
        'prelock_comparator':comparator,
        'note':'Do not change this notebook, the test lock, or the controller release after final test access.'
    }
    ctrl_file=OUT/'frozen'/'controller_lock.json'
    if ctrl_file.exists():
        if v4_read_json_strict(ctrl_file)!=controller:
            raise RuntimeError('Existing controller lock differs; refusing overwrite')
    else:
        write_json(ctrl_file,controller)
    status('LOCK','Experiment locked. Held-out test remains unopened until separately approved.')
    print('LOCK CREATED. Run mode TEST only after explicit independent review.')
    return lock


def v4_assert_locked_ready():
    path=OUT/'frozen'/'protocol_lock.json'
    second=OUT/'frozen'/'controller_lock.json'
    if not path.exists() or not second.exists():
        raise RuntimeError('Original and controller locks both required for test access.')
    guard_gpu_aware_restart()
    ctrl=v4_read_json_strict(second)
    if ctrl.get('controller_release')!='datepalm-v4-controlled-finalization-2026-10-17-v1':
        raise RuntimeError('Controller identity changed after lock')
    if ctrl.get('original_protocol_lock_sha256')!=digest(path):
        raise RuntimeError('Original scientific lock differs from independently recorded controller lock')
    if ctrl.get('controller_code_sha256')!=v4_controller_fingerprint():
        raise RuntimeError('Controller instructions changed after test lock')
    if ctrl.get('prelock_score_sha256')!=v4_prelock_output_hashes():
        raise RuntimeError('Original P4/P5 pre-lock score bytes changed after test lock')
    if ctrl.get('promotion_ledger_sha256')!=digest(V4_PROMOTION_LEDGER):
        raise RuntimeError('Promotion ledger changed after test lock')
    lock=v4_read_json_strict(path)
    if lock.get('code_fingerprint')!=code_fingerprint():
        raise RuntimeError('Executable scientific function fingerprint changed after lock')
    if lock.get('runtime_versions') != v4_runtime_versions():
        raise RuntimeError('Pinned evaluation dependency environment changed')
    ckpts=v4_collect_ckpts()
    for (name,seed), p in ckpts.items():
        if lock.get('checkpoints_sha256',{}).get(f'{name}_{seed}')!=digest(p):
            raise RuntimeError('Locked model fingerprint mismatch: '+name+'/'+str(seed))
    if lock.get('source_split_sha256')!=digest(OUT/'data'/'source_split.csv'):
        raise RuntimeError('Source split altered after lock')
    if lock.get('thresholds_sha256')!=digest(OUT/'calibration'/'thresholds.json'):
        raise RuntimeError('Calibrated threshold hash changed after lock')
    return ckpts, lock


def v4_test_stage():
    if not APPROVE_LOCKED_TEST or LOCKED_TEST_AUTHORIZATION_PHRASE!='I AUTHORIZE THE ONE LOCKED TEST':
        raise RuntimeError('ONE-TIME HELD-OUT TEST LOCKED. Set APPROVE_LOCKED_TEST=True and enter exact authorization phrase.')
    ckpts,lock=v4_assert_locked_ready()
    if (OUT/'reports'/'FINAL_DECISION.json').is_file():
        raise RuntimeError('Final scientific decision already exists; refuse a second test run')
    df=source_and_split()
    device='cuda'
    extra=download_assets(OUT/'auxiliary_baseline_data')
    extra['shared_baseline_data']=str(OUT/'data'/'baseline_shared')
    marker=OUT/'FINAL_TEST_USED.flag'
    if not marker.exists():

        marker.write_text('One official locked-test access. Do not remove. '
                          'Only identical-lock interrupted-job continuation is allowed.\n')
    status('P6','Running exactly one locked-test benchmark; identical-lock resume only')
    test=build_benchmark(df,RAW,OUT/'results'/'fixed_benchmark','locked_test')
    v4_verify_benchmark_materialization(test,'locked_test')
    for idx,((name,seed),ckpt) in enumerate(ckpts.items(),1):
        status('TEST',f'[{idx}/45] locked test {name}/{seed}')
        path=OUT/'results'/f'{name}_{seed}.csv'
        collect_and_score(ckpt,name,test,device,extra,path,True)
    status('P6','All 45 locked-test scores and edit-support metrics complete; marker and lock preserved')


def v4_post_test_stage():
    if not APPROVE_POST_TEST_ANALYSIS:
        raise RuntimeError('Post-test QA and statistics require APPROVE_POST_TEST_ANALYSIS=True')
    ckpts,lock=v4_assert_locked_ready()
    if not (OUT/'FINAL_TEST_USED.flag').is_file():
        raise RuntimeError('Test marker absent: never run post-test pipeline before test')
    device='cuda'
    extra=download_assets(OUT/'auxiliary_baseline_data')
    extra['shared_baseline_data']=str(OUT/'data'/'baseline_shared')
    test=OUT/'results'/'fixed_benchmark'/'benchmark.csv'

    if not test.is_file():

        status('POST_TEST','Reusing original build_benchmark on already-locked same dataset')
    df=source_and_split()
    test_samples=build_benchmark(df,RAW,OUT/'results'/'fixed_benchmark','locked_test')
    v4_verify_benchmark_materialization(test_samples,'locked_test')
    predictions={}
    for j,((name,seed),ckpt) in enumerate(ckpts.items(),1):
        output=OUT/'results'/f'{name}_{seed}.csv'
        provenance=output.with_suffix('.csv.provenance.json')
        if not output.is_file() or not provenance.is_file():
            raise RuntimeError('Locked test incomplete, cannot analyze or QA: '+str(output))
        expected=v4_score_contract(ckpt,test_samples,output,True)
        rec=v4_read_json_strict(provenance)
        if any(rec.get(k)!=v for k,v in expected.items()) or rec.get('score_csv_sha256')!=digest(output):
            raise RuntimeError('Locked test provenance differs from lock: '+str(output))
        frame=pd.read_csv(output,dtype={'sample_id':str,'source_id':str,'source_group':str})
        v4_verify_score_table(frame,test_samples,True)
        predictions[(name,seed)]=frame
    thresholds=v4_read_json_strict(OUT/'calibration'/'thresholds.json')
    comparator=v4_read_json_strict(OUT/'development'/'comparator_selected.json')['competitor']
    if comparator != lock.get('comparator'):
        raise RuntimeError('Postlock competitor selection has changed')

    status('P6-QA','Artifact-only shortcut control scoring (all 45 methods/seeds)')
    traps=build_artifact_traps(df,RAW,OUT/'results'/'artifact_traps')
    trap_rows=[]
    for j,((name,seed),ckpt) in enumerate(ckpts.items(),1):
        status('TRAP',f'[{j}/45] artifact-only control {name}/{seed}')
        output=OUT/'results'/'artifact_trap_scores'/f'{name}_{seed}.csv'
        frame=collect_and_score(ckpt,name,traps,device,extra,output,False)
        for kind,group in frame.groupby('kind'):
            trap_rows.append(dict(method=name,seed=seed,trap_type=kind,
                                  n_source_groups=int(group.source_group.nunique()),
                                  fpr_at_calibration_threshold=float((group.score>thresholds[f'{name}_{seed}']).mean())))
    (OUT/'reports').mkdir(parents=True,exist_ok=True)
    trap_path=OUT/'reports'/'artifact_trap_fpr.csv'
    if trap_path.is_file():
        if digest(trap_path)!=hashlib.sha256(pd.DataFrame(trap_rows).to_csv(index=False).encode()).hexdigest():
            v4_backup_file(trap_path,'old_reports','artifact_controls')
    pd.DataFrame(trap_rows).to_csv(trap_path,index=False)
    status('P7','Computing locked source-group paired intervals, tests and figures')
    final_report(OUT,predictions,thresholds,comparator,False)
    status('FINISHED','DatePalm V4 final experiment finished with frozen scientific protocol')
    print('Result:',OUT/'reports'/'publication_primary_table.csv')


def v4_run_controlled():
    if RUN_PHASE not in V4_STAGE_NAMES:
        raise ValueError('RUN_PHASE must be one of '+repr(V4_STAGE_NAMES))
    if RUN_PHASE=='AUDIT':
        return v4_audit_inventory(full_staged_hashes=AUDIT_STAGED_SHA256)
    if RUN_PHASE=='PROMOTE':
        return v4_promote_draem()
    if RUN_PHASE in ('PRELOCK','ALL_PRELOCK'):
        if not APPROVE_PRELOCK_EVALUATION:
            raise RuntimeError('PRELOCK/P4/P5 approval missing. Set APPROVE_PRELOCK_EVALUATION=True.')
        v4_check_no_test_started()
        df,device,extra,config=v4_prepare_runtime()
        ckpts=v4_collect_ckpts(permit_missing_non_draem=True,df=df,device=device,extra=extra)
        if not V4_PROMOTION_LEDGER.exists():
            raise RuntimeError('DRAEM promotion must be independently completed before P4')
        dev,choice=v4_dev_stage(df,device,extra,ckpts)
        v4_calibration_stage(df,dev,device,extra,ckpts)
        print('PRELOCK COMPLETE: Verify selected comparator, 45 development and 45 calibration scores. '
              'Then separately authorize LOCK. No test has been opened.')
        if RUN_PHASE=='ALL_PRELOCK':
            print('ALL_PRELOCK intentionally STOPS before automatic protocol lock and test.')
        return
    if RUN_PHASE=='LOCK':return v4_create_lock()
    if RUN_PHASE=='TEST':return v4_test_stage()
    if RUN_PHASE=='POST_TEST':return v4_post_test_stage()


print('Continuation controller ready; no training, promotion or test has been executed.')


In [ ]:





from contextlib import nullcontext
from importlib.metadata import version as _v4_package_version
from pathlib import Path
from functools import wraps
import hashlib, json, torch

V4_SAFELOAD_SCOPE_RELEASE = '2026-10-18-anomalib-2.2.0-efficientad-enum-scoped-load-v1'
V4_PRE_SAFELOAD_EVAL_RELEASE = V4_EVAL_RELEASE


V4_PRE_SAFELOAD_SCORING_SHA256 = v4_metric_release_signature()
V4_PRE_SAFELOAD_CODE_FINGERPRINT = code_fingerprint()

if _v4_package_version('anomalib') != '2.2.0':
    raise RuntimeError('SAFELOAD STOP: Expected anomalib==2.2.0; do not change dependencies.')
if not hasattr(torch.serialization, 'safe_globals'):
    raise RuntimeError('SAFELOAD STOP: PyTorch restricted loader API absent.')
from anomalib.models.image.efficient_ad.torch_model import EfficientAdModelSize
if (EfficientAdModelSize.__module__ != 'anomalib.models.image.efficient_ad.torch_model'
        or EfficientAdModelSize.__qualname__ != 'EfficientAdModelSize'):
    raise RuntimeError('SAFELOAD STOP: Unexpected EfficientAdModelSize identity.')


_v4_scan = getattr(torch.serialization, 'get_unsafe_globals_in_checkpoint', None)
if _v4_scan is None:
    raise RuntimeError('SAFELOAD STOP: PyTorch static checkpoint inspector unavailable.')
_v4_expected_global = 'anomalib.models.image.efficient_ad.torch_model.EfficientAdModelSize'
_v4_ckpts = []
for _v4_seed in (1049,2053,3067,4093,5051):
    _v4_path = OUT / 'models' / 'EfficientAd' / str(_v4_seed) / 'baseline.ckpt'
    if not _v4_path.is_file():
        raise RuntimeError('SAFELOAD STOP: Official EfficientAD checkpoint absent: '+str(_v4_path))
    _v4_unsafe = set(_v4_scan(str(_v4_path)))
    _v4_unexpected = sorted(_v4_unsafe - {_v4_expected_global})
    if _v4_unexpected:
        raise RuntimeError('SAFELOAD STOP: Unexpected checkpoint globals in seed '+str(_v4_seed)+': '+repr(_v4_unexpected))
    _v4_ckpts.append((_v4_seed,sorted(_v4_unsafe)))
print('SAFELOAD PREFLIGHT: 5/5 EfficientAD checkpoints contain no unexpected unsafe GLOBALs.',flush=True)


def v4_efficientad_only_checkpoint_scope(method):
    """Temporary trust for ONE known Enum in ONE named baseline's inference.

    `safe_globals` exits on success/error. All other methods remain at
    PyTorch's restricted defaults. No `weights_only=False` is ever set.
    """
    if method != 'EfficientAd':
        return nullcontext()
    return torch.serialization.safe_globals([EfficientAdModelSize])



_v4_before_globals = list(torch.serialization.get_safe_globals())
with v4_efficientad_only_checkpoint_scope('EfficientAd'):
    if EfficientAdModelSize not in torch.serialization.get_safe_globals():
        raise RuntimeError('SAFELOAD STOP: Temporary enum allowlisting failed.')
if list(torch.serialization.get_safe_globals()) != _v4_before_globals:
    raise RuntimeError('SAFELOAD STOP: Temporary safe_globals policy leaked globally.')




_V4_ORIGINAL_PREDICT_FROM_CHECKPOINT = predict_from_checkpoint
_V4_ORIGINAL_STREAMING_COLLATOR = _v4_prediction_collator

@wraps(_V4_ORIGINAL_PREDICT_FROM_CHECKPOINT)
def predict_from_checkpoint(checkpoint, name, *args, **kwargs):
    with v4_efficientad_only_checkpoint_scope(name):
        return _V4_ORIGINAL_PREDICT_FROM_CHECKPOINT(checkpoint, name, *args, **kwargs)

@wraps(_V4_ORIGINAL_STREAMING_COLLATOR)
def _v4_prediction_collator(checkpoint, name, *args, **kwargs):
    with v4_efficientad_only_checkpoint_scope(name):
        return _V4_ORIGINAL_STREAMING_COLLATOR(checkpoint, name, *args, **kwargs)



V4_EVAL_RELEASE = V4_SAFELOAD_SCOPE_RELEASE
_V4_ORIGINAL_SCORE_CONTRACT = v4_score_contract

@wraps(_V4_ORIGINAL_SCORE_CONTRACT)
def v4_score_contract(checkpoint, samples, output, do_localization):
    contract = dict(_V4_ORIGINAL_SCORE_CONTRACT(checkpoint,samples,output,do_localization))
    code = _stable_code(v4_efficientad_only_checkpoint_scope.__code__)
    contract['safe_load_policy'] = V4_SAFELOAD_SCOPE_RELEASE
    contract['safe_load_scope_code_sha256'] = hashlib.sha256(
        json.dumps(code,sort_keys=True,default=str).encode('utf-8')
    ).hexdigest()
    return contract


def v4_migrate_strictly_verified_prelock_score(checkpoint, name, samples, output, phase):
    """Preserve byte-for-byte prior P4/P5 scoring only when FULL provenance matches.

    Only a load-only compatibility change is allowed. The original metadata
    is backed up and SHA256-verified before any record is updated. Legacy
    7-image spot checks remain UNVERIFIED and will be recomputed as needed.
    """
    output = Path(output)
    metadata = output.with_suffix('.csv.provenance.json')
    if phase not in ('P4','P5') or not output.is_file() or not metadata.is_file():
        return False

    v4_check_no_test_started()
    if (OUT/'frozen'/'protocol_lock.json').exists() or (OUT/'frozen'/'controller_lock.json').exists():
        return False
    phase_root = (OUT / ('development' if phase == 'P4' else 'calibration') / 'scores').resolve()
    if output.resolve().parent != phase_root:
        return False
    record = v4_read_json_strict(metadata)
    if (record.get('release') != V4_PRE_SAFELOAD_EVAL_RELEASE or
            record.get('scoring_code_sha256') != V4_PRE_SAFELOAD_SCORING_SHA256):
        return False
    if record.get('cache_origin') == 'legacy_spot_checked_unverifiable_full_provenance':
        return False
    expected = v4_score_contract(checkpoint, samples, output, False)


    for field in ('schema_version','checkpoint_sha256','benchmark_sha256',
                  'localization','score_file_name'):
        if record.get(field) != expected.get(field):
            return False
    actual_csv_sha256 = digest(output)
    if record.get('score_csv_sha256') != actual_csv_sha256:
        raise RuntimeError('SAFELOAD STOP: P4/P5 score hash differs from existing provenance: '+str(output))

    frame = pd.read_csv(output,dtype={'sample_id':str,'source_id':str,'source_group':str})
    v4_verify_score_table(frame, samples, False)
    original_metadata_sha = digest(metadata)
    backup = v4_backup_file(metadata, 'safe_load_metadata_before_migration',phase+'_'+name)
    if backup is None or not Path(backup).is_file() or digest(backup) != original_metadata_sha:
        raise RuntimeError('SAFELOAD STOP: Pre-migration metadata not independently preserved.')
    amended = dict(record)
    amended.update(expected)
    amended.update({
        'score_csv_sha256':actual_csv_sha256,
        'serialization_compatibility_amendment':V4_SAFELOAD_SCOPE_RELEASE,
        'compatibility_changes_training_or_scoring_math':False,
        'safe_load_migration_prior_release':V4_PRE_SAFELOAD_EVAL_RELEASE,
        'safe_load_migration_prior_scoring_sha256':V4_PRE_SAFELOAD_SCORING_SHA256,
        'safe_load_migration_prior_code_fingerprint':V4_PRE_SAFELOAD_CODE_FINGERPRINT,
        'safe_load_migration_metadata_backup_sha256':original_metadata_sha,
        'safe_load_migration_metadata_backup_file':str(backup),
        'safe_load_migration_basis':'Temporary restricted-loader allowlist only; original evaluation and score bytes preserved',
    })
    write_json(metadata, amended)
    if digest(output) != actual_csv_sha256:
        raise RuntimeError('SAFELOAD STOP: Score CSV changed during provenance migration.')
    print('  SAFELOAD RESUME: verified pre-lock score reused '+output.name,flush=True)
    return True


_V4_ORIGINAL_PRELOCK_SCORE = v4_prelock_score

@wraps(_V4_ORIGINAL_PRELOCK_SCORE)
def v4_prelock_score(checkpoint, name, samples, device, extra, outfile, phase):



    v4_migrate_strictly_verified_prelock_score(checkpoint,name,samples,outfile,phase)
    return _V4_ORIGINAL_PRELOCK_SCORE(checkpoint,name,samples,device,extra,outfile,phase)



if v4_metric_release_signature() == V4_PRE_SAFELOAD_SCORING_SHA256:
    raise RuntimeError('SAFELOAD STOP: Compatibility wrapper not bound to score signature.')
if code_fingerprint() == V4_PRE_SAFELOAD_CODE_FINGERPRINT:
    raise RuntimeError('SAFELOAD STOP: Compatibility wrapper not bound to scientific code fingerprint.')
print('SAFELOAD FIX READY: Restricted checkpoint deserialization scoped ONLY to EfficientAd.',flush=True)
print('Prior verified P4 scores can be migrated with byte-for-byte backups; no rerun if valid.',flush=True)
print('No model training, promotion, protocol lock or held-out test authorized.',flush=True)


In [ ]:















import inspect, hashlib, json
from contextlib import contextmanager
from functools import wraps
from pathlib import Path
from importlib.metadata import version as _v4_super_version
from anomalib.visualization.image.visualizer import ImageVisualizer

V4_SUPER_VIS_RELEASE='2026-10-18-anomalib-2.2.0-supersimplenet-visualizer-scoped-v1'
V4_PRE_SUPER_VIS_RELEASE=V4_EVAL_RELEASE
V4_PRE_SUPER_VIS_SCORE_SIGNATURE=v4_metric_release_signature()
V4_PRE_SUPER_VIS_CODE_FINGERPRINT=code_fingerprint()

if _v4_super_version('anomalib')!='2.2.0':
    raise RuntimeError('SUPER VIS STOP: Expected pinned anomalib 2.2.0.')

_V4_IMAGE_VISUALIZER_ON_PREDICT_ORIGINAL=ImageVisualizer.on_predict_batch_end
_V4_IMAGE_VISUALIZER_ON_TEST_ORIGINAL=ImageVisualizer.on_test_batch_end
_v4_visualizer_predict_source=inspect.getsource(_V4_IMAGE_VISUALIZER_ON_PREDICT_ORIGINAL)
_v4_visualizer_test_source=inspect.getsource(_V4_IMAGE_VISUALIZER_ON_TEST_ORIGINAL)
if ('return self.on_test_batch_end(' not in _v4_visualizer_predict_source or
        'for item in batch:' not in _v4_visualizer_test_source):
    raise RuntimeError('SUPER VIS STOP: ImageVisualizer behavior differs from audited Anomalib 2.2.0.')


def _v4_non_scientific_visualizer_noop(self, trainer, pl_module, outputs, batch, batch_idx, dataloader_idx=0):
    """Skip PNG generation only; NEVER mutate prediction tensors or batch."""
    return None


@contextmanager
def v4_supersimplenet_visualizer_scope(method):
    """Modify a non-scientific callback for one Supersimplenet predict call.

    The original callback is restored on both success and exception. The
    exclusive PRELOCK process lock prevents competing experimental writers.
    """
    if method!='Supersimplenet':
        yield
        return
    if ImageVisualizer.on_predict_batch_end is not _V4_IMAGE_VISUALIZER_ON_PREDICT_ORIGINAL:
        raise RuntimeError('SUPER VIS STOP: ImageVisualizer was modified by another component.')
    ImageVisualizer.on_predict_batch_end=_v4_non_scientific_visualizer_noop
    try:
        yield
    finally:
        ImageVisualizer.on_predict_batch_end=_V4_IMAGE_VISUALIZER_ON_PREDICT_ORIGINAL



assert ImageVisualizer.on_predict_batch_end is _V4_IMAGE_VISUALIZER_ON_PREDICT_ORIGINAL
with v4_supersimplenet_visualizer_scope('Patchcore'):
    assert ImageVisualizer.on_predict_batch_end is _V4_IMAGE_VISUALIZER_ON_PREDICT_ORIGINAL
with v4_supersimplenet_visualizer_scope('Supersimplenet'):
    assert ImageVisualizer.on_predict_batch_end is _v4_non_scientific_visualizer_noop
assert ImageVisualizer.on_predict_batch_end is _V4_IMAGE_VISUALIZER_ON_PREDICT_ORIGINAL
try:
    with v4_supersimplenet_visualizer_scope('Supersimplenet'):
        raise ValueError('scoped preflight exception')
except ValueError:
    pass
assert ImageVisualizer.on_predict_batch_end is _V4_IMAGE_VISUALIZER_ON_PREDICT_ORIGINAL




_V4_PRE_SUPER_PREDICT_FROM_CHECKPOINT=predict_from_checkpoint
_V4_PRE_SUPER_STREAMING_COLLATOR=_v4_prediction_collator

@wraps(_V4_PRE_SUPER_PREDICT_FROM_CHECKPOINT)
def predict_from_checkpoint(checkpoint, name, *args, **kwargs):
    with v4_supersimplenet_visualizer_scope(name):
        return _V4_PRE_SUPER_PREDICT_FROM_CHECKPOINT(checkpoint,name,*args,**kwargs)

@wraps(_V4_PRE_SUPER_STREAMING_COLLATOR)
def _v4_prediction_collator(checkpoint, name, *args, **kwargs):
    with v4_supersimplenet_visualizer_scope(name):
        return _V4_PRE_SUPER_STREAMING_COLLATOR(checkpoint,name,*args,**kwargs)




V4_EVAL_RELEASE=V4_SUPER_VIS_RELEASE
_V4_PRE_SUPER_SCORE_CONTRACT=v4_score_contract

def v4_score_contract(checkpoint,samples,output,do_localization):
    record=dict(_V4_PRE_SUPER_SCORE_CONTRACT(checkpoint,samples,output,do_localization))
    name=Path(output).name.split('_',1)[0]
    record['visualization_callback_policy']=(
        'Supersimplenet_ImageVisualizer_prediction_render_disabled_only'
        if name=='Supersimplenet' else 'unchanged')
    record['visualizer_scope_release']=V4_SUPER_VIS_RELEASE
    record['visualizer_scope_code_sha256']=hashlib.sha256(
        json.dumps(_stable_code(v4_supersimplenet_visualizer_scope.__code__),
                   sort_keys=True,default=str).encode('utf-8')
    ).hexdigest()
    return record


def v4_migrate_previsualizer_verified_score(checkpoint,name,samples,output,phase):
    """Migrate only FULLY verified prelock score provenance, never score values.

    Every historical metadata file is SHA256-backed-up before a new record is
    written. Reject unknown evaluator releases, spot-checked legacy provenance,
    wrong score image populations, mismatched model or benchmark hashes.
    """
    output=Path(output)
    meta=output.with_suffix('.csv.provenance.json')
    if phase not in ('P4','P5') or not output.is_file() or not meta.is_file():
        return False
    v4_check_no_test_started()
    if (OUT/'frozen'/'controller_lock.json').exists():
        raise RuntimeError('SUPER VIS STOP: Controller lock already exists.')
    expected_folder=(OUT/('development' if phase=='P4' else 'calibration')/'scores').resolve()
    if output.resolve().parent!=expected_folder:
        raise RuntimeError('SUPER VIS STOP: Score directory not the registered P4/P5 path.')
    old=v4_read_json_strict(meta)
    if (old.get('release')!=V4_PRE_SUPER_VIS_RELEASE or
            old.get('scoring_code_sha256')!=V4_PRE_SUPER_VIS_SCORE_SIGNATURE):
        return False
    if old.get('cache_origin')=='legacy_spot_checked_unverifiable_full_provenance':
        return False
    contract=v4_score_contract(checkpoint,samples,output,False)
    for field in ('schema_version','checkpoint_sha256','benchmark_sha256',
                  'localization','score_file_name','safe_load_policy',
                  'safe_load_scope_code_sha256'):
        if old.get(field)!=contract.get(field):
            return False
    csv_hash=digest(output)
    if old.get('score_csv_sha256')!=csv_hash:
        raise RuntimeError('SUPER VIS STOP: Score CSV hash differs from its signed provenance: '+str(output))
    frame=pd.read_csv(output,dtype={'sample_id':str,'source_id':str,'source_group':str})
    v4_verify_score_table(frame,samples,False)
    old_meta_hash=digest(meta)
    backup=v4_backup_file(meta,'supersimplenet_visualizer_metadata_before_migration',
                          phase+'_'+name)
    if backup is None or not Path(backup).is_file() or digest(backup)!=old_meta_hash:
        raise RuntimeError('SUPER VIS STOP: Existing metadata backup was not hash-verified.')
    amended=dict(old)
    amended.update(contract)
    amended.update({
        'score_csv_sha256':csv_hash,
        'visualization_callback_amendment':V4_SUPER_VIS_RELEASE,
        'compatibility_changes_scoring_math':False,
        'visualization_migration_prior_release':V4_PRE_SUPER_VIS_RELEASE,
        'visualization_migration_prior_scoring_sha256':V4_PRE_SUPER_VIS_SCORE_SIGNATURE,
        'visualization_migration_prior_code_fingerprint':V4_PRE_SUPER_VIS_CODE_FINGERPRINT,
        'visualization_migration_metadata_backup_sha256':old_meta_hash,
        'visualization_migration_metadata_backup_file':str(backup),
        'visualization_migration_basis':
            'Visualization-only Anomalib callback repair; score CSV unchanged; '
            'source, benchmark, checkpoint, score population and source hashes verified',
    })
    write_json(meta,amended)
    if digest(output)!=csv_hash:
        raise RuntimeError('SUPER VIS STOP: Score CSV changed during migration.')
    print('  SUPER VIS RESUME: verified score preserved '+output.name,flush=True)
    return True


_V4_PRE_SUPER_PRELOCK_SCORE=v4_prelock_score

@wraps(_V4_PRE_SUPER_PRELOCK_SCORE)
def v4_prelock_score(checkpoint,name,samples,device,extra,outfile,phase):
    v4_migrate_previsualizer_verified_score(checkpoint,name,samples,outfile,phase)
    return _V4_PRE_SUPER_PRELOCK_SCORE(checkpoint,name,samples,device,extra,outfile,phase)



if v4_metric_release_signature()==V4_PRE_SUPER_VIS_SCORE_SIGNATURE:
    raise RuntimeError('SUPER VIS STOP: Amended prediction wrappers are not bound to evaluation fingerprint.')
if code_fingerprint()==V4_PRE_SUPER_VIS_CODE_FINGERPRINT:
    raise RuntimeError('SUPER VIS STOP: Amendment missing from scientific code fingerprint.')
print('SUPERSIMPLENET VISUALIZER FIX READY: only optional prediction-image rendering disabled.',flush=True)
print('Anomalib post-processor, CUDA/float32 math, scores/maps and parity checks unchanged.',flush=True)
print('Verified historical P4/P5 scores are hash-backed-up and re-used when safe.',flush=True)
print('No retraining, no promotion, no protocol locking, no locked test.',flush=True)


In [ ]:
















from contextlib import contextmanager
from functools import wraps
from importlib.metadata import version as _v4_scalar_package_version
from pathlib import Path
from anomalib.engine import Engine
import hashlib, json, torch

V4_SUPER_SCALAR_RELEASE='2026-10-18-anomalib-2.2.0-supersimplenet-singleton-score-shape-v1'
V4_PRE_SUPER_SCALAR_RELEASE=V4_EVAL_RELEASE
V4_PRE_SUPER_SCALAR_SCORING_SHA256=v4_metric_release_signature()
V4_PRE_SUPER_SCALAR_CODE_FINGERPRINT=code_fingerprint()

if _v4_scalar_package_version('anomalib')!='2.2.0':
    raise RuntimeError('SCALAR SHAPE STOP: Expected anomalib==2.2.0.')

_V4_ORIGINAL_ENGINE_PREDICT_SCALAR=Engine.predict


def v4_shape_only_normalize_singleton_scores(predictions):
    """Normalize only a verified one-image *postprocessed* scalar score.

    Never changes score value, model output, map data or prediction order.
    Returns the same collection of prediction batches.
    """
    if not isinstance(predictions,(list,tuple)):
        raise RuntimeError('SCALAR SHAPE STOP: Unexpected reference prediction container')
    for batch in predictions:
        paths=getattr(batch,'image_path',None)
        values=getattr(batch,'pred_score',None)
        if not torch.is_tensor(values) or values.ndim!=0:
            continue
        count=1 if isinstance(paths,(str,Path)) else len(paths) if paths is not None else -1
        if count!=1:
            raise RuntimeError('SCALAR SHAPE STOP: Scalar score accompanies non-singleton batch')
        adjusted=values.reshape(1)
        if (adjusted.ndim!=1 or adjusted.numel()!=1 or
                adjusted.dtype!=values.dtype or adjusted.device!=values.device or
                not torch.equal(adjusted[0],values)):
            raise RuntimeError('SCALAR SHAPE STOP: Non-identical score after reshape')
        setattr(batch,'pred_score',adjusted)
    return predictions


@contextmanager
def v4_supersimplenet_scalar_reference_scope(method):
    """Temporarily normalize only Anomalib Supersimplenet reference outputs."""
    if method!='Supersimplenet':
        yield
        return
    if Engine.predict is not _V4_ORIGINAL_ENGINE_PREDICT_SCALAR:
        raise RuntimeError('SCALAR SHAPE STOP: Engine.predict was modified elsewhere')

    @wraps(_V4_ORIGINAL_ENGINE_PREDICT_SCALAR)
    def v4_predict_only_shape_fix(self,*args,**kwargs):
        results=_V4_ORIGINAL_ENGINE_PREDICT_SCALAR(self,*args,**kwargs)
        if kwargs.get('return_predictions',None) is True:
            return v4_shape_only_normalize_singleton_scores(results)
        return results

    Engine.predict=v4_predict_only_shape_fix
    try:
        yield
    finally:
        Engine.predict=_V4_ORIGINAL_ENGINE_PREDICT_SCALAR




from types import SimpleNamespace as _V4_SimpleNamespace
_v4_probe=torch.tensor(0.28125,dtype=torch.float32)
_v4_batch=_V4_SimpleNamespace(image_path=['one.png'],pred_score=_v4_probe)
_v4_checked=v4_shape_only_normalize_singleton_scores([_v4_batch])
assert _v4_checked[0].pred_score.shape==(1,)
assert torch.equal(_v4_checked[0].pred_score[0],_v4_probe)
assert _v4_checked[0].pred_score.data_ptr()==_v4_probe.data_ptr()
assert _v4_checked[0].pred_score.dtype==_v4_probe.dtype
assert Engine.predict is _V4_ORIGINAL_ENGINE_PREDICT_SCALAR
with v4_supersimplenet_scalar_reference_scope('Patchcore'):
    assert Engine.predict is _V4_ORIGINAL_ENGINE_PREDICT_SCALAR
with v4_supersimplenet_scalar_reference_scope('Supersimplenet'):
    assert Engine.predict is not _V4_ORIGINAL_ENGINE_PREDICT_SCALAR
assert Engine.predict is _V4_ORIGINAL_ENGINE_PREDICT_SCALAR
try:
    with v4_supersimplenet_scalar_reference_scope('Supersimplenet'):
        raise ValueError('restoration test')
except ValueError:
    pass
assert Engine.predict is _V4_ORIGINAL_ENGINE_PREDICT_SCALAR



_V4_PRE_SCALAR_REFERENCE=predict_from_checkpoint
@wraps(_V4_PRE_SCALAR_REFERENCE)
def predict_from_checkpoint(checkpoint,name,*args,**kwargs):
    with v4_supersimplenet_scalar_reference_scope(name):
        return _V4_PRE_SCALAR_REFERENCE(checkpoint,name,*args,**kwargs)





V4_EVAL_RELEASE=V4_SUPER_SCALAR_RELEASE
_V4_PRE_SCALAR_CONTRACT=v4_score_contract

def v4_score_contract(checkpoint,samples,output,do_localization):
    record=dict(_V4_PRE_SCALAR_CONTRACT(checkpoint,samples,output,do_localization))
    method=Path(output).name.split('_',1)[0]
    record['singleton_score_shape_policy']=(
        'Supersimplenet_postprocessed_single_image_scalar_to_vector_view_only'
        if method=='Supersimplenet' else 'unchanged')
    record['singleton_shape_release']=V4_SUPER_SCALAR_RELEASE
    record['singleton_shape_code_sha256']=hashlib.sha256(
        json.dumps(_stable_code(v4_shape_only_normalize_singleton_scores.__code__),
                   sort_keys=True,default=str).encode('utf-8')
    ).hexdigest()
    return record


def v4_migrate_scalar_compatibility_score(checkpoint,name,samples,outfile,phase):
    """Retain scientifically verified P4/P5 score CSVs without recalculation."""
    outfile=Path(outfile)
    metadata=outfile.with_suffix('.csv.provenance.json')
    if phase not in ('P4','P5') or not outfile.is_file() or not metadata.is_file():
        return False
    v4_check_no_test_started()
    if (OUT/'frozen'/'protocol_lock.json').exists() or (OUT/'frozen'/'controller_lock.json').exists():
        return False
    expected_dir=(OUT/('development' if phase=='P4' else 'calibration')/'scores').resolve()
    if outfile.resolve().parent!=expected_dir:
        raise RuntimeError('SCALAR SHAPE STOP: Score location differs from frozen phase path')
    previous=v4_read_json_strict(metadata)
    if (previous.get('release')!=V4_PRE_SUPER_SCALAR_RELEASE or
            previous.get('scoring_code_sha256')!=V4_PRE_SUPER_SCALAR_SCORING_SHA256 or
            previous.get('cache_origin')=='legacy_spot_checked_unverifiable_full_provenance'):
        return False
    contract=v4_score_contract(checkpoint,samples,outfile,False)
    for field in ('schema_version','checkpoint_sha256','benchmark_sha256',
                  'localization','score_file_name','safe_load_policy',
                  'safe_load_scope_code_sha256','visualization_callback_policy',
                  'visualizer_scope_release','visualizer_scope_code_sha256'):
        if previous.get(field)!=contract.get(field):
            return False
    actual_hash=digest(outfile)
    if previous.get('score_csv_sha256')!=actual_hash:
        raise RuntimeError('SCALAR SHAPE STOP: Score digest contradicts its provenance')
    scoreframe=pd.read_csv(outfile,dtype={'sample_id':str,'source_id':str,'source_group':str})
    v4_verify_score_table(scoreframe,samples,False)
    original_meta_sha=digest(metadata)
    backup=v4_backup_file(metadata,'singleton_shape_metadata_before_migration',phase+'_'+name)
    if backup is None or not Path(backup).is_file() or digest(backup)!=original_meta_sha:
        raise RuntimeError('SCALAR SHAPE STOP: Original metadata backup not hash verified')
    amended=dict(previous)
    amended.update(contract)
    amended.update({
        'score_csv_sha256':actual_hash,
        'singleton_shape_compatibility_amendment':V4_SUPER_SCALAR_RELEASE,
        'singleton_shape_changes_scores_or_anomaly_maps':False,
        'singleton_shape_migration_previous_release':V4_PRE_SUPER_SCALAR_RELEASE,
        'singleton_shape_migration_previous_score_signature':V4_PRE_SUPER_SCALAR_SCORING_SHA256,
        'singleton_shape_migration_previous_code_fingerprint':V4_PRE_SUPER_SCALAR_CODE_FINGERPRINT,
        'singleton_shape_original_metadata_sha256':original_meta_sha,
        'singleton_shape_original_metadata_backup':str(backup),
        'singleton_shape_migration_basis':
            'Only Supersimplenet one-item reference score tensor container shape changed; '
            'original score CSV unchanged; benchmark/checkpoint/score population hashes verified',
    })
    write_json(metadata,amended)
    if digest(outfile)!=actual_hash:
        raise RuntimeError('SCALAR SHAPE STOP: Score CSV unexpectedly changed')
    print('  SCALAR SHAPE RESUME: verified pre-lock score reused '+outfile.name,flush=True)
    return True


_V4_PRE_SCALAR_PRELOCK_SCORE=v4_prelock_score
@wraps(_V4_PRE_SCALAR_PRELOCK_SCORE)
def v4_prelock_score(checkpoint,name,samples,device,extra,outfile,phase):
    v4_migrate_scalar_compatibility_score(checkpoint,name,samples,outfile,phase)
    return _V4_PRE_SCALAR_PRELOCK_SCORE(checkpoint,name,samples,device,extra,outfile,phase)

if v4_metric_release_signature()==V4_PRE_SUPER_SCALAR_SCORING_SHA256:
    raise RuntimeError('SCALAR SHAPE STOP: Prediction compatibility code not fingerprinted')
if code_fingerprint()==V4_PRE_SUPER_SCALAR_CODE_FINGERPRINT:
    raise RuntimeError('SCALAR SHAPE STOP: Amendment not included in executable fingerprint')
print('SUPERSIMPLENET SINGLETON SCORE SHAPE FIX READY',flush=True)
print('Only scalar->length-1 postprocessed score container is normalized.',flush=True)
print('Trained models, score arithmetic, maps and parity checks unchanged.',flush=True)
print('Previous verified scores retained after SHA256-backed-up provenance migration.',flush=True)
print('PRELOCK only; no training, promotion, lock or final test.',flush=True)


In [ ]:



from torchvision.transforms.v2 import functional as V4_TF

V4_HUE_TOLERANCE = 1.0e-6
V4_HUE_PROTOCOL = 'datepalm-v4-draem-hue-input-range-v1'

def install_v4_draem_hue_guard(draem, stats=None, *, verify_layout=True):
    import torch, math
    from importlib.metadata import version
    if version('anomalib')!='2.2.0' or type(draem).__name__!='Draem':
        raise RuntimeError('Only the verified Anomalib 2.2.0 Draem is supported')
    aug = draem.augmenter
    transforms = getattr(getattr(aug,'augmenters',None),'transforms',None)
    if not isinstance(transforms,(list,tuple,torch.nn.ModuleList)) or len(transforms)!=10:
        raise RuntimeError('Unexpected DRAEM texture transform list; fail closed')
    expected = ['ColorJitter','RandomPhotometricDistort','RandomAdjustSharpness',
                'ColorJitter','RandomSolarize','RandomPosterize','RandomInvert',
                'AutoAugment','RandomEqualize','RandomAffine']
    names = [type(t).__name__ for t in transforms]
    if names!=expected:
        raise RuntimeError(f'Anomalib texture transform list changed: {names!r}')
    if stats is None: stats={'hue_calls':0,'corrected_calls':0,'corrected_pixels':0}
    if getattr(draem,'_v4_hue_guard_installed',False):
        raise RuntimeError('Hue guard was already installed on this model')
    tracked=[]
    for i in (0,3):
        obj=transforms[i]
        original=obj._call_kernel
        def guarded_call(functional, inpt, *args, __original=original, __index=i, **kwargs):
            if functional is not V4_TF.adjust_hue:
                return __original(functional,inpt,*args,**kwargs)
            stats['hue_calls']+=1
            if not torch.is_tensor(inpt) or not inpt.is_floating_point():
                raise RuntimeError('Unexpected DRAEM hue input type; refuse implicit conversion')
            minimum=float(torch.amin(inpt).item())
            maximum=float(torch.amax(inpt).item())
            if not math.isfinite(minimum) or not math.isfinite(maximum):
                raise FloatingPointError('DRAEM hue input non-finite before ColorJitter')
            if minimum < -V4_HUE_TOLERANCE or maximum > 1.0+V4_HUE_TOLERANCE:
                raise FloatingPointError(
                    f'DRAEM hue input outside tiny-roundoff tolerance: min={minimum}, max={maximum}, transform_index={__index}')
            if minimum < 0:
                negatives=int((inpt<0).sum().item())
                stats['corrected_calls']+=1
                stats['corrected_pixels']+=negatives
                stats['most_negative_corrected']=min(float(stats.get('most_negative_corrected',0.0)),minimum)
                inpt=inpt.clamp_min(0.0)
            out=__original(functional,inpt,*args,**kwargs)
            if not torch.isfinite(out).all().item():
                raise FloatingPointError('Hue kernel still returned non-finite data after range guard')
            return out
        obj._call_kernel=guarded_call
        tracked.append((obj,original))
    draem._v4_hue_guard_installed=True
    draem._v4_hue_guard_original_methods=tracked
    return stats

def uninstall_v4_draem_hue_guard(draem):
    for obj,old in getattr(draem,'_v4_hue_guard_original_methods',[]):
        obj._call_kernel=old
    draem._v4_hue_guard_original_methods=[]
    draem._v4_hue_guard_installed=False

In [ ]:
def v4_repair_validate_checkpoint(path, require_ranges=True):
    import torch, numpy as np
    path=Path(path)
    try:
        d=torch.load(path,map_location='cpu',weights_only=True,mmap=True)
    except TypeError:
        d=torch.load(path,map_location='cpu',weights_only=True)
    if not isinstance(d,dict) or not isinstance(d.get('state_dict'),dict):
        raise RuntimeError('Invalid Lightning DRAEM checkpoint state_dict: '+str(path))
    sd=d['state_dict']
    bad=[];count=0
    for k,v in sd.items():
        if k.startswith('model.') and torch.is_tensor(v) and v.is_floating_point():
            count+=1
            if not torch.isfinite(v).all().item():bad.append(k)
    if bad:
        raise FloatingPointError(f'Checkpoint contains {len(bad)} non-finite learned tensors (e.g. {bad[:3]}): {path}')
    bounds={}
    for term in ('image_min','image_max','pixel_min','pixel_max'):
        matches=[(k,v) for k,v in sd.items() if k.endswith('post_processor.'+term)]
        if len(matches)>1: raise RuntimeError('Ambiguous normalization bounds: '+term)
        if len(matches)==1:
            val=matches[0][1]
            if not torch.is_tensor(val) or val.numel()!=1:raise RuntimeError('Malformed bound '+term)
            num=float(val.item())
            if not np.isfinite(num): raise FloatingPointError('Non-finite DRAEM normalization '+term+': '+str(path))
            bounds[term]=num
        elif require_ranges:
            raise RuntimeError('Missing DRAEM normalization '+term+': '+str(path))
    if require_ranges:
        for prefix in ('image','pixel'):
            if bounds[prefix+'_max']<=bounds[prefix+'_min']:
                raise FloatingPointError('Non-positive '+prefix+' normalization range in '+str(path))
    return {'sha256':digest(path),'learned_tensors_checked':count,'normalization':bounds,
            'global_step':int(d.get('global_step',0)),'epoch':int(d.get('epoch',-1))}

In [ ]:
if False:

    from pathlib import Path
    import shutil, os, json, traceback
    from datetime import datetime,timezone
    from importlib.metadata import version

    ALL_DRAEM_SEEDS=(1049,2053,3067,4093,5051)
    STAGE_ROOT=OUT/'diagnostics'/'draem_hue_guard_v1_stage'
    VERIFICATION=OUT/'diagnostics'/'draem_hue_guard_verification'/'step4036_verification.json'
    LOCK=OUT/'frozen'/'protocol_lock.json'
    LOCK_FLAG=OUT/'FINAL_TEST_USED.flag'
    APPROVE_ALL_FIVE_RETRAIN=True

    assert version('anomalib')=='2.2.0', 'Anomalib version changed'
    assert BASELINE_FIT_CONFIG['Draem']==dict(train_batch_size=8,eval_batch_size=8,max_epochs=60,check_val_every_n_epoch=5), 'Original DRAEM schedule changed'
    assert not LOCK.exists() and not LOCK_FLAG.exists(), 'Locked protocol or test accessed: do not retrain or replace any seed'
    assert (OUT/'data'/'source_split.csv').is_file(), 'Frozen split missing'
    assert RAW.is_dir(), 'Raw source dataset directory missing'
    assert VERIFICATION.is_file(), 'Exact failure replay verification must be run first'
    proof=json.loads(VERIFICATION.read_text())
    assert proof.get('result')=='REPAIRED_EXACT_STEP4036_REPLAY' and proof.get('protocol')==V4_HUE_PROTOCOL, 'Snapshot proof missing/incompatible'
    assert proof.get('snapshot_sha256')=='e3a73b5759cdd374c829964911f54af0accf89a0d9d440579d0a4adb638214bd', 'Snapshot hash mismatch'
    assert torch.cuda.is_available() and torch.cuda.get_device_name(0)=='NVIDIA GeForce RTX 5090', 'Expected RTX 5090 required'
    assert shutil.disk_usage(OUT).free>=11*1024**3, 'At least 11 GiB disk free required for five staged checkpoints'
    for seed in ALL_DRAEM_SEEDS:
        original=OUT/'models'/'Draem'/str(seed)
        if not original.is_dir():raise RuntimeError('Original seed directory missing: '+str(original))
        if seed in (2053,3067,4093):
            v4_repair_validate_checkpoint(original/'baseline.ckpt',require_ranges=True)

    if (OUT/'DATASET_MANIFEST_SHA256.txt').exists():
        expected=(OUT/'DATASET_MANIFEST_SHA256.txt').read_text().strip()
        actual=digest(OUT/'data'/'source_split.csv')
        if actual!=expected:raise RuntimeError('Frozen source split digest changed')

    print('PREFLIGHT PASS: pinned 5-seed correction, valid original references, locked test untouched')
    print('Staging root:',STAGE_ROOT)
    print('APPROVE_ALL_FIVE_RETRAIN =',APPROVE_ALL_FIVE_RETRAIN)



    if not APPROVE_ALL_FIVE_RETRAIN:
        print('DRY RUN: no GPU training, no checkpoint modifications. Set APPROVE_ALL_FIVE_RETRAIN=True only after the snapshot repair succeeds.')
    else:
        import subprocess
        proc=subprocess.run(['nvidia-smi','--query-compute-apps=pid','--format=csv,noheader,nounits'],capture_output=True,text=True,timeout=15)
        if proc.returncode==0:
            foreign=[x.strip() for x in proc.stdout.splitlines() if x.strip().isdigit() and int(x.strip())!=os.getpid()]
            if foreign:raise RuntimeError('Other GPU compute PID(s) detected; stop other jobs first: '+str(foreign))
        df=pd.read_csv(OUT/'data'/'source_split.csv',dtype={'source_id':str,'source_group':str,'relative_path':str})
        if not {'split','eligibility','relative_path','file_sha256'}.issubset(df.columns):
            raise RuntimeError('Frozen source split schema changed')
        for row in df[['relative_path','file_sha256']].drop_duplicates().to_dict('records'):
            if digest(RAW/row['relative_path'])!=row['file_sha256']:
                raise RuntimeError('Source changed: '+str(row['relative_path']))
        dtd=OUT/'auxiliary_baseline_data'/'dtd'
        assert (dtd/'images').is_dir(), 'Original DTD is absent'
        prepared=(OUT/'data'/'baseline_shared'/'_prepared.ok').exists() or (OUT/'models'/'Patchcore'/'1049'/'datamodule'/'_prepared.ok').exists()
        assert prepared,'Original baseline training data cache missing'
        EXTRA={'dtd_dir':str(dtd),'shared_baseline_data':str(OUT/'data'/'baseline_shared')}



        V4_HUE_STATS_BY_SEED={}
        ACTIVE_HUE_SEED=None
        original_build_model=build_model
        original_enable_accel=enable_v4_draem_safe_acceleration
        def build_model_with_guard(name,extra):
            model=original_build_model(name,extra)
            if name=='Draem':
                stats=install_v4_draem_hue_guard(model)
                V4_HUE_STATS_BY_SEED[int(ACTIVE_HUE_SEED)]=stats
                model._hue_guard_stats=stats
                def v4_aug_check(module,inputs,output):
                    def check(v):
                        if torch.is_tensor(v) and v.is_floating_point() and not torch.isfinite(v).all().item():
                            raise FloatingPointError('DRAEM generator still produced NaN/Inf; stop before optimizer')
                        if isinstance(v,(tuple,list)):
                            for e in v:check(e)
                        if isinstance(v,dict):
                            for e in v.values():check(e)
                    check(output)
                model.augmenter.register_forward_hook(v4_aug_check)
            return model
        def accelerated_to_stage(model,seed):
            global OUT
            previous=OUT
            try:
                OUT=STAGE_ROOT
                return original_enable_accel(model,seed)
            finally:OUT=previous
        build_model=build_model_with_guard
        enable_v4_draem_safe_acceleration=accelerated_to_stage
        STAGE_ROOT.mkdir(parents=True,exist_ok=True)
        try:
            for seed in ALL_DRAEM_SEEDS:
                base=STAGE_ROOT/str(seed)
                done=base/'COMPLETE.json'
                ckpt=base/'baseline.ckpt'
                if done.exists():
                    assert ckpt.is_file(), 'Completed stage marker without checkpoint'
                    info=v4_repair_validate_checkpoint(ckpt,require_ranges=True)
                    assert info['epoch']==60 and info['global_step']==27060, 'Staged complete checkpoint is not 60 epochs/27060 steps'
                    print(f'SKIPPING verified finished staged seed {seed}')
                    continue


                recovery=base/'in_progress_full_state.ckpt'
                if recovery.exists():
                    archive=base/'interrupted_evidence'/datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
                    archive.mkdir(parents=True,exist_ok=False)
                    os.replace(recovery,archive/recovery.name)
                    print('Moved non-reproducibly resumable partial checkpoint to',archive)
                print(f'\nSTAGED FIVE-SEED CORRECTION: start original seed {seed} from epoch 0')
                ACTIVE_HUE_SEED=seed
                trained=fit_baseline(df,RAW,base,'Draem',seed,'cuda',EXTRA)
                info=v4_repair_validate_checkpoint(trained,require_ranges=True)

                try:
                    doc=torch.load(trained,map_location='cpu',weights_only=True,mmap=True)
                except TypeError:
                    doc=torch.load(trained,map_location='cpu',weights_only=True)
                optimizer_states=doc.get('optimizer_states',[])
                if len(optimizer_states)!=1:
                    raise RuntimeError('Expected one Adam optimizer state in DRAEM checkpoint')
                bad_state=[]
                checked_opt=0
                for state in optimizer_states[0].get('state',{}).values():
                    for key,value in state.items():
                        if torch.is_tensor(value) and value.is_floating_point():
                            checked_opt+=1
                            if not torch.isfinite(value).all().item():bad_state.append(str(key))
                del doc
                if checked_opt<100 or bad_state:
                    raise FloatingPointError(f'Optimizer state corrupted or missing, seed {seed}: checked={checked_opt}, bad={bad_state[:5]}')
                if info['epoch']!=60 or info['global_step']!=27060:
                    raise RuntimeError(f'DRAEM staged seed {seed} did not execute full 60-epoch 27,060-step budget: {info}')
                marker=base/'COMPLETE.json'
                if not marker.is_file():raise RuntimeError('Missing stage COMPLETE after fit')
                meta=json.loads(marker.read_text())
                meta['protocol_amendment']=V4_HUE_PROTOCOL
                meta['hue_guard_tolerance']=V4_HUE_TOLERANCE
                meta['source_snapshot_sha256']=proof['snapshot_sha256']
                meta['five_seed_consistency_required']=True
                meta['hue_guard_usage']=V4_HUE_STATS_BY_SEED.get(seed,{})
                meta['optimizer_float_tensors_checked']=checked_opt
                write_json(marker,meta)
                write_json(base/'hue_guard_protocol.json',{'protocol':V4_HUE_PROTOCOL,'seed':seed,'checkpoint_sha256':info['sha256'],
                                                             'range_tolerance':V4_HUE_TOLERANCE,'training_steps':info['global_step'],
                                                             'hue_guard_usage':V4_HUE_STATS_BY_SEED.get(seed,{}),
                                                             'version_anomalib':'2.2.0'})
                print(f'FINISHED VALID STAGED SEED: {seed}; SHA256 {info["sha256"]}')
            print('ALL FIVE DRAEM CORRECTED TRAINING RUNS COMPLETE. Promote only after an independent final audit.')
        finally:
            build_model=original_build_model
            enable_v4_draem_safe_acceleration=original_enable_accel

    accept={}
    for seed in ALL_DRAEM_SEEDS:
        p=STAGE_ROOT/str(seed)/'baseline.ckpt'
        q=STAGE_ROOT/str(seed)/'COMPLETE.json'
        if not p.is_file() or not q.is_file():
            accept[str(seed)]={'ready':False,'reason':'staged checkpoint or marker missing'}
            continue
        try:
            info=v4_repair_validate_checkpoint(p,require_ranges=True)
            meta=json.loads(q.read_text())
            if meta.get('protocol_amendment')!=V4_HUE_PROTOCOL:
                raise RuntimeError('Missing hue repair protocol metadata')
            if info['epoch']!=60 or info['global_step']!=27060:
                raise RuntimeError('Incomplete training state')
            accept[str(seed)]={'ready':True,'sha256':info['sha256'],'global_step':info['global_step'],'epoch':info['epoch']}
        except Exception as e:
            accept[str(seed)]={'ready':False,'error':str(e)}
    print(json.dumps({'all_five_staged_ready':all(x['ready'] for x in accept.values()),
                      'promotion_performed':False,'seeds':accept},indent=2))

In [ ]:







import contextlib
from functools import wraps
import pandas as pd
import numpy as np

@contextlib.contextmanager
def v4_p7_only_pandas_concat_metadata_guard():
    original=pd.concat
    calls={'sanitized':0, 'objects':0}
    @wraps(original)
    def metadata_safe_concat(objs,*args,**kwargs):

        if not isinstance(objs,(list,tuple)):
            return original(objs,*args,**kwargs)
        altered=[]
        try:
            seen=set()
            for obj in objs:
                if isinstance(obj,(pd.Series,pd.DataFrame)) and obj.attrs and id(obj) not in seen:
                    seen.add(id(obj))
                    metadata=obj.attrs


                    obj.attrs={}
                    altered.append((obj,metadata))
            if altered:
                calls['sanitized']+=1
                calls['objects']+=len(altered)
            return original(objs,*args,**kwargs)
        finally:
            for obj,metadata in reversed(altered):
                obj.attrs=metadata
    pd.concat=metadata_safe_concat
    try:
        yield calls
    finally:
        pd.concat=original

def v4_p7_validate_metadata_guard():
    from sklearn.metrics import roc_auc_score
    original_concat=pd.concat

    a=pd.Series([0.1,0.9,0.2],name='score')
    b=pd.Series([0.4,0.7],name='score')
    a.attrs={'_v4_auroc_sorted_cache':(np.arange(3),np.arange(6))}
    b.attrs={'_v4_auroc_sorted_cache':(np.arange(3),np.arange(6)+1)}
    old_a=a.attrs;old_b=b.attrs
    try:
        with v4_p7_only_pandas_concat_metadata_guard() as stats:
            observed=pd.concat([a,b])
        expected=pd.concat([pd.Series([0.1,0.9,0.2],name='score'),
                            pd.Series([0.4,0.7],name='score')])
        pd.testing.assert_series_equal(observed,expected,check_exact=True)
        assert stats['sanitized']>=1
        assert a.attrs is not None and b.attrs is not None
        assert np.array_equal(a.attrs['_v4_auroc_sorted_cache'][0],old_a['_v4_auroc_sorted_cache'][0])
        assert np.array_equal(b.attrs['_v4_auroc_sorted_cache'][1],old_b['_v4_auroc_sorted_cache'][1])

        rng=np.random.default_rng(604)
        rows=[]
        for g in range(5):
            for k in ('original','benign_0','benign_1'):
                rows.append({'source_group':str(g),'source_id':str(g),'family':'none',
                             'severity':'none','label':0,'kind':k,'score':float(rng.random())})
            for fam in FAMILIES_TEST:
                for sev in SEVERITIES:
                    rows.append({'source_group':str(g),'source_id':str(g),'family':fam,
                                 'severity':sev,'label':1,'kind':fam+'_'+sev,
                                 'score':float(rng.random())})
        frame=pd.DataFrame(rows)
        expected_macro=macro(frame)
        frame.attrs={'_v4_auroc_sorted_cache':(np.arange(20),np.arange(15))}
        with v4_p7_only_pandas_concat_metadata_guard():
            actual_macro=macro(frame)
        assert expected_macro==actual_macro, 'Numerical invariance check failed'
        assert pd.concat is original_concat, 'Pandas concat not restored after scoped preflight'
        print('P7 METADATA FIX PRETEST PASSED: exact numeric/index/dtype parity; frozen macro unchanged.',flush=True)
    finally:
        assert pd.concat is original_concat, 'Pandas concat was not restored'

v4_p7_validate_metadata_guard()


In [ ]:

import csv, json, shutil, hashlib, tempfile
from pathlib import Path

assert REPRO_MODE in ('VERIFY_ARCHIVE','REBUILD_TABLES')
if not ARCHIVE_ROOT.is_dir():
    raise FileNotFoundError('Extracted full artifact archive absent at '+str(ARCHIVE_ROOT))
assert (ARCHIVE_ROOT/'frozen'/'protocol_lock.json').is_file()
assert (ARCHIVE_ROOT/'frozen'/'controller_lock.json').is_file()
assert (ARCHIVE_ROOT/'FINAL_TEST_USED.flag').is_file(), 'Original one-time test marker must be present'
assert (ARCHIVE_ROOT/'reports'/'FINAL_DECISION.json').is_file()
lock=json.loads((ARCHIVE_ROOT/'frozen'/'protocol_lock.json').read_text())
controller=json.loads((ARCHIVE_ROOT/'frozen'/'controller_lock.json').read_text())
assert hashlib.sha256((ARCHIVE_ROOT/'frozen'/'protocol_lock.json').read_bytes()).hexdigest()==controller['original_protocol_lock_sha256']
assert digest(ARCHIVE_ROOT/'data'/'source_split.csv')==lock['source_split_sha256']
assert digest(ARCHIVE_ROOT/'calibration'/'thresholds.json')==lock['thresholds_sha256']
assert lock['comparator']=='EfficientAd' and controller['prelock_comparator']=='EfficientAd'
EXPECTED_METHODS=('C00','C10','C01','C11','Patchcore','Padim','EfficientAd','Draem','Supersimplenet')
EXPECTED_SEEDS=(1049,2053,3067,4093,5051)
for name in EXPECTED_METHODS:
    for seed in EXPECTED_SEEDS:
        key=f'{name}_{seed}'
        p=ARCHIVE_ROOT/'models'/name/str(seed)/('baseline.ckpt' if not name.startswith('C') else 'model.pt')

        paths=[ARCHIVE_ROOT/'models'/name/str(seed)/('model.pt' if name.startswith('C') else 'baseline.ckpt')]
        matches=[q for q in paths if q.is_file() and digest(q)==lock['checkpoints_sha256'][key]]
        if paths[0].is_file() and not matches:
            raise RuntimeError('Present checkpoint fails frozen SHA256: '+key)
for rel, expected in controller['prelock_score_sha256'].items():
    phase, name = rel.split('/',1)
    p=ARCHIVE_ROOT/phase/'scores'/f'{name}.csv'
    if not p.is_file() or digest(p)!=expected:
        raise RuntimeError('Original '+phase+' score missing/changed: '+str(p))
for name in EXPECTED_METHODS:
    for seed in EXPECTED_SEEDS:
        for prefix in ('results','results/artifact_trap_scores'):
            p=ARCHIVE_ROOT/prefix/f'{name}_{seed}.csv'
            meta=p.with_suffix('.csv.provenance.json')
            if not p.is_file() or not meta.is_file():
                raise RuntimeError('Per-image score/provenance missing: '+str(p))
            rec=json.loads(meta.read_text())
            if rec.get('score_csv_sha256')!=digest(p):
                raise RuntimeError('Per-image score SHA mismatch: '+str(p))


seedtab=pd.read_csv(ARCHIVE_ROOT/'reports'/'all_seed_metrics.csv')
pubtab=pd.read_csv(ARCHIVE_ROOT/'reports'/'publication_primary_table.csv')
assert len(seedtab)==45 and len(pubtab)==9
computed=seedtab.groupby('method')['macro_auroc'].agg(['mean','std'])
for r in pubtab.to_dict('records'):
    v=computed.loc[r['method']]
    if not (np.isclose(v['mean'],r['macro_auroc_mean'],rtol=0,atol=1e-11) and
            np.isclose(v['std'],r['macro_auroc_sd'],rtol=0,atol=1e-11)):
        raise RuntimeError('Publication metrics do not reproduce: '+str(r['method']))
for comp in ('C00','EfficientAd'):
    q=json.loads((ARCHIVE_ROOT/'reports'/f'paired_stats_C11_vs_{comp}.json').read_text())
    contract=q['contract']
    assert contract['comparison']==['C11',comp]
    assert contract['B']==2000 and contract['permutations']==999
    for key,original_hash in contract['inputs'].items():
        p=ARCHIVE_ROOT/'results'/f'{key}.csv'
        if digest(p)!=original_hash:
            raise RuntimeError('Registered paired test was computed from different score CSV: '+key)
print('ARCHIVE REPRODUCIBILITY CHECK: PASS')
print('90 P4/P5 scores, 45 P6 scores, 45 controls and registered comparisons: VERIFIED; checkpoint hashes retained in frozen lock (weights not distributed)')
print('One-time test marker: PRESERVED; no held-out inference executed')
print(pubtab[['method','macro_auroc_mean','macro_auroc_sd']].sort_values('macro_auroc_mean',ascending=False).to_string(index=False))

if REPRO_MODE=='REBUILD_TABLES':

    if SCRATCH_ROOT.exists() and any(SCRATCH_ROOT.iterdir()):
        raise RuntimeError('Scratch directory not empty. Select a NEW scratch directory; never overwrite previous outputs.')
    for prefix in ('reports','results','data','calibration'):
        (SCRATCH_ROOT/prefix).mkdir(parents=True,exist_ok=True)
    shutil.copy2(ARCHIVE_ROOT/'data'/'source_split.csv',SCRATCH_ROOT/'data'/'source_split.csv')
    (SCRATCH_ROOT/'results'/'fixed_benchmark').mkdir(parents=True,exist_ok=True)
    shutil.copy2(ARCHIVE_ROOT/'results'/'fixed_benchmark'/'manifest.csv', SCRATCH_ROOT/'results'/'fixed_benchmark'/'manifest.csv')
    shutil.copy2(ARCHIVE_ROOT/'calibration'/'thresholds.json',SCRATCH_ROOT/'calibration'/'thresholds.json')

    _local_boot_code_hash=hashlib.sha256(repr(_stable_code(bootstrap_comparison.__code__)).encode()).hexdigest()
    for comp in ('C00','EfficientAd'):
        original_cache=ARCHIVE_ROOT/'reports'/f'paired_stats_C11_vs_{comp}.json'
        cache_contract=json.loads(original_cache.read_text())['contract']
        if cache_contract.get('code')==_local_boot_code_hash:
            shutil.copy2(original_cache,SCRATCH_ROOT/'reports'/original_cache.name)
            print('Reused original paired-test cache:',comp)
        else:
            print('Code-bytecode hash differs in this runtime; independently recomputing paired test:',comp)
    frames={}
    for name in EXPECTED_METHODS:
        for seed in EXPECTED_SEEDS:
            p=ARCHIVE_ROOT/'results'/f'{name}_{seed}.csv'
            shutil.copy2(p,SCRATCH_ROOT/'results'/p.name)
            frames[(name,seed)]=pd.read_csv(p,dtype={'sample_id':str,'source_id':str,'source_group':str})
    thresholds=json.loads((ARCHIVE_ROOT/'calibration'/'thresholds.json').read_text())
    with v4_p7_only_pandas_concat_metadata_guard():
        final_report(SCRATCH_ROOT,frames,thresholds,'EfficientAd',False)
    orig=pd.read_csv(ARCHIVE_ROOT/'reports'/'publication_primary_table.csv').sort_values('method').reset_index(drop=True)
    fresh=pd.read_csv(SCRATCH_ROOT/'reports'/'publication_primary_table.csv').sort_values('method').reset_index(drop=True)
    from pandas.testing import assert_frame_equal
    assert_frame_equal(orig,fresh,check_exact=False,rtol=1e-10,atol=1e-10)
    print('FULL TABLE REPLAY: PASS. Generated reports/figures ONLY under',SCRATCH_ROOT)
